# Section 00 — Research Overview: Weather Station Anomaly Detection

**Working title:** Uncertainty-Aware Event-Level Verification of Weather Station Anomalies Using Model Stability, Temporal Consistency and Spatial Evidence



## Section implementation status (second-pass audit)

| Section | Status |
|---|---|
| 00 Research Overview | Implemented; documentation only |
| 01 Environment and Configuration | Implemented; local tested, Colab pending |
| 02 Data Acquisition and Provenance | Partial; Phase 02 audit not implemented; real retrieval pending |
| 03 Raw Data Validation | Implemented; synthetic tests pass; real data pending |
| 04 Exploratory Data Analysis | Partial; summaries only, plots absent |
| 05 Feature Engineering | Implemented; synthetic tests pass; real parity pending |
| 06 Preprocessing and Dataset Split | Implemented; synthetic tests pass; real data pending |
| 07 Baseline Anomaly Detection | Implemented API; full research run not executed |
| 08 Model Stability and Consensus | Implemented; complete timestamps enforced; synthetic tests pass |
| 09 Temporal Event Construction | Implemented; synthetic tests pass; real candidates pending |
| 10 Matched-Control Analysis | Partial; code tested, inferential validity pending |
| 11 Synthetic Fault Experiments | Partial; injector tested, full evaluation protocol absent |
| 12 Spatial Corroboration | Partial; availability/comparison helpers only |
| 13 Physical Sensor-Quality Evidence | Partial; broad range/dew-point checks only |
| 14 Integrated Evidence Framework | Partial; rule scaffold tested, thresholds uncalibrated |
| 15 Robustness and Ablation | Not implemented |
| 16 Results and Research Reporting | Partial; status only, exports absent |

This notebook is a clean, modular workflow. It keeps original exploratory work and saved outputs in the companion archive. No real-data retrieval or model fitting runs unless explicitly enabled. Synthetic tests validate software behavior only.

**Research question.** Can model-configuration stability, temporal event consistency, independent neighboring-station observations, and physical sensor-quality evidence distinguish unusual events from suspicious station readings while preserving uncertainty?

**Evidence groups:** E1 model stability; E2 temporal consistency; E3 spatial corroboration; E4 physical sensor-quality screening. Outcomes are regional support, station-specific suspicion, or uncertain/insufficient evidence. An anomaly is not automatically a sensor fault.

**Primary pipeline:** multi-year VILD0 (Pipeline B). Pipeline A remains a separate historical baseline. Real Pipeline B verification is pending Colab/Drive artifacts.

**Second-pass status:** IMPLEMENTED — research identity and evidence outcomes documented; no empirical findings.


## Section 01 — Environment and Configuration

Install dependencies in a separate Colab setup cell if needed. This notebook does not download data automatically. Set paths/configuration explicitly.

**Second-pass status:** IMPLEMENTED — local package setup; Colab setup instructions are provided. Colab was not run.


**Colab setup:** place the entire package folder in Google Drive, mount Drive in Colab, set `WEATHER_RESEARCH_ROOT` to the package folder (or run with that folder as the working directory), then run `%pip install -r requirements.txt`. These steps install dependencies only; they do not retrieve weather data.


In [ ]:
from pathlib import Path
import os, sys, json, platform
from datetime import datetime
import numpy as np
import pandas as pd

_root_candidates = []
if os.environ.get("WEATHER_RESEARCH_ROOT"):
    _root_candidates.append(Path(os.environ["WEATHER_RESEARCH_ROOT"]).expanduser())
_root_candidates += [
    Path.cwd(),
    Path("/content/drive/MyDrive/weather_anomaly_research_cleaned"),
    Path("/content/drive/My Drive/weather_anomaly_research_cleaned"),
]
PROJECT_ROOT = next((p.resolve() for p in _root_candidates
                     if (p / "src" / "weather_research" / "__init__.py").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Cannot locate the package. In Colab, mount Drive, set WEATHER_RESEARCH_ROOT "
        "to the weather_anomaly_research_cleaned folder, or run from that folder."
    )
SRC = PROJECT_ROOT / "src"
sys.path.insert(0, str(SRC))

from weather_research import (
    add_segments, engineer_features, add_month_cycle, chronological_split,
    prepare_train_test, default_model_configs, fit_predict_configurations,
    stability_summary, build_episodes, physical_quality_flags,
    spatial_evidence, classify_evidence, phase01_integrity_audit,
)
from weather_research.temporal import episode_gap_sensitivity
from weather_research.validation import validate_observations, cadence_report
from weather_research.faults import inject_fault, evaluate_injection, inject_fault_with_feature_recompute
from weather_research.experiments import (
    matched_episode_values, paired_permutation_bootstrap, block_length_sensitivity,
)

CONFIG = {
    "timezone": "Asia/Kolkata", "station_id": "VILD0", "pipeline": "B",
    "split_cutoff": "2025-03-21 17:30:00+05:30", "random_seed": 42,
    "run_retrieval": False, "run_models": False, "candidate_vote_cutoff": 8,
    "raw_csv": os.environ.get("WEATHER_RAW_CSV", ""),
    "output_dir": PROJECT_ROOT / "outputs",
}
CONFIG["output_dir"].mkdir(parents=True, exist_ok=True)
ENVIRONMENT = {"python": platform.python_version(), "pandas": pd.__version__,
               "numpy": np.__version__}
try:
    import sklearn
    ENVIRONMENT["scikit_learn"] = sklearn.__version__
except ImportError:
    ENVIRONMENT["scikit_learn"] = "not installed"
print(ENVIRONMENT)
raw_observations = None
HISTORICAL_PIPELINE_B = None  # explicitly set to dict(model_df=..., x_multi=..., x_train=..., x_test=...)


{'python': '3.13.16', 'pandas': '2.2.3', 'numpy': '2.4.6', 'scikit_learn': '1.6.1'}


## Section 02 — Data Acquisition and Provenance

Historical acquisition used Meteostat observation-only settings. `model=False` or `include_model_data=False` requests exclusion of model-filled values; neither certifies instrument-level provenance. Retrieval is opt-in and records version/API branch.

**Second-pass status:** PARTIAL — explicit observation-only retrieval helper exists; the Phase 02 station inventory/readback/candidate coverage audit is NOT IMPLEMENTED in this package; no real retrieval was run.


In [ ]:
from weather_research.retrieval import fetch_observations
STATION_IDS = ["VILD0", "42099", "42101", "42105", "42096"]
# Explicit acquisition example (not run automatically):
# station_frames, retrieval_manifest = fetch_observations(
#     STATION_IDS, datetime(2025,3,28), datetime(2026,3,9,23,59), CONFIG["timezone"]
# )
retrieval_manifest = None


## Section 03 — Raw Data Validation

Load only the documented Pipeline B station file. Preserve raw observations; reject duplicate/unsorted timestamps for analysis instead of silently modifying them.

**Second-pass status:** IMPLEMENTED — validation primitives are synthetically tested; real data validation awaits Pipeline B inputs.


In [ ]:
if CONFIG["raw_csv"]:
    raw_path = Path(CONFIG["raw_csv"])
    if not raw_path.is_file(): raise FileNotFoundError(raw_path)
    raw_observations = pd.read_csv(raw_path, index_col=0, parse_dates=True)
    if raw_observations.index.tz is None:
        raise ValueError("CSV timestamps must retain timezone or be explicitly localized")
    raw_validation = validate_observations(raw_observations)
    coverage = cadence_report(raw_observations.index)
else:
    raw_validation = coverage = None
print("Raw data loaded:", raw_observations is not None)


Raw data loaded: False


## Section 04 — Exploratory Data Analysis

Exploration is descriptive. Coverage and missingness must accompany distribution summaries; do not treat missing station records as normal weather.

**Second-pass status:** PARTIAL — descriptive statistics and missingness table only; time-series/distribution/missingness plots are NOT IMPLEMENTED.


In [ ]:
if raw_observations is not None:
    descriptive = raw_observations.describe(include="all").T
    missingness = raw_observations.isna().agg(["sum", "mean"]).T
else:
    descriptive = missingness = None


## Section 05 — Feature Engineering

One canonical implementation preserves the historical previous-six-observation rolling mean with `shift(1)`. It distinguishes historical previous-row change from strict one-hour change, which is null when continuity is not exactly hourly. Segment IDs follow observed gaps; missing segment IDs retain pandas groupby drop-null behavior.

**Second-pass status:** IMPLEMENTED — canonical features are synthetically tested; real historical parity is pending. Strict-hour change changes historical model inputs.


In [ ]:
BASE_VARIABLES = ["temp", "dwpt", "rhum", "prcp", "wspd", "pres"]
ENGINEERED_COLUMNS = ["temp_change_1h", "temp_rolling_mean_6obs", "temp_deviation_6obs"]
if raw_observations is not None:
    segmented = raw_observations if "segment_id" in raw_observations else add_segments(raw_observations)
    engineered = engineer_features(segmented)
    seasonal_features = add_month_cycle(engineered)
    # Historical row-based feature is retained as temp_change_row. Historical predictions
    # are not reusable with strict temp_change_1h unless their original inputs match.
else:
    engineered = None


## Section 06 — Preprocessing and Dataset Split

Chronological cutoff is kept explicit. Training medians and scaling are fit on training rows only. If the underlying index has no records between train end and test start, that historical gap remains visible; this code does not fill it.

**Second-pass status:** IMPLEMENTED — chronological split and train-only preprocessing are synthetically tested; real split/imputation audit awaits Pipeline B.


In [ ]:
if engineered is not None:
    canonical_features = BASE_VARIABLES + ENGINEERED_COLUMNS
    model_frame = engineered[canonical_features].dropna(subset=ENGINEERED_COLUMNS)
    train_frame, test_frame = chronological_split(model_frame, CONFIG["split_cutoff"])
    X_train, X_test, prep = prepare_train_test(train_frame, test_frame, canonical_features)
    seasonal_train = add_month_cycle(train_frame)
    seasonal_test = add_month_cycle(test_frame)
    seasonal_columns = canonical_features + ["month_sin", "month_cos"]
    X_train_seasonal, X_test_seasonal, seasonal_prep = prepare_train_test(
        seasonal_train, seasonal_test, seasonal_columns)
    X_train_scaled, X_test_scaled, scaled_prep = prepare_train_test(
        seasonal_train, seasonal_test, seasonal_columns, scale=True)
    phase01_audit = (phase01_integrity_audit(**HISTORICAL_PIPELINE_B)
                     if HISTORICAL_PIPELINE_B is not None else None)
    split_audit = {"train_end": train_frame.index.max(), "test_start": test_frame.index.min(),
                   "gap": test_frame.index.min() - train_frame.index.max(),
                   "overlap": len(train_frame.index.intersection(test_frame.index))}
else:
    train_frame = test_frame = X_train = X_test = prep = split_audit = None
    X_train_seasonal = X_test_seasonal = X_train_scaled = X_test_scaled = None
    phase01_audit = None


## Section 07 — Baseline Anomaly Detection

Historical model definitions: Isolation Forest on unscaled seasonal features; LOF and One-Class SVM on train-scaled inputs. The master defines the registry but does not retrain by default. Set `run_models=True` only after confirming the feature/data version.

**Second-pass status:** IMPLEMENTED — model definitions and training API exist; limited synthetic model test passed; full 12-configuration research run is NOT EXECUTED.


In [ ]:
MODEL_CONFIGS = default_model_configs(CONFIG["random_seed"])
prediction_table = None
if CONFIG["run_models"]:
    if X_train_seasonal is None:
        raise RuntimeError("Load and validate Pipeline B data before model fitting")
    prediction_table = fit_predict_configurations(
        X_train_seasonal, X_test_seasonal, X_train_scaled, X_test_scaled,
        configs=MODEL_CONFIGS, n_estimators=200)


## Section 08 — Model Stability and Consensus

Configuration votes and family-balanced agreement are agreement summaries, not probabilities. Prediction joins use exact timestamps.

**Second-pass status:** IMPLEMENTED — complete timestamp alignment is now enforced and synthetically tested; real prediction matrices are not available.


In [ ]:
stability, jaccard = stability_summary(prediction_table) if prediction_table is not None else (None, None)


## Section 09 — Temporal Event Construction

Episode boundaries respect segment changes and configurable maximum gaps. Compare one-, two-, and three-hour definitions; report elapsed duration and observed count separately.

**Second-pass status:** IMPLEMENTED — gap/segment episode construction and sensitivity helper are synthetically tested; not run on real candidates.


In [ ]:
episode_table = None
if stability is not None:
    candidate_times = stability.index[stability["configuration_votes"] >= CONFIG["candidate_vote_cutoff"]]
    segment_ids = engineered["segment_id"].reindex(candidate_times)
    episode_table = episode_gap_sensitivity(candidate_times, segment_ids,
                                               thresholds=("1h", "2h", "3h"))


## Section 10 — Matched-Control Analysis

Match by month, hour, event length and complete hourly coverage; exclude candidate/control overlap. Inference resamples matched episodes and tests the same mean paired difference it reports. This is pending real candidate data and a preregistered matching rule.

**Second-pass status:** PARTIAL — matching and block-resampling utilities exist and have synthetic tests; matching validity, selection sensitivity and inferential assumptions are NOT scientifically validated.


In [ ]:
matched_pairs = None
matched_inference = None
matched_block_sensitivity = None
if episode_table is not None:
    one_hour_episodes = episode_table.loc[episode_table["gap_threshold"] == "1h"]
    matched_pairs = matched_episode_values(
        engineered, one_hour_episodes, "temp", candidate_times, max_controls=1,
        eligible_control_start=test_frame.index.min())
    if len(matched_pairs) >= 2:
        matched_inference = paired_permutation_bootstrap(matched_pairs)
        matched_block_sensitivity = block_length_sensitivity(matched_pairs, block_lengths=(1,2,3,4))


## Section 11 — Synthetic Fault Experiments

Inject only into held-out raw observations and label injected timestamps. Keep calibration data separate. Metrics from these controlled injections apply only to injected labels and cannot establish real-world accuracy.

**Second-pass status:** PARTIAL — fault types and label isolation are tested; a full controlled fault campaign, independent-reference protocol and event-level detection/delay evaluation are NOT IMPLEMENTED.


In [ ]:
fault_registry = pd.DataFrame()
# Explicit example after selecting held-out timestamps:
# Use an explicitly selected held-out raw interval and retain the clean reference features:
# injected_raw, labels, registry, clean_features, injected_features = inject_fault_with_feature_recompute(
#     segmented, "temp", start, end, kind="drift", magnitude=3.0,
#     random_seed=CONFIG["random_seed"], calibration_end=train_frame.index.max())
# The shifted rolling reference at each timestamp uses prior observations only.


## Section 12 — Spatial Corroboration

Exact-time observed values only. Candidate status distinguishes observed timestamp, before/after coverage, internal gap, no returned station data and variable unavailable. No interpolation or model-assisted fill is used as independent corroboration.

**Second-pass status:** PARTIAL — exact-time availability and caller-tolerance comparison exist; station inventory, distance-aware/seasonal comparisons and Phase 02 coverage audit are NOT IMPLEMENTED.


In [ ]:
spatial_table = None
# spatial_table = spatial_evidence(candidate_times, observation_only_station_frames, "temp")


## Section 13 — Physical Sensor-Quality Evidence

Broad limits are screening rules, not station calibration specifications or proof of sensor failure. Exact raw values and timestamps are retained.

**Second-pass status:** PARTIAL — broad range and dew-point screening are implemented; flatline, discontinuity, stuck-sensor and cross-variable behavior evidence is NOT IMPLEMENTED as a general module.


In [ ]:
physical_flags = physical_quality_flags(raw_observations) if raw_observations is not None else None


## Section 14 — Integrated Evidence Framework

No arbitrary probability or weighted score. Region support requires a declared number of independent observed neighbors. A station-specific suspicious outcome requires a hard physical screening violation and conflicting observed spatial evidence. Missing/conflicting/insufficient evidence abstains. Rules require domain review before research claims.

**Second-pass status:** PARTIAL — three-outcome abstaining rule scaffold is synthetically tested; thresholds are not calibrated or validated on real candidates.


In [ ]:
candidate_evidence = None
# Supply a timestamp-indexed evidence DataFrame after E1/E2/E3/E4 inputs have been
# assembled and audited. Missing E3 remains UNAVAILABLE; it is never disagreement.
if candidate_evidence is not None:
    candidate_evidence = classify_evidence(candidate_evidence, min_supporting_stations=2)


## Section 15 — Robustness and Ablation

Evaluate model and evidence-group ablations only after real-data provenance and synthetic fault protocols are fixed. Report evidence coverage separately from detection outcomes.

**Second-pass status:** NOT IMPLEMENTED — ablation runner and results are absent; `ablation_results` remains an explicit placeholder.


In [ ]:
ablation_results = None  # Pending validated Phase 03–12 outputs


## Section 16 — Results and Research Reporting

Export only executed results. Historical notebook outputs remain historical until reproduced. Do not report accuracy for unlabeled weather observations.

**Second-pass status:** PARTIAL — status display exists; report/figure/result export pipeline is NOT IMPLEMENTED.


In [ ]:
experiment_registry_path = PROJECT_ROOT / "experiment_registry.csv"
research_status = {"real_pipeline_b_available": raw_observations is not None,
                   "phase01_real_data_verified": False, "historical_results_reproduced": False}
print(research_status)


{'real_pipeline_b_available': False, 'phase01_real_data_verified': False, 'historical_results_reproduced': False}


In [1]:

# STEP 3: Mount Google Drive and configure project root

from google.colab import drive
from pathlib import Path
import os
import sys

# Mount Google Drive
drive.mount('/content/drive')

# Set the project folder path
WEATHER_RESEARCH_ROOT = Path(
    "/content/drive/MyDrive/weather_anomaly_research"
).resolve()

# Make the path available to project modules
os.environ["WEATHER_RESEARCH_ROOT"] = str(WEATHER_RESEARCH_ROOT)

# Add src directory to Python import path
SRC_PATH = WEATHER_RESEARCH_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

print("Project root:", WEATHER_RESEARCH_ROOT)
print("Project root exists:", WEATHER_RESEARCH_ROOT.exists())
print("Source folder exists:", SRC_PATH.exists())


Mounted at /content/drive
Project root: /content/drive/MyDrive/weather_anomaly_research
Project root exists: False
Source folder exists: False


In [3]:

from pathlib import Path

drive_root = Path("/content/drive/MyDrive")

print("Folders in MyDrive:\n")

for item in sorted(drive_root.iterdir()):
    print(("📁 " if item.is_dir() else "📄 ") + item.name)


Folders in MyDrive:

📄 
i. ExplaintheworkingprincipleofQmeter.
ii. Whatarethedifferenttypesoferrorinmeasurement?
iii. Definesensitivityandresoluti~ block diagram of digital multimetre. Also write its applications
d. Draw and explain the block diagram of pulse generator..gsheet
📄 BM Electronics And Communication Engineering DIPLOMA 6th - PSBTE Very Short Questions | BRpaper.com
📄 CHANDAN_SAINI[2]new.gdoc
📁 Colab Notebooks
📄 Copy of CN Previous Years.pdf
📄 Copy of Quick Walkthrough
📄 Copy of Untitled2.ipynb
📄 Copy of amazon_logo.png
📄 Copy of short numericals.pptx
📄 DOC-20250326-WA0002..gslides
📄 DOC-20260605-WA0000.gdoc
📄 DOC-20260917-WA0040.gdoc
📄 EAadhaar_0647003122614120210701105404_13052024174625.pdf
📄 EPEA 204 PPTs
📄 Eng 140 Unit 8 to 14 Self assessment.gdoc
📄 Eng 140 unit 1 to 7 Self assessment.gdoc
📄 Far_demonic suggestions ai.gdoc
📄 Far_demonic(story).gdoc
📁 Farewell 2025
📄 Getting started.pdf
📄 Google Keep Document.gdoc
📄 Gym Membership & Fitness Survey (1).gform
📄 Gym Membersh

In [4]:

from pathlib import Path

drive_root = Path("/content/drive/MyDrive")

matches = list(
    drive_root.rglob("Weather_Station_Anomaly_Detection_Master.ipynb")
)

print("Master notebook matches:", len(matches))

for path in matches:
    print(path)


Master notebook matches: 1
/content/drive/MyDrive/weather_anomaly_research_cleaned/Weather_Station_Anomaly_Detection_Master.ipynb


In [5]:
from pathlib import Path
import os
import sys

WEATHER_RESEARCH_ROOT = Path(
    "/content/drive/MyDrive/weather_anomaly_research_cleaned"
).resolve()

os.environ["WEATHER_RESEARCH_ROOT"] = str(WEATHER_RESEARCH_ROOT)

SRC_PATH = WEATHER_RESEARCH_ROOT / "src"

if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

print("Project root exists:", WEATHER_RESEARCH_ROOT.exists())
print("Source folder exists:", SRC_PATH.exists())
print("\nProject contents:")

for item in sorted(WEATHER_RESEARCH_ROOT.iterdir()):
    print(("📁 " if item.is_dir() else "📄 ") + item.name)

print("\nSource modules:")
if SRC_PATH.exists():
    for item in sorted((SRC_PATH / "weather_research").glob("*.py")):
        print(item.name)

Project root exists: True
Source folder exists: True

Project contents:
📄 .gitignore
📄 DEPENDENCY_MAP.md
📄 FINAL_BUGFIX_REPORT.md
📄 PROJECT_STATUS.md
📄 README.md
📄 REFACTORING_REPORT.md
📄 SECOND_PASS_AUDIT.md
📄 TESTING_REPORT.md
📄 Weather_Station_Anomaly_Detection_Exploratory_Archive.ipynb
📄 Weather_Station_Anomaly_Detection_Master.ipynb
📄 cell_audit_inventory.csv
📄 cell_migration_manifest.csv
📄 decision_log.md
📄 experiment_registry.csv
📁 outputs
📄 record.md
📄 requirements.txt
📁 src
📁 tests

Source modules:
__init__.py
audit.py
evidence.py
experiments.py
faults.py
features.py
models.py
preprocessing.py
quality.py
retrieval.py
spatial.py
temporal.py
validation.py


In [6]:
import json
import ast
from pathlib import Path

notebook_path = (
    WEATHER_RESEARCH_ROOT
    / "Weather_Station_Anomaly_Detection_Master.ipynb"
)

audit_path = (
    SRC_PATH
    / "weather_research"
    / "audit.py"
)

# 1. Inspect notebook structure
with open(notebook_path, "r", encoding="utf-8") as f:
    notebook = json.load(f)

cells = notebook["cells"]

print("=" * 65)
print("MASTER NOTEBOOK STRUCTURE")
print("=" * 65)
print("Total cells:", len(cells))

for i, cell in enumerate(cells):
    source = "".join(cell.get("source", []))
    first_line = next(
        (line.strip() for line in source.splitlines() if line.strip()),
        "(empty)"
    )

    if cell["cell_type"] == "markdown":
        print(f"\n[Markdown {i}] {first_line[:180]}")
    else:
        print(f"\n[Code {i}] {first_line[:180]}")

# 2. Inspect audit.py without importing or executing it
print("\n" + "=" * 65)
print("AUDIT MODULE FUNCTIONS")
print("=" * 65)

source = audit_path.read_text(encoding="utf-8")
tree = ast.parse(source)

for node in tree.body:
    if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
        print(f"Function: {node.name} | line {node.lineno}")
    elif isinstance(node, ast.ClassDef):
        print(f"Class: {node.name} | line {node.lineno}")

print("\nAudit module path:", audit_path)
print("Audit module exists:", audit_path.exists())

MASTER NOTEBOOK STRUCTURE
Total cells: 37

[Markdown 0] # Section 00 — Research Overview: Weather Station Anomaly Detection

[Markdown 1] ## Section 01 — Environment and Configuration

[Code 2] from pathlib import Path

[Markdown 3] ## Section 02 — Data Acquisition and Provenance

[Code 4] from weather_research.retrieval import fetch_observations

[Markdown 5] ## Section 03 — Raw Data Validation

[Code 6] if CONFIG["raw_csv"]:

[Markdown 7] ## Section 04 — Exploratory Data Analysis

[Code 8] if raw_observations is not None:

[Markdown 9] ## Section 05 — Feature Engineering

[Code 10] BASE_VARIABLES = ["temp", "dwpt", "rhum", "prcp", "wspd", "pres"]

[Markdown 11] ## Section 06 — Preprocessing and Dataset Split

[Code 12] if engineered is not None:

[Markdown 13] ## Section 07 — Baseline Anomaly Detection

[Code 14] MODEL_CONFIGS = default_model_configs(CONFIG["random_seed"])

[Markdown 15] ## Section 08 — Model Stability and Consensus

[Code 16] stability, jaccard = stability_summary(p

In [7]:
import ast
import inspect
import textwrap
import json
from pathlib import Path

# Load notebook as text
notebook_path = WEATHER_RESEARCH_ROOT / "Weather_Station_Anomaly_Detection_Master.ipynb"

with open(notebook_path, "r", encoding="utf-8") as f:
    nb = json.load(f)

# Inspect configuration, acquisition, validation, features and preprocessing
inspect_cells = [2, 4, 6, 8, 10, 12]

for i in inspect_cells:
    cell = nb["cells"][i]
    source = "".join(cell.get("source", []))

    print("\n" + "=" * 75)
    print(f"NOTEBOOK CELL {i}")
    print("=" * 75)
    print(source)

# Inspect audit.py without importing or executing it
audit_path = SRC_PATH / "weather_research" / "audit.py"
audit_source = audit_path.read_text(encoding="utf-8")
audit_tree = ast.parse(audit_source)

print("\n" + "=" * 75)
print("PHASE 01 AUDIT FUNCTION")
print("=" * 75)

for node in audit_tree.body:
    if isinstance(node, ast.FunctionDef):
        if node.name == "phase01_integrity_audit":
            print("Signature:")
            print(ast.unparse(node.args))
            print("\nFunction source:")
            print(ast.get_source_segment(audit_source, node))

print("\n" + "=" * 75)
print("FEATURE RECOMPUTATION FUNCTION")
print("=" * 75)

for node in audit_tree.body:
    if isinstance(node, ast.FunctionDef):
        if node.name == "recompute_historical_features":
            print("Signature:")
            print(ast.unparse(node.args))
            print("\nFunction source:")
            print(ast.get_source_segment(audit_source, node))


NOTEBOOK CELL 2
from pathlib import Path
import os, sys, json, platform
from datetime import datetime
import numpy as np
import pandas as pd

_root_candidates = []
if os.environ.get("WEATHER_RESEARCH_ROOT"):
    _root_candidates.append(Path(os.environ["WEATHER_RESEARCH_ROOT"]).expanduser())
_root_candidates += [
    Path.cwd(),
    Path("/content/drive/MyDrive/weather_anomaly_research_cleaned"),
    Path("/content/drive/My Drive/weather_anomaly_research_cleaned"),
]
PROJECT_ROOT = next((p.resolve() for p in _root_candidates
                     if (p / "src" / "weather_research" / "__init__.py").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Cannot locate the package. In Colab, mount Drive, set WEATHER_RESEARCH_ROOT "
        "to the weather_anomaly_research_cleaned folder, or run from that folder."
    )
SRC = PROJECT_ROOT / "src"
sys.path.insert(0, str(SRC))

from weather_research import (
    add_segments, engineer_features, add_month_cycle, chrono

In [8]:
from pathlib import Path
import ast
import inspect

package_path = SRC_PATH / "weather_research"

files_to_inspect = [
    "audit.py",
    "features.py",
    "preprocessing.py",
]

for filename in files_to_inspect:
    path = package_path / filename
    source = path.read_text(encoding="utf-8")
    tree = ast.parse(source)

    print("\n" + "=" * 70)
    print(filename)
    print("=" * 70)

    for node in tree.body:
        if isinstance(node, ast.Assign):
            names = [
                target.id for target in node.targets
                if isinstance(target, ast.Name)
            ]
            if names:
                print("Constant/variable:", ", ".join(names))
                print(ast.get_source_segment(source, node)[:1500])

        elif isinstance(node, ast.FunctionDef):
            print(f"\nFunction: {node.name}")
            print("Signature:", ast.unparse(node.args))
            print("First line:", ast.get_source_segment(source, node).splitlines()[0])

print("\n" + "=" * 70)
print("PROJECT OUTPUT FILES")
print("=" * 70)

outputs_path = WEATHER_RESEARCH_ROOT / "outputs"

for path in sorted(outputs_path.rglob("*")):
    if path.is_file():
        print(path.relative_to(WEATHER_RESEARCH_ROOT), "|", path.stat().st_size, "bytes")


audit.py
Constant/variable: HISTORICAL_FEATURES
HISTORICAL_FEATURES = ("temp_change_1h", "temp_rolling_mean_6h", "temp_deviation_6h")

Function: recompute_historical_features
Signature: model_df: pd.DataFrame, segment_col='segment_id'
First line: def recompute_historical_features(model_df: pd.DataFrame, segment_col="segment_id") -> pd.DataFrame:

Function: _compare_features
Signature: actual, expected, tolerance
First line: def _compare_features(actual, expected, tolerance):

Function: phase01_integrity_audit
Signature: model_df: pd.DataFrame, x_multi: pd.DataFrame, x_train: pd.DataFrame, x_test: pd.DataFrame, split_boundary='2025-03-21 17:30:00+05:30', segment_col='segment_id', tolerance=1e-08
First line: def phase01_integrity_audit(model_df: pd.DataFrame, x_multi: pd.DataFrame,

features.py

Function: add_segments
Signature: frame: pd.DataFrame, max_gap='1h', segment_col='segment_id'
First line: def add_segments(frame: pd.DataFrame, max_gap="1h", segment_col="segment_id") -> pd.Data

In [9]:
import inspect

from weather_research import features, audit, preprocessing

functions_to_inspect = [
    ("features.engineer_features", features.engineer_features),
    ("features.add_segments", features.add_segments),
    ("audit.recompute_historical_features", audit.recompute_historical_features),
    ("audit.phase01_integrity_audit", audit.phase01_integrity_audit),
    ("preprocessing.chronological_split", preprocessing.chronological_split),
    ("preprocessing.prepare_train_test", preprocessing.prepare_train_test),
]

for name, func in functions_to_inspect:
    print("\n" + "=" * 80)
    print(name)
    print("=" * 80)
    print(inspect.getsource(func))


features.engineer_features
def engineer_features(
    frame: pd.DataFrame,
    segment_col="segment_id",
    rolling_observations=6,
    expected_interval="1h",
) -> pd.DataFrame:
    """Add canonical features without mutating input.

    ``temp_change_row`` preserves the historic grouped previous-row difference.
    ``temp_change_1h`` is the strict elapsed-time version and is null unless the
    preceding row is exactly one hour earlier. The rolling mean is the historic
    previous-six-observation mean (shift(1)); missing segment IDs follow pandas
    groupby(dropna=True) semantics and therefore yield null engineered features.
    """
    missing = set(BASE_VARIABLES + (segment_col,)) - set(frame.columns)
    if missing:
        raise KeyError(f"missing feature inputs: {sorted(missing)}")
    if not isinstance(frame.index, pd.DatetimeIndex):
        raise TypeError("feature engineering requires a DatetimeIndex")
    if frame.index.has_duplicates or not frame.index.is_monotonic_incre

In [10]:
import pandas as pd

names = ["model_df", "X_multi", "X_train", "X_test",
         "raw_observations", "engineered", "train_frame", "test_frame"]

print("PIPELINE B RUNTIME CHECK")
print("=" * 55)

for name in names:
    exists = name in globals()
    value = globals().get(name)

    if isinstance(value, pd.DataFrame):
        print(f"{name}: EXISTS | shape={value.shape} | "
              f"index={type(value.index).__name__}")
    else:
        print(f"{name}: {'EXISTS' if exists else 'NOT FOUND'} | "
              f"type={type(value).__name__ if exists else '—'}")

PIPELINE B RUNTIME CHECK
model_df: NOT FOUND | type=—
X_multi: NOT FOUND | type=—
X_train: NOT FOUND | type=—
X_test: NOT FOUND | type=—
raw_observations: NOT FOUND | type=—
engineered: NOT FOUND | type=—
train_frame: NOT FOUND | type=—
test_frame: NOT FOUND | type=—


In [11]:
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive")

keywords = [
    "weather",
    "anomaly",
    "vild0",
    "ludhiana",
    "meteostat",
    "pipeline",
]

matches = []

for path in DRIVE_ROOT.rglob("*"):
    if not path.is_file():
        continue

    name = path.name.lower()

    if path.suffix.lower() in [".csv", ".ipynb", ".parquet", ".pkl"]:
        if any(word in name for word in keywords):
            matches.append(path)

print("POTENTIAL SOURCE FILES")
print("=" * 75)

for path in sorted(matches):
    print(f"{path} | {path.stat().st_size:,} bytes")

print("\nTotal matching files:", len(matches))

POTENTIAL SOURCE FILES
/content/drive/MyDrive/Colab Notebooks/Copy_of_Weather_Anomaly_Detection_2 (6)_phase01.ipynb | 4,012,503 bytes
/content/drive/MyDrive/weather_anomaly_phase00/VILD0_observed.csv | 58,115 bytes
/content/drive/MyDrive/weather_anomaly_research_cleaned/Weather_Station_Anomaly_Detection_Exploratory_Archive.ipynb | 4,104,749 bytes
/content/drive/MyDrive/weather_anomaly_research_cleaned/Weather_Station_Anomaly_Detection_Master.ipynb | 88,159 bytes

Total matching files: 4


In [12]:
import json
import hashlib
from pathlib import Path
import pandas as pd

original_path = Path(
    "/content/drive/MyDrive/Colab Notebooks/"
    "Copy_of_Weather_Anomaly_Detection_2 (6)_phase01.ipynb"
)

archive_path = Path(
    "/content/drive/MyDrive/weather_anomaly_research_cleaned/"
    "Weather_Station_Anomaly_Detection_Exploratory_Archive.ipynb"
)

csv_path = Path(
    "/content/drive/MyDrive/weather_anomaly_phase00/VILD0_observed.csv"
)

# --------------------------------------------------
# 1. Compare notebook copies
# --------------------------------------------------

print("=" * 75)
print("NOTEBOOK COMPARISON")
print("=" * 75)

for path in [original_path, archive_path]:
    raw_bytes = path.read_bytes()
    notebook = json.loads(raw_bytes)

    code_cells = [
        c for c in notebook["cells"]
        if c["cell_type"] == "code"
    ]

    executed = [
        c for c in code_cells
        if c.get("execution_count") is not None
    ]

    print("\nFile:", path.name)
    print("Size:", len(raw_bytes))
    print("SHA256:", hashlib.sha256(raw_bytes).hexdigest())
    print("Total cells:", len(notebook["cells"]))
    print("Code cells:", len(code_cells))
    print("Executed code cells:", len(executed))

# --------------------------------------------------
# 2. Inspect original notebook's relevant cells
# --------------------------------------------------

notebook = json.loads(original_path.read_text(encoding="utf-8"))

keywords = [
    "read_csv",
    "model_df",
    "X_multi",
    "X_train",
    "X_test",
    "Cleaned_dataset",
    "VILD0",
    "Date_of_journey",
    "temp_rolling_mean",
    "split_boundary",
    "2025-03-21",
]

print("\n" + "=" * 75)
print("ORIGINAL NOTEBOOK: RELEVANT CELLS")
print("=" * 75)

shown = 0

for i, cell in enumerate(notebook["cells"]):
    if cell["cell_type"] != "code":
        continue

    source = "".join(cell.get("source", []))
    matching_lines = [
        line for line in source.splitlines()
        if any(k.lower() in line.lower() for k in keywords)
    ]

    if matching_lines:
        print(f"\nCELL {i} | execution_count={cell.get('execution_count')}")
        print("\n".join(matching_lines[:35]))
        shown += 1

    if shown >= 35:
        print("\nStopped after 35 matching cells.")
        break

# --------------------------------------------------
# 3. Inspect VILD0 CSV
# --------------------------------------------------

print("\n" + "=" * 75)
print("VILD0 CSV INSPECTION")
print("=" * 75)

if csv_path.exists():
    df = pd.read_csv(csv_path, index_col=0, parse_dates=True)

    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    print("Index type:", type(df.index).__name__)
    print("Timezone:", getattr(df.index, "tz", None))
    print("First timestamp:", df.index.min())
    print("Last timestamp:", df.index.max())
    print("Duplicate timestamps:", df.index.has_duplicates)
    print("\nMissing values:")
    print(df.isna().sum())
else:
    print("CSV not found:", csv_path)

NOTEBOOK COMPARISON

File: Copy_of_Weather_Anomaly_Detection_2 (6)_phase01.ipynb
Size: 4012503
SHA256: 43e4ad04c75d41dd99c48f5e9b9022cc9397dc871c01fa9ae54a7aaba1f2128a
Total cells: 222
Code cells: 222
Executed code cells: 160

File: Weather_Station_Anomaly_Detection_Exploratory_Archive.ipynb
Size: 4104749
SHA256: 14df99bfe9cd1b25e8370bc35fd004ad9c44a550e53adc1b079f0c611ee082dc
Total cells: 222
Code cells: 222
Executed code cells: 160

ORIGINAL NOTEBOOK: RELEVANT CELLS

CELL 4 | execution_count=176
station_id = "VILD0"

CELL 11 | execution_count=183
hourly_df["temp_rolling_mean_6h"] = (
    hourly_df["temp_rolling_mean_6h"]
        ["temp", "temp_rolling_mean_6h",

CELL 14 | execution_count=186
model_df = df[base_features].copy()
model_df = model_df.asfreq("h")
model_df["temp_change_1h"] = model_df["temp"].diff()
model_df["temp_rolling_mean_6h"] = (
    model_df["temp"]
model_df["temp_deviation_6h"] = (
    model_df["temp"] -
    model_df["temp_rolling_mean_6h"]
print("Shape:", model_df

In [13]:
import json
from pathlib import Path

notebook_path = Path(
    "/content/drive/MyDrive/Colab Notebooks/"
    "Copy_of_Weather_Anomaly_Detection_2 (6)_phase01.ipynb"
)

nb = json.loads(notebook_path.read_text(encoding="utf-8"))

# Inspect the early data-loading cells and the historical
# data preparation cells immediately before Pipeline B.
cell_indices = list(range(0, 15)) + list(range(40, 58))

for i in cell_indices:
    if i >= len(nb["cells"]):
        continue

    cell = nb["cells"][i]
    source = "".join(cell.get("source", []))

    print("\n" + "=" * 85)
    print(
        f"CELL {i} | TYPE={cell['cell_type']} | "
        f"EXECUTION={cell.get('execution_count')}"
    )
    print("=" * 85)
    print(source)


CELL 0 | TYPE=code | EXECUTION=172

!pip install -q pandas==2.2.3 meteostat matplotlib

CELL 1 | TYPE=code | EXECUTION=173

import pandas as pd
import meteostat as ms
import matplotlib

print("Pandas version:", pd.__version__)
print("Meteostat imported successfully")
print("Matplotlib version:", matplotlib.__version__)

CELL 2 | TYPE=code | EXECUTION=174

import meteostat

print("Meteostat version:", meteostat.__version__)
print("Available functions:")
print(dir(meteostat))

CELL 3 | TYPE=code | EXECUTION=175

from meteostat import Stations
import pandas as pd

# Ludhiana coordinates
latitude = 30.90
longitude = 75.85

# Search nearby weather stations
stations = (
    Stations()
    .nearby(latitude, longitude)
    .fetch(20)
)

display(stations)

CELL 4 | TYPE=code | EXECUTION=176

from meteostat import Hourly
from datetime import datetime
import pandas as pd

# Selected station
station_id = "VILD0"

# One-year period based on its inventory
start = datetime(2025, 3, 10)
end = datetim

In [14]:
import json
from pathlib import Path

path = Path(
    "/content/drive/MyDrive/Colab Notebooks/"
    "Copy_of_Weather_Anomaly_Detection_2 (6)_phase01.ipynb"
)

nb = json.loads(path.read_text(encoding="utf-8"))

selected_cells = [40, 41, 42, 43, 44, 46, 49, 50, 51, 52, 53, 54, 55, 56, 57, 64, 65]

for i in selected_cells:
    cell = nb["cells"][i]

    print("\n" + "=" * 75)
    print(f"CELL {i} | execution={cell.get('execution_count')}")
    print("=" * 75)

    for output in cell.get("outputs", []):
        if "text" in output:
            print("".join(output["text"]))

        elif "data" in output:
            data = output["data"]

            if "text/plain" in data:
                print("".join(data["text/plain"]))

            elif "text/html" in data:
                print("[HTML table output available]")


CELL 40 | execution=212
Dataset shape: (45229, 11)
Start: 2020-06-18 11:30:00+05:30
End: 2026-03-09 23:30:00+05:30

                           temp  dwpt  rhum  prcp  snow   wdir  wspd  wpgt  \
time                                                                         
2020-06-18 11:30:00+05:30  37.0  23.8  47.0  <NA>  <NA>  140.0   4.0  <NA>   
2020-06-18 12:30:00+05:30  38.0  24.0  45.0  <NA>  <NA>   <NA>   0.0  <NA>   
2020-06-18 14:30:00+05:30  40.0  21.6  35.0  <NA>  <NA>   <NA>   0.0  <NA>   
2020-06-18 15:30:00+05:30  41.0  22.9  36.0  <NA>  <NA>   <NA>   0.0  <NA>   
2020-06-18 16:30:00+05:30  41.0  22.9  36.0  <NA>  <NA>  220.0   2.0  <NA>   

                            pres  tsun  coco  
time                                          
2020-06-18 11:30:00+05:30  999.0  <NA>   5.0  
2020-06-18 12:30:00+05:30  998.0  <NA>   5.0  
2020-06-18 14:30:00+05:30  997.0  <NA>   5.0  
2020-06-18 15:30:00+05:30  996.0  <NA>   5.0  
2020-06-18 16:30:00+05:30  995.0  <NA>   5.0  

CELL 4

In [16]:
from meteostat import Hourly
from datetime import datetime
import pandas as pd

station_id = "VILD0"

start_multi = datetime(2020, 6, 18)
end_multi = datetime(2026, 3, 9, 23, 59)

df_multi_recovered = Hourly(
    station_id,
    start_multi,
    end_multi,
    timezone="Asia/Kolkata"
).fetch()

print("RETRIEVED DATASET")
print("=" * 55)
print("Shape:", df_multi_recovered.shape)
print("Start:", df_multi_recovered.index.min())
print("End:", df_multi_recovered.index.max())
print("Timezone:", df_multi_recovered.index.tz)
print("Duplicate timestamps:", df_multi_recovered.index.has_duplicates)
print("\nMissing values:")
print(df_multi_recovered.isna().sum())

# Save a separate recovery copy; do not overwrite existing data.
recovery_path = (
    "/content/drive/MyDrive/weather_anomaly_research_cleaned/"
    "outputs/VILD0_PipelineB_raw_recovered.csv"
)

df_multi_recovered.to_csv(recovery_path)

print("\nSaved recovery copy:", recovery_path)

ModuleNotFoundError: No module named 'meteostat'

In [17]:
%pip install -q meteostat==1.7.6

In [18]:
import meteostat
from meteostat import Hourly
import pandas as pd

print("Meteostat:", meteostat.__version__)
print("Pandas:", pd.__version__)
print("Hourly import: successful")

Meteostat: 1.7.6
Pandas: 2.2.3
Hourly import: successful


In [19]:
from meteostat import Hourly
from datetime import datetime

df_multi_recovered = Hourly(
    "VILD0",
    datetime(2020, 6, 18),
    datetime(2026, 3, 9, 23, 59),
    timezone="Asia/Kolkata"
).fetch()

print("Shape:", df_multi_recovered.shape)
print("Start:", df_multi_recovered.index.min())
print("End:", df_multi_recovered.index.max())

Shape: (45229, 11)
Start: 2020-06-18 11:30:00+05:30
End: 2026-03-09 23:30:00+05:30


In [20]:
import pandas as pd
import numpy as np

# 1. Validate recovered raw data
df = df_multi_recovered.copy()

assert isinstance(df.index, pd.DatetimeIndex)
assert not df.index.has_duplicates
assert df.index.is_monotonic_increasing
assert str(df.index.tz) == "Asia/Kolkata"

print("RAW DATA:", df.shape)

# 2. Reconstruct historical segment IDs
time_diff = df.index.to_series().diff()
segment_start = time_diff > pd.Timedelta(hours=1)
segment_id = segment_start.cumsum()

segment_summary = pd.DataFrame({
    "segment_id": segment_id,
    "timestamp": df.index
}).groupby("segment_id").agg(
    start=("timestamp", "min"),
    end=("timestamp", "max"),
    observations=("timestamp", "count")
)

# 3. Retain segments with at least 100 observations
major_segments = segment_summary[
    segment_summary["observations"] >= 100
].sort_values("observations", ascending=False)

major_segment_ids = major_segments.index

df_major = df.loc[
    segment_id.isin(major_segment_ids)
].copy()

df_major["segment_id"] = segment_id.loc[df_major.index]

# 4. Reconstruct historical model_df
base_features = [
    "temp", "dwpt", "rhum", "prcp", "wspd", "pres"
]

model_df = df_major[
    base_features + ["segment_id"]
].copy()

grouped = model_df.groupby("segment_id")["temp"]

model_df["temp_change_1h"] = grouped.diff()

model_df["temp_rolling_mean_6h"] = grouped.transform(
    lambda s: s.shift(1).rolling(
        window=6,
        min_periods=6
    ).mean()
)

model_df["temp_deviation_6h"] = (
    model_df["temp"] - model_df["temp_rolling_mean_6h"]
)

# 5. Reconstruct feature-ready X_multi
feature_columns = base_features + [
    "temp_change_1h",
    "temp_rolling_mean_6h",
    "temp_deviation_6h"
]

X_multi = model_df[feature_columns].copy()

X_multi = X_multi.dropna(
    subset=[
        "temp_change_1h",
        "temp_rolling_mean_6h",
        "temp_deviation_6h"
    ]
)

# 6. Chronological split
cutoff = pd.Timestamp(
    "2025-03-21 17:30:00",
    tz="Asia/Kolkata"
)

train_mask = X_multi.index < cutoff

X_train_multi = X_multi.loc[train_mask].copy()
X_test_multi = X_multi.loc[~train_mask].copy()

# 7. Training-only precipitation imputation
precip_median = X_train_multi["prcp"].median()

X_train_multi["prcp"] = (
    X_train_multi["prcp"].fillna(precip_median)
)

X_test_multi["prcp"] = (
    X_test_multi["prcp"].fillna(precip_median)
)

# 8. Historical baseline comparison
checks = {
    "Raw observations": (len(df), 45229),
    "Major segment observations": (len(df_major), 44879),
    "Feature-ready observations": (len(X_multi), 44849),
    "Training observations": (len(X_train_multi), 36434),
    "Testing observations": (len(X_test_multi), 8415),
    "Missing precipitation before imputation": (
        int(X_multi["prcp"].isna().sum()), 6
    ),
    "Training precipitation median": (precip_median, 0.0),
}

print("\n" + "=" * 75)
print("PIPELINE B RECONSTRUCTION AUDIT")
print("=" * 75)

for name, (actual, expected) in checks.items():
    passed = np.isclose(actual, expected)
    print(
        f"{name}: actual={actual} | expected={expected} | "
        f"{'PASS' if passed else 'DIFFERENCE'}"
    )

print("\nSelected segments:")
print(major_segments)

print("\nTraining period:",
      X_train_multi.index.min(), "to", X_train_multi.index.max())

print("Testing period:",
      X_test_multi.index.min(), "to", X_test_multi.index.max())

print("\nRemaining missing values:")
print(
    X_train_multi.isna().sum().sum()
    + X_test_multi.isna().sum().sum()
)

RAW DATA: (45229, 11)

PIPELINE B RECONSTRUCTION AUDIT
Raw observations: actual=45229 | expected=45229 | PASS
Major segment observations: actual=44879 | expected=44879 | PASS
Feature-ready observations: actual=44849 | expected=44849 | PASS
Training observations: actual=36434 | expected=36434 | PASS
Testing observations: actual=8415 | expected=8415 | PASS
Missing precipitation before imputation: actual=0 | expected=6 | DIFFERENCE
Training precipitation median: actual=0.0 | expected=0.0 | PASS

Selected segments:
                               start                       end  observations
segment_id                                                                  
76         2021-01-05 19:30:00+05:30 2024-08-28 05:30:00+05:30         31931
145        2025-03-21 17:30:00+05:30 2026-02-11 17:30:00+05:30          7849
103        2024-09-04 08:30:00+05:30 2024-12-10 11:30:00+05:30          2332
118        2024-12-13 19:30:00+05:30 2025-03-14 23:30:00+05:30          2189
154        2026-02-13

In [21]:
print("=" * 75)
print("PRECIPITATION MISSINGNESS INVESTIGATION")
print("=" * 75)

frames = {
    "Original recovered raw data": df_multi_recovered,
    "Selected major segments": df_major,
    "Historical model_df": model_df,
    "Feature-ready X_multi": X_multi,
    "Training partition": X_train_multi,
    "Testing partition": X_test_multi,
}

for name, frame in frames.items():
    print(f"\n{name}")
    print("-" * 55)

    if "prcp" in frame.columns:
        print("Rows:", len(frame))
        print("Missing precipitation:", frame["prcp"].isna().sum())
        print("Precipitation dtype:", frame["prcp"].dtype)

# Exact timestamps with missing precipitation
raw_missing = df_multi_recovered.index[
    df_multi_recovered["prcp"].isna()
]

major_missing = df_major.index[
    df_major["prcp"].isna()
]

model_missing = model_df.index[
    model_df["prcp"].isna()
]

feature_ready_missing = X_multi.index[
    X_multi["prcp"].isna()
]

print("\n" + "=" * 75)
print("MISSING TIMESTAMP COMPARISON")
print("=" * 75)

print("Raw missing timestamps:", len(raw_missing))
print("Major-segment missing timestamps:", len(major_missing))
print("Model missing timestamps:", len(model_missing))
print("Feature-ready missing timestamps:", len(feature_ready_missing))

print("\nRaw missing timestamps retained in major segments:")
print(major_missing.tolist())

print("\nMissing timestamps remaining after feature filtering:")
print(feature_ready_missing.tolist())

print("\nMissing timestamps excluded during feature filtering:")
print(model_missing.difference(X_multi.index).tolist())

PRECIPITATION MISSINGNESS INVESTIGATION

Original recovered raw data
-------------------------------------------------------
Rows: 45229
Missing precipitation: 356
Precipitation dtype: Float64

Selected major segments
-------------------------------------------------------
Rows: 44879
Missing precipitation: 6
Precipitation dtype: Float64

Historical model_df
-------------------------------------------------------
Rows: 44879
Missing precipitation: 6
Precipitation dtype: Float64

Feature-ready X_multi
-------------------------------------------------------
Rows: 44849
Missing precipitation: 0
Precipitation dtype: Float64

Training partition
-------------------------------------------------------
Rows: 36434
Missing precipitation: 0
Precipitation dtype: Float64

Testing partition
-------------------------------------------------------
Rows: 8415
Missing precipitation: 0
Precipitation dtype: Float64

MISSING TIMESTAMP COMPARISON
Raw missing timestamps: 356
Major-segment missing timestamps

In [22]:
from weather_research.audit import phase01_integrity_audit

phase01_results = phase01_integrity_audit(
    model_df=model_df,
    x_multi=X_multi,
    x_train=X_train_multi,
    x_test=X_test_multi,
    split_boundary="2025-03-21 17:30:00+05:30"
)

print("=" * 75)
print("PHASE 01 INTEGRITY AUDIT")
print("=" * 75)

print("\nOVERALL RESULT:")
print("Phase 01 passed:", phase01_results["phase01_passed"])

reports = [
    "feature_report",
    "per_segment_feature_report",
    "index_report",
    "split_report",
    "model_matrix_report",
    "split_value_report",
    "physical_summary",
    "precipitation_imputation",
    "lag1_autocorrelation",
]

for name in reports:
    print("\n" + "=" * 75)
    print(name.upper())
    print("=" * 75)

    result = phase01_results[name]

    if isinstance(result, pd.DataFrame):
        display(result)
    else:
        print(result)

print("\nTRAIN END:", phase01_results["train_end"])
print("TEST START:", phase01_results["test_start"])
print("TRAIN-TEST GAP:", phase01_results["train_test_time_gap"])

PHASE 01 INTEGRITY AUDIT

OVERALL RESULT:
Phase 01 passed: True

FEATURE_REPORT


,feature,mismatch_count,missingness_mismatch_count,max_absolute_difference,passed
0,temp_change_1h,0,0,0.0,True
1,temp_rolling_mean_6h,0,0,0.0,True
2,temp_deviation_6h,0,0,0.0,True



PER_SEGMENT_FEATURE_REPORT


,feature,mismatch_count,missingness_mismatch_count,max_absolute_difference,passed,segment_id
0,temp_change_1h,0,0,0.0,True,76
1,temp_rolling_mean_6h,0,0,0.0,True,76
2,temp_deviation_6h,0,0,0.0,True,76
3,temp_change_1h,0,0,0.0,True,103
4,temp_rolling_mean_6h,0,0,0.0,True,103
5,temp_deviation_6h,0,0,0.0,True,103
6,temp_change_1h,0,0,0.0,True,118
7,temp_rolling_mean_6h,0,0,0.0,True,118
8,temp_deviation_6h,0,0,0.0,True,118
9,temp_change_1h,0,0,0.0,True,145



INDEX_REPORT


,check,expected_rows,actual_rows,missing_from_X_multi,unexpected_in_X_multi,passed
0,expected feature-ready index equals X_multi index,44849,44849,0,0,True


... (9 outputs left out)

,timestamp,partition,training_median,imputed_value,matches_training_median



LAG1_AUTOCORRELATION


,variable,adjacent_hour_pairs,lag1_correlation,interpretation
0,temp,44874,0.985950,"descriptive; temporal dependence, not iid infe..."
1,dwpt,44874,0.989599,"descriptive; temporal dependence, not iid infe..."
2,rhum,44874,0.969035,"descriptive; temporal dependence, not iid infe..."
3,prcp,44868,0.470071,"descriptive; temporal dependence, not iid infe..."
4,wspd,44874,0.828282,"descriptive; temporal dependence, not iid infe..."
5,pres,44874,0.996428,"descriptive; temporal dependence, not iid infe..."



TRAIN END: 2025-03-14 23:30:00+05:30
TEST START: 2025-03-21 23:30:00+05:30
TRAIN-TEST GAP: 7 days 00:00:00


In [23]:
from pathlib import Path
import hashlib
import json
import pandas as pd

# Create a separate directory for verified Phase 01 data
OUTPUT_DIR = Path(
    "/content/drive/MyDrive/weather_anomaly_research_cleaned/"
    "verified_phase01"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Save the verified datasets
datasets = {
    "model_df": model_df,
    "X_multi": X_multi,
    "X_train_multi": X_train_multi,
    "X_test_multi": X_test_multi,
}

manifest = {
    "phase": "Phase 01 - Historical Pipeline B Integrity",
    "audit_passed": bool(phase01_results["phase01_passed"]),
    "split_cutoff": "2025-03-21 17:30:00+05:30",
    "train_end": str(X_train_multi.index.max()),
    "test_start": str(X_test_multi.index.min()),
    "datasets": {}
}

for name, frame in datasets.items():
    path = OUTPUT_DIR / f"{name}.csv"

    frame.to_csv(path, index=True)

    sha256 = hashlib.sha256(path.read_bytes()).hexdigest()

    manifest["datasets"][name] = {
        "rows": len(frame),
        "columns": list(frame.columns),
        "sha256": sha256,
        "missing_values": int(frame.isna().sum().sum()),
        "start": str(frame.index.min()),
        "end": str(frame.index.max())
    }

manifest_path = OUTPUT_DIR / "phase01_manifest.json"

manifest_path.write_text(
    json.dumps(manifest, indent=4),
    encoding="utf-8"
)

print("PHASE 01 SNAPSHOT SAVED")
print("Location:", OUTPUT_DIR)
print("Audit passed:", manifest["audit_passed"])

for name, details in manifest["datasets"].items():
    print(
        name,
        "| Rows:", details["rows"],
        "| Missing:", details["missing_values"],
        "| SHA256:", details["sha256"][:16]
    )

PHASE 01 SNAPSHOT SAVED
Location: /content/drive/MyDrive/weather_anomaly_research_cleaned/verified_phase01
Audit passed: True
model_df | Rows: 44879 | Missing: 71 | SHA256: 7921f8da901f48ac
X_multi | Rows: 44849 | Missing: 0 | SHA256: 8856aa75ac43c47b
X_train_multi | Rows: 36434 | Missing: 0 | SHA256: 5d03f82bde4fb126
X_test_multi | Rows: 8415 | Missing: 0 | SHA256: 90563fcf4f06a74f


In [24]:

from pathlib import Path
import os
import json
import hashlib
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler

from weather_research.models import (
    default_model_configs,
    fit_predict_configurations,
    stability_summary
)

# --------------------------------------------------
# 1. VERIFY THE SAVED SNAPSHOT
# --------------------------------------------------

ROOT = Path(
    "/content/drive/MyDrive/weather_anomaly_research_cleaned"
)

SNAPSHOT_DIR = ROOT / "verified_phase01"

manifest_path = SNAPSHOT_DIR / "phase01_manifest.json"

assert manifest_path.exists(), "Phase 01 manifest not found."

manifest = json.loads(manifest_path.read_text())

assert manifest["audit_passed"] is True, (
    "Phase 01 audit was not marked as passed."
)

def verify_file(name):
    path = SNAPSHOT_DIR / f"{name}.csv"

    assert path.exists(), f"Missing snapshot: {path}"

    actual_hash = hashlib.sha256(path.read_bytes()).hexdigest()
    expected_hash = manifest["datasets"][name]["sha256"]

    assert actual_hash == expected_hash, (
        f"SHA256 mismatch for {name}"
    )

    print(f"{name}: SHA256 verified")

for name in [
    "X_train_multi",
    "X_test_multi"
]:
    verify_file(name)


# --------------------------------------------------
# 2. LOAD VERIFIED TRAINING AND TESTING DATA
# --------------------------------------------------

def load_snapshot(name):
    path = SNAPSHOT_DIR / f"{name}.csv"

    frame = pd.read_csv(path, index_col=0)

    frame.index = pd.DatetimeIndex(
        pd.to_datetime(frame.index)
    )

    return frame

X_train_seasonal = load_snapshot("X_train_multi")
X_test_seasonal = load_snapshot("X_test_multi")

assert len(X_train_seasonal) == 36434
assert len(X_test_seasonal) == 8415

assert X_train_seasonal.index.is_unique
assert X_test_seasonal.index.is_unique

assert X_train_seasonal.index.is_monotonic_increasing
assert X_test_seasonal.index.is_monotonic_increasing

assert X_train_seasonal.index.max() < X_test_seasonal.index.min()

assert X_train_seasonal.columns.equals(
    X_test_seasonal.columns
)

assert X_train_seasonal.isna().sum().sum() == 0
assert X_test_seasonal.isna().sum().sum() == 0

print("\nVerified input shapes:")
print("Training:", X_train_seasonal.shape)
print("Testing:", X_test_seasonal.shape)


# --------------------------------------------------
# 3. RECONSTRUCT HISTORICAL SEASONAL FEATURES
# --------------------------------------------------

def add_month_features(frame):
    result = frame.copy()

    month = result.index.month

    result["month_sin"] = np.sin(
        2 * np.pi * month / 12
    )

    result["month_cos"] = np.cos(
        2 * np.pi * month / 12
    )

    return result

X_train_seasonal = add_month_features(
    X_train_seasonal
)

X_test_seasonal = add_month_features(
    X_test_seasonal
)

assert X_train_seasonal.shape == (36434, 11)
assert X_test_seasonal.shape == (8415, 11)

assert X_train_seasonal.columns.equals(
    X_test_seasonal.columns
)

assert np.isfinite(
    X_train_seasonal.to_numpy(dtype=float)
).all()

assert np.isfinite(
    X_test_seasonal.to_numpy(dtype=float)
).all()

print("\nSeasonal feature preparation passed.")
print("Features:", list(X_train_seasonal.columns))


# --------------------------------------------------
# 4. SCALE USING TRAINING DATA ONLY
# --------------------------------------------------

scaler = StandardScaler()

X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train_seasonal),
    index=X_train_seasonal.index,
    columns=X_train_seasonal.columns
)

X_test_scaled = pd.DataFrame(
    scaler.transform(X_test_seasonal),
    index=X_test_seasonal.index,
    columns=X_test_seasonal.columns
)

assert X_train_scaled.index.equals(
    X_train_seasonal.index
)

assert X_test_scaled.index.equals(
    X_test_seasonal.index
)

print("\nTraining-only scaling passed.")


# --------------------------------------------------
# 5. VERIFY THE 12 CONFIGURATIONS
# --------------------------------------------------

configs = default_model_configs(random_seed=42)

assert len(configs) == 12

from collections import Counter

family_counts = Counter(
    config["family"] for config in configs
)

assert family_counts == {
    "IsolationForest": 4,
    "LocalOutlierFactor": 4,
    "OneClassSVM": 4
}

print("\nConfigurations:")
for config in configs:
    print(config)


# --------------------------------------------------
# 6. TRAIN AND PREDICT
# --------------------------------------------------

predictions = fit_predict_configurations(
    x_train_unscaled=X_train_seasonal,
    x_test_unscaled=X_test_seasonal,
    x_train_scaled=X_train_scaled,
    x_test_scaled=X_test_scaled,
    configs=configs,
    n_estimators=200
)

assert predictions.shape[0] == 12 * len(X_test_seasonal)

assert predictions.index.is_monotonic_increasing

assert predictions["prediction"].isin([-1, 1]).all()

assert not predictions["is_candidate"].isna().any()

print("\nModel execution completed.")
print("Prediction rows:", len(predictions))


# --------------------------------------------------
# 7. CONFIGURATION-LEVEL RESULTS
# --------------------------------------------------

configuration_results = (
    predictions
    .groupby(["configuration", "family", "parameter_value"])
    .agg(
        test_observations=("is_candidate", "size"),
        anomalies=("is_candidate", "sum")
    )
    .reset_index()
)

configuration_results["anomaly_percentage"] = (
    configuration_results["anomalies"]
    / configuration_results["test_observations"]
    * 100
)

display(
    configuration_results.round(3)
)


# --------------------------------------------------
# 8. MODEL STABILITY AND AGREEMENT
# --------------------------------------------------

stability, jaccard = stability_summary(
    predictions
)

assert len(stability) == len(X_test_seasonal)

print("\nConfiguration vote distribution:")
display(
    stability["configuration_votes"]
    .value_counts()
    .sort_index()
    .rename_axis("number_of_anomaly_votes")
    .to_frame("observation_count")
)

print("\nJaccard similarity:")
display(jaccard.round(3))


# --------------------------------------------------
# 9. SAVE PHASE 03 RESULTS
# --------------------------------------------------

OUTPUT_DIR = ROOT / "phase03_model_revalidation"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

predictions.to_csv(
    OUTPUT_DIR / "model_predictions.csv"
)

configuration_results.to_csv(
    OUTPUT_DIR / "configuration_results.csv",
    index=False
)

stability.to_csv(
    OUTPUT_DIR / "stability_results.csv"
)

jaccard.to_csv(
    OUTPUT_DIR / "jaccard_similarity.csv"
)

print("\nPHASE 03 MODEL REVALIDATION COMPLETED")
print("Saved to:", OUTPUT_DIR)


X_train_multi: SHA256 verified
X_test_multi: SHA256 verified

Verified input shapes:
Training: (36434, 9)
Testing: (8415, 9)

Seasonal feature preparation passed.
Features: ['temp', 'dwpt', 'rhum', 'prcp', 'wspd', 'pres', 'temp_change_1h', 'temp_rolling_mean_6h', 'temp_deviation_6h', 'month_sin', 'month_cos']

Training-only scaling passed.

Configurations:
{'family': 'IsolationForest', 'name': 'IF_0.01', 'parameter': 'contamination', 'value': 0.01, 'random_seed': 42}
{'family': 'IsolationForest', 'name': 'IF_0.02', 'parameter': 'contamination', 'value': 0.02, 'random_seed': 42}
{'family': 'IsolationForest', 'name': 'IF_0.03', 'parameter': 'contamination', 'value': 0.03, 'random_seed': 42}
{'family': 'IsolationForest', 'name': 'IF_0.05', 'parameter': 'contamination', 'value': 0.05, 'random_seed': 42}
{'family': 'LocalOutlierFactor', 'name': 'LOF_10', 'parameter': 'n_neighbors', 'value': 10, 'random_seed': 42}
{'family': 'LocalOutlierFactor', 'name': 'LOF_20', 'parameter': 'n_neighbors',

,configuration,family,parameter_value,test_observations,anomalies,anomaly_percentage
0,IF_0.01,IsolationForest,0.01,8415,94,1.117
1,IF_0.02,IsolationForest,0.02,8415,189,2.246
2,IF_0.03,IsolationForest,0.03,8415,270,3.209
3,IF_0.05,IsolationForest,0.05,8415,521,6.191
4,LOF_10,LocalOutlierFactor,10.00,8415,531,6.310
5,LOF_20,LocalOutlierFactor,20.00,8415,386,4.587
6,LOF_30,LocalOutlierFactor,30.00,8415,326,3.874
7,LOF_50,LocalOutlierFactor,50.00,8415,263,3.125
8,SVM_0.01,OneClassSVM,0.01,8415,157,1.866
9,SVM_0.02,OneClassSVM,0.02,8415,279,3.316



Configuration vote distribution:


,observation_count
number_of_anomaly_votes,
0,7204
1,416
2,197
3,166
4,164
5,54
6,50
7,41
8,53



Jaccard similarity:


,IF_0.01,IF_0.02,IF_0.03,IF_0.05,LOF_10,LOF_20,LOF_30,LOF_50,SVM_0.01,SVM_0.02,SVM_0.03,SVM_0.05
IF_0.01,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
IF_0.02,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
IF_0.03,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
IF_0.05,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
LOF_10,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
LOF_20,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
LOF_30,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
LOF_50,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
SVM_0.01,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
SVM_0.02,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0



PHASE 03 MODEL REVALIDATION COMPLETED
Saved to: /content/drive/MyDrive/weather_anomaly_research_cleaned/phase03_model_revalidation


In [27]:

import numpy as np
import pandas as pd

# --------------------------------------------------
# 1. Preserve timestamps as an explicit column
# --------------------------------------------------

prediction_data = predictions.copy()

prediction_data = prediction_data.assign(
    timestamp=predictions.index
)

# Check duplicate timestamp/configuration combinations
duplicate_count = prediction_data.duplicated(
    subset=["timestamp", "configuration"]
).sum()

print("Duplicate timestamp/configuration rows:", duplicate_count)

assert duplicate_count == 0

# --------------------------------------------------
# 2. Build timestamp x configuration matrix
# --------------------------------------------------

wide = prediction_data.pivot(
    index="timestamp",
    columns="configuration",
    values="is_candidate"
).astype(bool)

assert wide.shape == (8415, 12)
assert not wide.isna().any().any()

print("\nPrediction matrix:", wide.shape)

# --------------------------------------------------
# 3. Independently calculate Jaccard similarity
# --------------------------------------------------

manual_jaccard = pd.DataFrame(
    index=wide.columns,
    columns=wide.columns,
    dtype=float
)

for a in wide.columns:
    for b in wide.columns:

        intersection = (wide[a] & wide[b]).sum()
        union = (wide[a] | wide[b]).sum()

        manual_jaccard.loc[a, b] = (
            intersection / union if union > 0 else np.nan
        )

print("\nINDEPENDENT JACCARD MATRIX")
display(manual_jaccard.round(4))

# --------------------------------------------------
# 4. Compare with package output
# --------------------------------------------------

print("\nPACKAGE JACCARD MATRIX")
display(jaccard.round(4))

print(
    "\nMatrices identical:",
    np.allclose(
        jaccard.to_numpy(dtype=float),
        manual_jaccard.to_numpy(dtype=float),
        equal_nan=True
    )
)

# --------------------------------------------------
# 5. Check distinct anomaly sets
# --------------------------------------------------

signatures = {
    col: frozenset(wide.index[wide[col]])
    for col in wide.columns
}

print("\nUnique anomaly sets:", len(set(signatures.values())))
print("Total configurations:", len(signatures))

# --------------------------------------------------
# 6. Cross-family comparisons
# --------------------------------------------------

metadata = (
    prediction_data[["configuration", "family"]]
    .drop_duplicates()
    .set_index("configuration")
)

cross_family = []

for i, a in enumerate(wide.columns):
    for b in wide.columns[i + 1:]:

        family_a = metadata.loc[a, "family"]
        family_b = metadata.loc[b, "family"]

        if family_a != family_b:
            cross_family.append({
                "configuration_a": a,
                "configuration_b": b,
                "family_a": family_a,
                "family_b": family_b,
                "jaccard": manual_jaccard.loc[a, b]
            })

cross_family = pd.DataFrame(cross_family)

print("\nLOWEST 15 CROSS-FAMILY SIMILARITIES")
display(
    cross_family.sort_values("jaccard").head(15)
)



Duplicate timestamp/configuration rows: 0

Prediction matrix: (8415, 12)

INDEPENDENT JACCARD MATRIX


configuration,IF_0.01,IF_0.02,IF_0.03,IF_0.05,LOF_10,LOF_20,LOF_30,LOF_50,SVM_0.01,SVM_0.02,SVM_0.03,SVM_0.05
configuration,,,,,,,,,,,,
IF_0.01,1.0000,0.4974,0.3481,0.1804,0.0504,0.0619,0.0660,0.0721,0.3568,0.2862,0.2370,0.1615
IF_0.02,0.4974,1.0000,0.7000,0.3628,0.0827,0.0890,0.0911,0.1024,0.3730,0.4012,0.3936,0.2862
IF_0.03,0.3481,0.7000,1.0000,0.5182,0.0928,0.0970,0.0976,0.1081,0.3261,0.3969,0.4214,0.3477
IF_0.05,0.1804,0.3628,0.5182,1.0000,0.1050,0.0954,0.0943,0.1042,0.2129,0.3180,0.3877,0.4087
LOF_10,0.0504,0.0827,0.0928,0.1050,1.0000,0.5893,0.4476,0.3124,0.1223,0.1376,0.1429,0.1485
LOF_20,0.0619,0.0890,0.0970,0.0954,0.5893,1.0000,0.6912,0.4717,0.1578,0.1687,0.1621,0.1592
LOF_30,0.0660,0.0911,0.0976,0.0943,0.4476,0.6912,1.0000,0.6361,0.1780,0.1748,0.1647,0.1641
LOF_50,0.0721,0.1024,0.1081,0.1042,0.3124,0.4717,0.6361,1.0000,0.2034,0.1912,0.1817,0.1704
SVM_0.01,0.3568,0.3730,0.3261,0.2129,0.1223,0.1578,0.1780,0.2034,1.0000,0.5627,0.4121,0.2730



PACKAGE JACCARD MATRIX


,IF_0.01,IF_0.02,IF_0.03,IF_0.05,LOF_10,LOF_20,LOF_30,LOF_50,SVM_0.01,SVM_0.02,SVM_0.03,SVM_0.05
IF_0.01,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
IF_0.02,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
IF_0.03,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
IF_0.05,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
LOF_10,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
LOF_20,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
LOF_30,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
LOF_50,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
SVM_0.01,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
SVM_0.02,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0



Matrices identical: False

Unique anomaly sets: 12
Total configurations: 12

LOWEST 15 CROSS-FAMILY SIMILARITIES


,configuration_a,configuration_b,family_a,family_b,jaccard
0,IF_0.01,LOF_10,IsolationForest,LocalOutlierFactor,0.050420
1,IF_0.01,LOF_20,IsolationForest,LocalOutlierFactor,0.061947
2,IF_0.01,LOF_30,IsolationForest,LocalOutlierFactor,0.065990
3,IF_0.01,LOF_50,IsolationForest,LocalOutlierFactor,0.072072
8,IF_0.02,LOF_10,IsolationForest,LocalOutlierFactor,0.082707
9,IF_0.02,LOF_20,IsolationForest,LocalOutlierFactor,0.089015
10,IF_0.02,LOF_30,IsolationForest,LocalOutlierFactor,0.091102
16,IF_0.03,LOF_10,IsolationForest,LocalOutlierFactor,0.092769
26,IF_0.05,LOF_30,IsolationForest,LocalOutlierFactor,0.094315
25,IF_0.05,LOF_20,IsolationForest,LocalOutlierFactor,0.095411


In [28]:

import inspect
import pandas as pd
import numpy as np

# Independently calculate votes at every timestamp
manual_votes = wide.sum(axis=1).astype(int)

# Align package vote counts by timestamp
package_votes = stability["configuration_votes"].reindex(
    manual_votes.index
)

assert package_votes.notna().all(), (
    "Package stability results are missing timestamps."
)

vote_comparison = pd.DataFrame({
    "manual_votes": manual_votes,
    "package_votes": package_votes.astype(int)
})

vote_comparison["matches"] = (
    vote_comparison["manual_votes"]
    == vote_comparison["package_votes"]
)

print("TIMESTAMP-LEVEL VOTE VERIFICATION")
print("Matching timestamps:", vote_comparison["matches"].sum())
print("Mismatching timestamps:", (~vote_comparison["matches"]).sum())

display(
    vote_comparison.loc[~vote_comparison["matches"]].head(20)
)

# Independent vote distribution
print("\nINDEPENDENT VOTE DISTRIBUTION")
display(
    manual_votes.value_counts()
    .sort_index()
    .rename_axis("anomaly_votes")
    .to_frame("observations")
)

# Inspect the actual package function to locate the defect
print("\nSTABILITY_SUMMARY SOURCE CODE")
print(inspect.getsource(stability_summary))


TIMESTAMP-LEVEL VOTE VERIFICATION
Matching timestamps: 8415
Mismatching timestamps: 0


,manual_votes,package_votes,matches
timestamp,,,



INDEPENDENT VOTE DISTRIBUTION


,observations
anomaly_votes,
0,7204
1,416
2,197
3,166
4,164
5,54
6,50
7,41
8,53



STABILITY_SUMMARY SOURCE CODE
def stability_summary(predictions: pd.DataFrame):
    required = {"configuration", "family", "is_candidate"}
    if not required.issubset(predictions.columns):
        raise KeyError(f"predictions need columns: {sorted(required)}")
    long = predictions.rename_axis("timestamp").reset_index()
    if long.duplicated(["timestamp", "configuration"]).any():
        raise ValueError("duplicate timestamp/configuration predictions")
    if long["is_candidate"].isna().any():
        raise ValueError("missing predictions cannot be treated as normal votes")
    if long[["configuration","family"]].isna().any().any():
        raise ValueError("configuration and family metadata cannot be missing")
    if not long["is_candidate"].isin([True,False,0,1]).all():
        raise ValueError("is_candidate must contain boolean/0-1 values")
    per_config=long.groupby("configuration")["timestamp"].agg(lambda x:frozenset(x))
    reference=per_config.iloc[0]
    if any(times!=refe

In [29]:
manual_jaccard.to_csv(
    OUTPUT_DIR / "jaccard_similarity_verified.csv"
)

In [30]:
import importlib
import weather_research.models as models

importlib.reload(models)

corrected_stability, corrected_jaccard = (
    models.stability_summary(predictions)
)

In [31]:
# Verify all timestamp-level vote counts
assert corrected_stability.index.equals(wide.index)

assert np.array_equal(
    corrected_stability["configuration_votes"].to_numpy(),
    manual_votes.to_numpy()
)

# Verify Jaccard matrix
assert np.allclose(
    corrected_jaccard.to_numpy(dtype=float),
    manual_jaccard.to_numpy(dtype=float),
    equal_nan=True
)

# Verify that all 12 configurations have distinct anomaly sets
unique_sets = {
    frozenset(wide.index[wide[c]])
    for c in wide.columns
}

assert len(unique_sets) == 12

print("All Phase 03 stability checks passed.")

display(corrected_jaccard.round(4))

All Phase 03 stability checks passed.


,IF_0.01,IF_0.02,IF_0.03,IF_0.05,LOF_10,LOF_20,LOF_30,LOF_50,SVM_0.01,SVM_0.02,SVM_0.03,SVM_0.05
IF_0.01,1.0000,0.4974,0.3481,0.1804,0.0504,0.0619,0.0660,0.0721,0.3568,0.2862,0.2370,0.1615
IF_0.02,0.4974,1.0000,0.7000,0.3628,0.0827,0.0890,0.0911,0.1024,0.3730,0.4012,0.3936,0.2862
IF_0.03,0.3481,0.7000,1.0000,0.5182,0.0928,0.0970,0.0976,0.1081,0.3261,0.3969,0.4214,0.3477
IF_0.05,0.1804,0.3628,0.5182,1.0000,0.1050,0.0954,0.0943,0.1042,0.2129,0.3180,0.3877,0.4087
LOF_10,0.0504,0.0827,0.0928,0.1050,1.0000,0.5893,0.4476,0.3124,0.1223,0.1376,0.1429,0.1485
LOF_20,0.0619,0.0890,0.0970,0.0954,0.5893,1.0000,0.6912,0.4717,0.1578,0.1687,0.1621,0.1592
LOF_30,0.0660,0.0911,0.0976,0.0943,0.4476,0.6912,1.0000,0.6361,0.1780,0.1748,0.1647,0.1641
LOF_50,0.0721,0.1024,0.1081,0.1042,0.3124,0.4717,0.6361,1.0000,0.2034,0.1912,0.1817,0.1704
SVM_0.01,0.3568,0.3730,0.3261,0.2129,0.1223,0.1578,0.1780,0.2034,1.0000,0.5627,0.4121,0.2730
SVM_0.02,0.2862,0.4012,0.3969,0.3180,0.1376,0.1687,0.1748,0.1912,0.5627,1.0000,0.7323,0.4852


In [32]:
corrected_jaccard.to_csv(
    OUTPUT_DIR / "jaccard_similarity_verified.csv"
)

corrected_stability.to_csv(
    OUTPUT_DIR / "stability_results_verified.csv"
)

print("Verified results saved.")

Verified results saved.


In [34]:
print("Corrected stability:")
print("Shape:", corrected_stability.shape)
print("Index type:", type(corrected_stability.index))
print("Timezone:", corrected_stability.index.tz)
print("First:", corrected_stability.index.min())
print("Last:", corrected_stability.index.max())
print("Sorted:", corrected_stability.index.is_monotonic_increasing)
print("Duplicates:", corrected_stability.index.has_duplicates)

print("\nX_test_multi:")
print("Shape:", X_test_multi.shape)
print("Index type:", type(X_test_multi.index))
print("Timezone:", X_test_multi.index.tz)
print("First:", X_test_multi.index.min())
print("Last:", X_test_multi.index.max())
print("Sorted:", X_test_multi.index.is_monotonic_increasing)
print("Duplicates:", X_test_multi.index.has_duplicates)

print("\nIndex comparison:")
print(
    "Same timestamp sets:",
    set(corrected_stability.index) == set(X_test_multi.index)
)
print(
    "Stability only:",
    len(corrected_stability.index.difference(X_test_multi.index))
)
print(
    "Test only:",
    len(X_test_multi.index.difference(corrected_stability.index))
)

Corrected stability:
Shape: (8415, 21)
Index type: <class 'pandas.core.indexes.datetimes.DatetimeIndex'>
Timezone: UTC+05:30
First: 2025-03-21 23:30:00+05:30
Last: 2026-03-09 23:30:00+05:30
Sorted: True
Duplicates: False

X_test_multi:
Shape: (8415, 9)
Index type: <class 'pandas.core.indexes.datetimes.DatetimeIndex'>
Timezone: Asia/Kolkata
First: 2025-03-21 23:30:00+05:30
Last: 2026-03-09 23:30:00+05:30
Sorted: True
Duplicates: False

Index comparison:
Same timestamp sets: True
Stability only: 0
Test only: 0


In [35]:
print("Predictions shape:", predictions.shape)
print("Prediction timestamps:", predictions.index.nunique())

print(
    "Prediction and stability timestamp sets match:",
    set(predictions.index) == set(corrected_stability.index)
)

print(
    "Stability and test timestamp sets match:",
    set(corrected_stability.index) == set(X_test_multi.index)
)

Predictions shape: (100980, 6)
Prediction timestamps: 8415
Prediction and stability timestamp sets match: True
Stability and test timestamp sets match: True


In [36]:
corrected_stability_aligned = corrected_stability.copy()

corrected_stability_aligned.index = (
    corrected_stability_aligned.index.tz_convert("Asia/Kolkata")
)

assert corrected_stability_aligned.index.equals(
    X_test_multi.index
)

print("Timestamp alignment verified.")

Timestamp alignment verified.


In [37]:
test_segments = model_df.loc[
    X_test_multi.index, "segment_id"
]

assert test_segments.notna().all()
assert test_segments.index.equals(X_test_multi.index)

print("Segment IDs recovered:", len(test_segments))
print("Unique test segments:", test_segments.nunique())

Segment IDs recovered: 8415
Unique test segments: 2


In [38]:
candidate_mask = (
    corrected_stability_aligned["configuration_votes"] >= 8
)

print("Candidate timestamps:", int(candidate_mask.sum()))

assert candidate_mask.sum() == 123

Candidate timestamps: 123


In [39]:
candidate_mask = (
    corrected_stability_aligned["configuration_votes"] >= 8
)

print("Candidate timestamps:", int(candidate_mask.sum()))

assert candidate_mask.sum() == 123

Candidate timestamps: 123


In [41]:
def build_temporal_episodes(
    candidate_mask,
    segment_ids,
    stability,
    max_gap_hours=1
):
    if not candidate_mask.index.equals(segment_ids.index):
        raise ValueError("Candidate and segment indexes differ.")

    if not candidate_mask.index.equals(stability.index):
        raise ValueError("Candidate and stability indexes differ.")

    candidates = pd.DataFrame({
        "segment_id": segment_ids,
        "configuration_votes": stability[
            "configuration_votes"
        ]
    }).loc[candidate_mask].copy()

    if candidates.empty:
        return pd.DataFrame()

    time_gap = candidates.index.to_series().diff()

    segment_changed = (
        candidates["segment_id"].ne(
            candidates["segment_id"].shift()
        )
    )

    new_episode = (
        time_gap > pd.Timedelta(hours=max_gap_hours)
    ) | segment_changed

    candidates["episode_id"] = new_episode.cumsum()

    episodes = (
        candidates.groupby("episode_id")
        .agg(
            segment_id=("segment_id", "first"),
            start=("configuration_votes", lambda x: x.index.min()),
            end=("configuration_votes", lambda x: x.index.max()),
            observations=("configuration_votes", "size"),
            mean_votes=("configuration_votes", "mean"),
            max_votes=("configuration_votes", "max"),
            min_votes=("configuration_votes", "min")
        )
        .reset_index()
    )

    episodes["duration_hours"] = (
        episodes["end"] - episodes["start"]
    ).dt.total_seconds() / 3600

    return episodes

In [42]:
episode_results = {}

for gap in [1, 2, 3]:
    episodes = build_temporal_episodes(
        candidate_mask=candidate_mask,
        segment_ids=test_segments,
        stability=corrected_stability_aligned,
        max_gap_hours=gap
    )

    episode_results[gap] = episodes

    print(f"\nMaximum gap: {gap} hour(s)")
    print("Episodes:", len(episodes))
    print(
        "Singleton episodes:",
        (episodes["observations"] == 1).sum()
    )
    print(
        "Multi-observation episodes:",
        (episodes["observations"] > 1).sum()
    )

    display(episodes.head(10))


Maximum gap: 1 hour(s)
Episodes: 81
Singleton episodes: 56
Multi-observation episodes: 25


,episode_id,segment_id,start,end,observations,mean_votes,max_votes,min_votes,duration_hours
0,1,145,2025-03-28 15:30:00+05:30,2025-03-28 17:30:00+05:30,3,8.666667,9,8,2.0
1,2,145,2025-03-29 09:30:00+05:30,2025-03-29 09:30:00+05:30,1,12.000000,12,12,0.0
2,3,145,2025-03-29 14:30:00+05:30,2025-03-29 14:30:00+05:30,1,10.000000,10,10,0.0
3,4,145,2025-03-30 09:30:00+05:30,2025-03-30 09:30:00+05:30,1,12.000000,12,12,0.0
4,5,145,2025-03-31 12:30:00+05:30,2025-03-31 13:30:00+05:30,2,8.000000,8,8,1.0
5,6,145,2025-04-02 09:30:00+05:30,2025-04-02 09:30:00+05:30,1,10.000000,10,10,0.0
6,7,145,2025-04-08 13:30:00+05:30,2025-04-08 13:30:00+05:30,1,10.000000,10,10,0.0
7,8,145,2025-04-11 12:30:00+05:30,2025-04-11 12:30:00+05:30,1,11.000000,11,11,0.0
8,9,145,2025-04-18 11:30:00+05:30,2025-04-18 11:30:00+05:30,1,11.000000,11,11,0.0
9,10,145,2025-04-18 18:30:00+05:30,2025-04-18 18:30:00+05:30,1,10.000000,10,10,0.0



Maximum gap: 2 hour(s)
Episodes: 75
Singleton episodes: 51
Multi-observation episodes: 24


,episode_id,segment_id,start,end,observations,mean_votes,max_votes,min_votes,duration_hours
0,1,145,2025-03-28 15:30:00+05:30,2025-03-28 17:30:00+05:30,3,8.666667,9,8,2.0
1,2,145,2025-03-29 09:30:00+05:30,2025-03-29 09:30:00+05:30,1,12.000000,12,12,0.0
2,3,145,2025-03-29 14:30:00+05:30,2025-03-29 14:30:00+05:30,1,10.000000,10,10,0.0
3,4,145,2025-03-30 09:30:00+05:30,2025-03-30 09:30:00+05:30,1,12.000000,12,12,0.0
4,5,145,2025-03-31 12:30:00+05:30,2025-03-31 13:30:00+05:30,2,8.000000,8,8,1.0
5,6,145,2025-04-02 09:30:00+05:30,2025-04-02 09:30:00+05:30,1,10.000000,10,10,0.0
6,7,145,2025-04-08 13:30:00+05:30,2025-04-08 13:30:00+05:30,1,10.000000,10,10,0.0
7,8,145,2025-04-11 12:30:00+05:30,2025-04-11 12:30:00+05:30,1,11.000000,11,11,0.0
8,9,145,2025-04-18 11:30:00+05:30,2025-04-18 11:30:00+05:30,1,11.000000,11,11,0.0
9,10,145,2025-04-18 18:30:00+05:30,2025-04-18 18:30:00+05:30,1,10.000000,10,10,0.0



Maximum gap: 3 hour(s)
Episodes: 70
Singleton episodes: 47
Multi-observation episodes: 23


,episode_id,segment_id,start,end,observations,mean_votes,max_votes,min_votes,duration_hours
0,1,145,2025-03-28 15:30:00+05:30,2025-03-28 17:30:00+05:30,3,8.666667,9,8,2.0
1,2,145,2025-03-29 09:30:00+05:30,2025-03-29 09:30:00+05:30,1,12.000000,12,12,0.0
2,3,145,2025-03-29 14:30:00+05:30,2025-03-29 14:30:00+05:30,1,10.000000,10,10,0.0
3,4,145,2025-03-30 09:30:00+05:30,2025-03-30 09:30:00+05:30,1,12.000000,12,12,0.0
4,5,145,2025-03-31 12:30:00+05:30,2025-03-31 13:30:00+05:30,2,8.000000,8,8,1.0
5,6,145,2025-04-02 09:30:00+05:30,2025-04-02 09:30:00+05:30,1,10.000000,10,10,0.0
6,7,145,2025-04-08 13:30:00+05:30,2025-04-08 13:30:00+05:30,1,10.000000,10,10,0.0
7,8,145,2025-04-11 12:30:00+05:30,2025-04-11 12:30:00+05:30,1,11.000000,11,11,0.0
8,9,145,2025-04-18 11:30:00+05:30,2025-04-18 11:30:00+05:30,1,11.000000,11,11,0.0
9,10,145,2025-04-18 18:30:00+05:30,2025-04-18 18:30:00+05:30,1,10.000000,10,10,0.0


In [43]:
sensitivity_summary = pd.DataFrame([
    {
        "max_gap_hours": gap,
        "episode_count": len(episodes),
        "singleton_episodes": int(
            (episodes["observations"] == 1).sum()
        ),
        "multi_observation_episodes": int(
            (episodes["observations"] > 1).sum()
        ),
        "candidate_observations": int(
            episodes["observations"].sum()
        )
    }
    for gap, episodes in episode_results.items()
])

display(sensitivity_summary)

,max_gap_hours,episode_count,singleton_episodes,multi_observation_episodes,candidate_observations
0,1,81,56,25,123
1,2,75,51,24,123
2,3,70,47,23,123


In [44]:
primary_episodes = episode_results[1].copy()

candidate_detail = model_df.loc[
    candidate_mask[candidate_mask].index
].copy()

candidate_detail["configuration_votes"] = (
    corrected_stability_aligned.loc[
        candidate_detail.index,
        "configuration_votes"
    ]
)

candidate_detail["segment_id"] = (
    test_segments.reindex(candidate_detail.index)
)

print("Candidate observations:", len(candidate_detail))
display(candidate_detail.head())

Candidate observations: 123


,temp,dwpt,rhum,prcp,wspd,pres,segment_id,temp_change_1h,temp_rolling_mean_6h,temp_deviation_6h,configuration_votes
timestamp,,,,,,,,,,,
2025-03-28 15:30:00+05:30,29.1,-7.0,9.0,0.0,15.8,1008.7,145,-0.3,26.733333,2.366667,9
2025-03-28 16:30:00+05:30,28.3,-5.0,11.0,0.0,16.9,1008.7,145,-0.8,27.766667,0.533333,9
2025-03-28 17:30:00+05:30,26.8,-3.9,13.0,0.0,16.2,1008.9,145,-1.5,28.333333,-1.533333,8
2025-03-29 09:30:00+05:30,23.0,0.7,23.0,0.0,9.0,1013.0,145,6.4,13.600000,9.4,12
2025-03-29 14:30:00+05:30,28.8,-0.4,15.0,0.0,22.0,1008.7,145,-0.2,23.933333,4.866667,10


In [45]:
candidate_detail["episode_id"] = pd.NA

for _, episode in primary_episodes.iterrows():

    mask = (
        candidate_detail["segment_id"].eq(episode["segment_id"])
        & (candidate_detail.index >= episode["start"])
        & (candidate_detail.index <= episode["end"])
    )

    candidate_detail.loc[mask, "episode_id"] = (
        episode["episode_id"]
    )

assert candidate_detail["episode_id"].notna().all()

candidate_detail["episode_id"] = (
    candidate_detail["episode_id"].astype(int)
)

print("Episode assignment verified.")

Episode assignment verified.


In [46]:
variables = [
    "temp",
    "dwpt",
    "rhum",
    "prcp",
    "wspd",
    "pres"
]

episode_profiles = (
    candidate_detail.groupby("episode_id")
    .agg(
        segment_id=("segment_id", "first"),
        start=("configuration_votes", lambda x: x.index.min()),
        end=("configuration_votes", lambda x: x.index.max()),
        observations=("configuration_votes", "size"),
        mean_votes=("configuration_votes", "mean"),
        max_votes=("configuration_votes", "max"),
        min_votes=("configuration_votes", "min"),
        temp_min=("temp", "min"),
        temp_max=("temp", "max"),
        dewpoint_min=("dwpt", "min"),
        dewpoint_max=("dwpt", "max"),
        humidity_min=("rhum", "min"),
        humidity_max=("rhum", "max"),
        rainfall_max=("prcp", "max"),
        wind_speed_max=("wspd", "max"),
        pressure_min=("pres", "min"),
        pressure_max=("pres", "max")
    )
    .reset_index()
)

episode_profiles["duration_hours"] = (
    episode_profiles["end"] - episode_profiles["start"]
).dt.total_seconds() / 3600

display(episode_profiles.sort_values(
    ["observations", "max_votes"],
    ascending=False
).head(20))

,episode_id,segment_id,start,end,observations,mean_votes,max_votes,min_votes,temp_min,temp_max,dewpoint_min,dewpoint_max,humidity_min,humidity_max,rainfall_max,wind_speed_max,pressure_min,pressure_max,duration_hours
53,54,145,2025-09-03 03:30:00+05:30,2025-09-03 12:30:00+05:30,10,9.800000,12,8,23.7,28.7,22.5,23.6,74.0,96.0,11.0,29.5,1002.0,1005.9,9.0
18,19,145,2025-04-26 10:30:00+05:30,2025-04-26 14:30:00+05:30,5,10.400000,12,9,36.4,40.0,-4.6,1.2,6.0,11.0,0.0,15.8,1002.9,1004.6,4.0
51,52,145,2025-09-01 20:30:00+05:30,2025-09-01 22:30:00+05:30,3,10.333333,12,8,23.5,24.1,22.5,23.1,94.0,94.0,24.1,19.4,1000.0,1001.3,2.0
76,77,145,2026-01-27 09:30:00+05:30,2026-01-27 11:30:00+05:30,3,10.666667,12,9,12.0,14.0,11.1,13.0,88.0,100.0,7.6,5.4,1020.0,1020.0,2.0
75,76,145,2026-01-24 10:30:00+05:30,2026-01-24 12:30:00+05:30,3,11.000000,11,11,12.0,14.0,-1.0,1.0,36.0,47.0,0.0,11.2,1025.0,1026.0,2.0
0,1,145,2025-03-28 15:30:00+05:30,2025-03-28 17:30:00+05:30,3,8.666667,9,8,26.8,29.1,-7.0,-3.9,9.0,13.0,0.0,16.9,1008.7,1008.9,2.0
74,75,145,2026-01-23 14:30:00+05:30,2026-01-23 16:30:00+05:30,3,8.666667,9,8,11.3,13.0,8.2,12.1,81.0,94.0,8.9,33.1,1013.2,1014.0,2.0
52,53,145,2025-09-02 00:30:00+05:30,2025-09-02 02:30:00+05:30,3,8.000000,8,8,23.2,23.4,21.5,22.4,90.0,94.0,9.9,20.9,1000.7,1001.0,2.0
10,11,145,2025-04-19 11:30:00+05:30,2025-04-19 12:30:00+05:30,2,10.500000,12,9,33.6,36.3,14.8,17.3,28.0,38.0,0.0,31.7,1001.8,1003.8,1.0
12,13,145,2025-04-20 11:30:00+05:30,2025-04-20 12:30:00+05:30,2,10.000000,12,8,35.8,37.1,9.6,13.3,19.0,26.0,0.0,25.6,1004.1,1005.0,1.0


In [47]:
candidate_detail.to_csv(
    OUTPUT_DIR / "candidate_observations_8of12.csv"
)

episode_profiles.to_csv(
    OUTPUT_DIR / "episode_profiles_1h.csv",
    index=False
)

print("Episode characterization saved.")

Episode characterization saved.


In [48]:

import numpy as np
import pandas as pd

physical_data = model_df[
    ["temp", "dwpt", "rhum", "segment_id"]
].copy()

def calculate_expected_rh(temp, dewpoint):
    es_temp = 6.112 * np.exp(
        (17.67 * temp) / (temp + 243.5)
    )

    es_dewpoint = 6.112 * np.exp(
        (17.67 * dewpoint) / (dewpoint + 243.5)
    )

    return 100 * es_dewpoint / es_temp


physical_data["expected_rh"] = calculate_expected_rh(
    physical_data["temp"],
    physical_data["dwpt"]
)

physical_data["rh_difference"] = (
    physical_data["rhum"] - physical_data["expected_rh"]
).abs()

physical_data["rh_consistency_flag"] = (
    physical_data["rh_difference"] > 5
)

print("Physical consistency assessment")
print(
    "Records assessed:",
    physical_data["rh_difference"].notna().sum()
)

print(
    "Differences above 5 percentage points:",
    int(physical_data["rh_consistency_flag"].sum())
)


Physical consistency assessment
Records assessed: 44879
Differences above 5 percentage points: 0


In [49]:
candidate_timestamps = candidate_detail.index

physical_data["is_candidate"] = (
    physical_data.index.isin(candidate_timestamps)
)

physical_comparison = (
    physical_data.groupby("is_candidate")
    .agg(
        observations=("rh_difference", "count"),
        mean_rh_difference=("rh_difference", "mean"),
        median_rh_difference=("rh_difference", "median"),
        maximum_rh_difference=("rh_difference", "max"),
        inconsistent_records=("rh_consistency_flag", "sum")
    )
)

physical_comparison["inconsistent_percentage"] = (
    physical_comparison["inconsistent_records"]
    / physical_comparison["observations"] * 100
)

display(physical_comparison)

,observations,mean_rh_difference,median_rh_difference,maximum_rh_difference,inconsistent_records,inconsistent_percentage
is_candidate,,,,,,
False,44756,0.124554,0.113325,0.332251,0,0.0
True,123,0.11584,0.097333,0.308796,0,0.0


In [50]:
candidate_physical = physical_data.loc[
    candidate_timestamps
].copy()

candidate_physical = candidate_physical.sort_values(
    "rh_difference",
    ascending=False
)

display(
    candidate_physical[
        [
            "temp",
            "dwpt",
            "rhum",
            "expected_rh",
            "rh_difference",
            "segment_id",
            "rh_consistency_flag"
        ]
    ].head(20)
)

,temp,dwpt,rhum,expected_rh,rh_difference,segment_id,rh_consistency_flag
timestamp,,,,,,,
2026-01-27 23:30:00+05:30,8.1,7.3,95.0,94.691204,0.308796,145,False
2026-01-27 11:30:00+05:30,14.0,13.0,94.0,93.693303,0.306697,145,False
2025-09-03 06:30:00+05:30,23.7,22.8,95.0,94.703286,0.296714,145,False
2025-07-21 09:30:00+05:30,30.0,25.9,79.0,78.708249,0.291751,145,False
2025-08-24 07:30:00+05:30,26.4,24.2,88.0,87.720834,0.279166,145,False
2025-08-26 12:30:00+05:30,27.0,24.6,87.0,86.728204,0.271796,145,False
2026-01-23 02:30:00+05:30,14.5,11.6,83.0,82.730262,0.269738,145,False
2025-04-18 18:30:00+05:30,28.8,18.5,54.0,53.730516,0.269484,145,False
2026-01-01 07:30:00+05:30,11.3,10.8,97.0,96.734347,0.265653,145,False


In [51]:
print("Candidate observations by segment:")
display(
    candidate_detail["segment_id"].value_counts().sort_index()
)

print("\nEpisodes by segment:")
display(
    primary_episodes["segment_id"].value_counts().sort_index()
)

Candidate observations by segment:


,count
segment_id,
145,122
154,1



Episodes by segment:


,count
segment_id,
145,80
154,1


In [52]:
from pathlib import Path
import pandas as pd

PHASE00_DIR = Path(
    "/content/drive/My Drive/weather_anomaly_phase00"
)

station_files = [
    "VILD0_observed.csv",
    "42099_observed.csv",
    "42101_observed.csv",
    "42105_observed.csv",
    "42096_observed.csv"
]

for filename in station_files:
    path = PHASE00_DIR / filename

    print(f"\n{'='*55}")
    print("Station:", filename)

    if not path.exists():
        print("FILE NOT FOUND")
        continue

    station_df = pd.read_csv(path)

    print("Shape:", station_df.shape)
    print("Columns:", station_df.columns.tolist())
    print("First five rows:")
    display(station_df.head())


Station: VILD0_observed.csv
Shape: (1002, 12)
Columns: ['time', 'temp', 'dwpt', 'rhum', 'prcp', 'snow', 'wdir', 'wspd', 'wpgt', 'pres', 'tsun', 'coco']
First five rows:


,time,temp,dwpt,rhum,prcp,snow,wdir,wspd,wpgt,pres,tsun,coco
0,2025-03-28 11:30:00+05:30,26.0,-2.6,15.0,NaN,NaN,340.0,11.0,NaN,1013.0,NaN,NaN
1,2025-03-29 09:30:00+05:30,23.0,0.7,23.0,NaN,NaN,310.0,9.0,NaN,1013.0,NaN,NaN
2,2025-03-29 10:30:00+05:30,23.0,1.9,25.0,NaN,NaN,300.0,7.0,NaN,1013.0,NaN,NaN
3,2025-03-29 11:30:00+05:30,25.0,1.8,22.0,NaN,NaN,290.0,7.0,NaN,1012.0,NaN,NaN
4,2025-03-29 12:30:00+05:30,27.0,1.4,19.0,NaN,NaN,320.0,9.0,NaN,1012.0,NaN,NaN



Station: 42099_observed.csv
Shape: (567, 12)
Columns: ['time', 'temp', 'dwpt', 'rhum', 'prcp', 'snow', 'wdir', 'wspd', 'wpgt', 'pres', 'tsun', 'coco']
First five rows:


,time,temp,dwpt,rhum,prcp,snow,wdir,wspd,wpgt,pres,tsun,coco
0,2025-03-28 08:30:00+05:30,22.5,10.0,45.0,NaN,NaN,320.0,3.6,NaN,1010.6,NaN,NaN
1,2025-03-28 11:30:00+05:30,25.6,-3.0,15.0,NaN,NaN,340.0,11.2,NaN,1011.8,NaN,NaN
2,2025-03-28 14:30:00+05:30,27.6,1.9,19.0,NaN,NaN,320.0,5.4,NaN,1010.4,NaN,NaN
3,2025-03-28 17:30:00+05:30,26.4,2.3,21.0,NaN,NaN,320.0,5.4,NaN,1009.4,NaN,NaN
4,2025-03-29 08:30:00+05:30,19.0,3.2,35.0,NaN,NaN,340.0,3.6,NaN,1011.6,NaN,NaN



Station: 42101_observed.csv
Shape: (1188, 12)
Columns: ['time', 'temp', 'dwpt', 'rhum', 'prcp', 'snow', 'wdir', 'wspd', 'wpgt', 'pres', 'tsun', 'coco']
First five rows:


,time,temp,dwpt,rhum,prcp,snow,wdir,wspd,wpgt,pres,tsun,coco
0,2025-03-28 02:30:00+05:30,20.7,15.5,72.0,NaN,NaN,290.0,1.8,NaN,1007.8,NaN,NaN
1,2025-03-28 05:30:00+05:30,18.6,14.9,79.0,NaN,NaN,270.0,1.8,NaN,1008.6,NaN,NaN
2,2025-03-28 08:30:00+05:30,21.7,13.6,60.0,NaN,NaN,270.0,1.8,NaN,1010.9,NaN,NaN
3,2025-03-28 11:30:00+05:30,27.0,12.6,41.0,NaN,NaN,0.0,0.0,NaN,1012.1,NaN,NaN
4,2025-03-28 14:30:00+05:30,28.2,7.4,27.0,NaN,NaN,270.0,7.6,NaN,1010.9,NaN,NaN



Station: 42105_observed.csv
Shape: (6, 12)
Columns: ['time', 'temp', 'dwpt', 'rhum', 'prcp', 'snow', 'wdir', 'wspd', 'wpgt', 'pres', 'tsun', 'coco']
First five rows:


,time,temp,dwpt,rhum,prcp,snow,wdir,wspd,wpgt,pres,tsun,coco
0,2025-07-16 09:30:00+05:30,26.0,25.0,94.0,NaN,NaN,110.0,15.0,NaN,1003.0,NaN,5.0
1,2025-11-13 01:30:00+05:30,11.0,9.1,88.0,NaN,NaN,NaN,4.0,NaN,1016.0,NaN,5.0
2,2025-11-13 02:30:00+05:30,11.0,9.1,88.0,NaN,NaN,NaN,4.0,NaN,1016.0,NaN,5.0
3,2025-11-13 03:30:00+05:30,10.0,9.1,94.0,NaN,NaN,NaN,4.0,NaN,1016.0,NaN,5.0
4,2025-11-13 04:30:00+05:30,10.0,7.9,87.0,NaN,NaN,NaN,2.0,NaN,1016.0,NaN,5.0



Station: 42096_observed.csv
Shape: (0, 12)
Columns: ['time', 'temp', 'dwpt', 'rhum', 'prcp', 'snow', 'wdir', 'wspd', 'wpgt', 'pres', 'tsun', 'coco']
First five rows:


,time,temp,dwpt,rhum,prcp,snow,wdir,wspd,wpgt,pres,tsun,coco


In [53]:
from pathlib import Path
import pandas as pd
import numpy as np

PHASE00_DIR = Path(
    "/content/drive/My Drive/weather_anomaly_phase00"
)

station_ids = ["VILD0", "42099", "42101", "42105", "42096"]

station_data = {}
coverage_results = []

for station in station_ids:

    path = PHASE00_DIR / f"{station}_observed.csv"

    df = pd.read_csv(path)

    df["time"] = pd.to_datetime(df["time"])

    df = df.sort_values("time").drop_duplicates(
        subset="time"
    )

    df = df.set_index("time")

    station_data[station] = df

    intervals = df.index.to_series().diff().dropna()

    coverage_results.append({
        "station": station,
        "records": len(df),
        "start": df.index.min() if len(df) else None,
        "end": df.index.max() if len(df) else None,
        "median_interval_hours": (
            intervals.median().total_seconds() / 3600
            if len(intervals) else np.nan
        ),
        "maximum_gap_hours": (
            intervals.max().total_seconds() / 3600
            if len(intervals) else np.nan
        ),
        "temperature_records": (
            df["temp"].notna().sum() if len(df) else 0
        ),
        "pressure_records": (
            df["pres"].notna().sum() if len(df) else 0
        ),
        "humidity_records": (
            df["rhum"].notna().sum() if len(df) else 0
        )
    })

coverage_df = pd.DataFrame(coverage_results)

display(coverage_df)

,station,records,start,end,median_interval_hours,maximum_gap_hours,temperature_records,pressure_records,humidity_records
0,VILD0,1002,2025-03-28 11:30:00+05:30,2026-03-09 14:30:00+05:30,1.0,1467.0,633,1002,633
1,42099,567,2025-03-28 08:30:00+05:30,2025-08-24 17:30:00+05:30,3.0,21.0,566,567,566
2,42101,1188,2025-03-28 02:30:00+05:30,2025-08-24 23:30:00+05:30,3.0,9.0,1187,1188,1187
3,42105,6,2025-07-16 09:30:00+05:30,2025-11-13 05:30:00+05:30,1.0,2872.0,6,6,6
4,42096,0,NaT,NaT,NaN,NaN,0,0,0


In [54]:
candidate_times = pd.DatetimeIndex(
    candidate_detail.index
)

matching_results = []

for station, df in station_data.items():

    if df.empty:
        matching_results.append({
            "station": station,
            "exact_matches": 0,
            "within_1h": 0,
            "within_90min": 0
        })
        continue

    station_times = df.index

    differences = []

    for timestamp in candidate_times:

        delta = abs(
            station_times - timestamp
        )

        differences.append(delta.min())

    differences = pd.TimedeltaIndex(differences)

    matching_results.append({
        "station": station,
        "exact_matches": int(
            (differences == pd.Timedelta(0)).sum()
        ),
        "within_1h": int(
            (differences <= pd.Timedelta(hours=1)).sum()
        ),
        "within_90min": int(
            (differences <= pd.Timedelta(minutes=90)).sum()
        )
    })

candidate_coverage = pd.DataFrame(matching_results)

display(candidate_coverage)

,station,exact_matches,within_1h,within_90min
0,VILD0,23,31,31
1,42099,15,50,50
2,42101,17,60,60
3,42105,0,0,0
4,42096,0,0,0


In [55]:
episode_station_coverage = []

for _, episode in primary_episodes.iterrows():

    start = episode["start"]
    end = episode["end"]

    row = {
        "episode_id": episode["episode_id"],
        "start": start,
        "end": end,
        "candidate_observations": episode["observations"]
    }

    for station in ["42099", "42101"]:

        df = station_data[station]

        available = df.loc[
            (df.index >= start) &
            (df.index <= end)
        ]

        row[f"{station}_records"] = len(available)
        row[f"{station}_temp_records"] = (
            available["temp"].notna().sum()
        )

    episode_station_coverage.append(row)

episode_station_coverage = pd.DataFrame(
    episode_station_coverage
)

display(
    episode_station_coverage.sort_values(
        ["42099_records", "42101_records"],
        ascending=False
    ).head(20)
)

,episode_id,start,end,candidate_observations,42099_records,42099_temp_records,42101_records,42101_temp_records
18,19,2025-04-26 10:30:00+05:30,2025-04-26 14:30:00+05:30,5,2,2,2,2
0,1,2025-03-28 15:30:00+05:30,2025-03-28 17:30:00+05:30,3,1,1,1,1
2,3,2025-03-29 14:30:00+05:30,2025-03-29 14:30:00+05:30,1,1,1,1,1
8,9,2025-04-18 11:30:00+05:30,2025-04-18 11:30:00+05:30,1,1,1,1,1
10,11,2025-04-19 11:30:00+05:30,2025-04-19 12:30:00+05:30,2,1,1,1,1
12,13,2025-04-20 11:30:00+05:30,2025-04-20 12:30:00+05:30,2,1,1,1,1
13,14,2025-04-21 17:30:00+05:30,2025-04-21 18:30:00+05:30,2,1,1,1,1
14,15,2025-04-22 11:30:00+05:30,2025-04-22 11:30:00+05:30,1,1,1,1,1
19,20,2025-04-26 17:30:00+05:30,2025-04-26 18:30:00+05:30,2,1,1,1,1
32,33,2025-06-28 17:30:00+05:30,2025-06-28 17:30:00+05:30,1,1,1,1,1


In [60]:

def match_candidate_observations(
    candidate_times,
    station_df,
    station_name,
    tolerance_minutes=90
):
    candidates = pd.DataFrame({
        "candidate_time": pd.DatetimeIndex(candidate_times)
    }).sort_values("candidate_time")

    station = station_df.copy()

    # Normalize both indexes to the same timezone representation.
    candidates["candidate_time"] = (
        candidates["candidate_time"].dt.tz_convert("UTC")
    )

    station.index = station.index.tz_convert("UTC")

    # Retain unique station timestamps.
    station = station.loc[
        ~station.index.duplicated(keep="first")
    ].sort_index()

    station = (
        station.rename_axis("station_time")
        .reset_index()
        .sort_values("station_time")
    )

    matched = pd.merge_asof(
        candidates,
        station,
        left_on="candidate_time",
        right_on="station_time",
        direction="nearest",
        tolerance=pd.Timedelta(minutes=tolerance_minutes)
    )

    matched["station"] = station_name

    matched["offset_minutes"] = (
        matched["station_time"] - matched["candidate_time"]
    ).dt.total_seconds() / 60

    matched["exact_match"] = (
        matched["offset_minutes"] == 0
    )

    matched["matched"] = matched["station_time"].notna()

    # Display timestamps in Indian Standard Time.
    matched["candidate_time"] = (
        matched["candidate_time"].dt.tz_convert("Asia/Kolkata")
    )

    matched["station_time"] = (
        matched["station_time"].dt.tz_convert("Asia/Kolkata")
    )

    return matched



In [61]:

spatial_matches = {}

for station in ["42099", "42101"]:

    matched = match_candidate_observations(
        candidate_times=candidate_detail.index,
        station_df=station_data[station],
        station_name=station,
        tolerance_minutes=90
    )

    spatial_matches[station] = matched

    valid = matched.loc[matched["matched"]]

    print(f"\nStation: {station}")
    print("Matched candidate timestamps:", len(valid))
    print("Exact matches:", int(valid["exact_match"].sum()))

    print(
        "Unique station observations used:",
        valid["station_time"].nunique()
    )

    print(
        "Reused station observations:",
        int(valid["station_time"].duplicated().sum())
    )

    print(
        "Maximum absolute time offset (minutes):",
        valid["offset_minutes"].abs().max()
    )



Station: 42099
Matched candidate timestamps: 50
Exact matches: 15
Unique station observations used: 42
Reused station observations: 8
Maximum absolute time offset (minutes): 60.0

Station: 42101
Matched candidate timestamps: 60
Exact matches: 17
Unique station observations used: 51
Reused station observations: 9
Maximum absolute time offset (minutes): 60.0


In [62]:
for station, matched in spatial_matches.items():

    reused = matched.loc[
        matched["matched"]
        & matched["station_time"].duplicated(keep=False)
    ].sort_values("station_time")

    print(f"\nRepeated matches: {station}")
    display(
        reused[
            [
                "candidate_time",
                "station_time",
                "offset_minutes",
                "temp",
                "dwpt",
                "rhum",
                "pres"
            ]
        ].head(20)
    )


Repeated matches: 42099


,candidate_time,station_time,offset_minutes,temp,dwpt,rhum,pres
1,2025-03-28 16:30:00+05:30,2025-03-28 17:30:00+05:30,60.0,26.4,2.3,21.0,1009.4
2,2025-03-28 17:30:00+05:30,2025-03-28 17:30:00+05:30,0.0,26.4,2.3,21.0,1009.4
13,2025-04-19 11:30:00+05:30,2025-04-19 11:30:00+05:30,0.0,33.0,20.2,47.0,1003.2
14,2025-04-19 12:30:00+05:30,2025-04-19 11:30:00+05:30,-60.0,33.0,20.2,47.0,1003.2
16,2025-04-20 11:30:00+05:30,2025-04-20 11:30:00+05:30,0.0,33.4,18.7,42.0,1003.3
17,2025-04-20 12:30:00+05:30,2025-04-20 11:30:00+05:30,-60.0,33.4,18.7,42.0,1003.3
18,2025-04-21 17:30:00+05:30,2025-04-21 17:30:00+05:30,0.0,38.0,13.9,24.0,1003.0
19,2025-04-21 18:30:00+05:30,2025-04-21 17:30:00+05:30,-60.0,38.0,13.9,24.0,1003.0
26,2025-04-26 10:30:00+05:30,2025-04-26 11:30:00+05:30,60.0,39.0,9.5,17.0,1003.8
27,2025-04-26 11:30:00+05:30,2025-04-26 11:30:00+05:30,0.0,39.0,9.5,17.0,1003.8



Repeated matches: 42101


,candidate_time,station_time,offset_minutes,temp,dwpt,rhum,pres
1,2025-03-28 16:30:00+05:30,2025-03-28 17:30:00+05:30,60.0,25.6,7.7,32.0,1010.1
2,2025-03-28 17:30:00+05:30,2025-03-28 17:30:00+05:30,0.0,25.6,7.7,32.0,1010.1
13,2025-04-19 11:30:00+05:30,2025-04-19 11:30:00+05:30,0.0,31.2,21.1,55.0,1005.3
14,2025-04-19 12:30:00+05:30,2025-04-19 11:30:00+05:30,-60.0,31.2,21.1,55.0,1005.3
16,2025-04-20 11:30:00+05:30,2025-04-20 11:30:00+05:30,0.0,33.6,20.4,46.0,1004.6
17,2025-04-20 12:30:00+05:30,2025-04-20 11:30:00+05:30,-60.0,33.6,20.4,46.0,1004.6
18,2025-04-21 17:30:00+05:30,2025-04-21 17:30:00+05:30,0.0,38.0,10.3,19.0,1004.0
19,2025-04-21 18:30:00+05:30,2025-04-21 17:30:00+05:30,-60.0,38.0,10.3,19.0,1004.0
25,2025-04-25 20:30:00+05:30,2025-04-25 20:30:00+05:30,0.0,29.4,11.4,33.0,1004.0
24,2025-04-25 19:30:00+05:30,2025-04-25 20:30:00+05:30,60.0,29.4,11.4,33.0,1004.0


In [63]:

import numpy as np
import pandas as pd
from scipy.optimize import linear_sum_assignment


def one_to_one_station_matching(
    candidate_times,
    station_df,
    station_name,
    tolerance_minutes=90
):
    candidates = pd.DatetimeIndex(candidate_times)

    # Normalize timestamps to UTC for consistent comparison.
    candidates = candidates.tz_convert("UTC")

    station = station_df.copy()
    station.index = station.index.tz_convert("UTC")

    station = station.loc[
        ~station.index.duplicated(keep="first")
    ].sort_index()

    station_times = station.index

    n_candidates = len(candidates)
    n_station = len(station_times)

    if n_candidates == 0 or n_station == 0:
        return pd.DataFrame({
            "candidate_time": candidates.tz_convert("Asia/Kolkata"),
            "station_time": pd.NaT,
            "offset_minutes": np.nan,
            "matched": False,
            "exact_match": False,
            "station": station_name
        })

    # Calculate absolute time differences in minutes.
    differences = np.abs(
        (
            candidates.to_numpy()[:, None]
            - station_times.to_numpy()[None, :]
        ) / np.timedelta64(1, "m")
    )

    # Cost matrix: real station observations + dummy unmatched columns.
    dummy_cost = (n_candidates + 1) * (
        tolerance_minutes + 1
    )

    invalid_cost = dummy_cost * (n_candidates + 1)

    cost = np.full(
        (n_candidates, n_station + n_candidates),
        dummy_cost,
        dtype=float
    )

    valid = differences <= tolerance_minutes

    cost[:, :n_station] = np.where(
        valid,
        differences,
        invalid_cost
    )

    # Find the globally optimal one-to-one assignment.
    row_indices, col_indices = linear_sum_assignment(cost)

    assignments = {}

    for row, col in zip(row_indices, col_indices):
        if col < n_station and valid[row, col]:
            assignments[row] = col

    records = []

    for i, candidate_time in enumerate(candidates):

        if i in assignments:
            j = assignments[i]
            station_time = station_times[j]
            offset = (
                station_time - candidate_time
            ).total_seconds() / 60

            values = station.iloc[j].to_dict()

            records.append({
                "candidate_time": candidate_time.tz_convert(
                    "Asia/Kolkata"
                ),
                "station_time": station_time.tz_convert(
                    "Asia/Kolkata"
                ),
                "offset_minutes": offset,
                "matched": True,
                "exact_match": offset == 0,
                "station": station_name,
                **values
            })

        else:
            records.append({
                "candidate_time": candidate_time.tz_convert(
                    "Asia/Kolkata"
                ),
                "station_time": pd.NaT,
                "offset_minutes": np.nan,
                "matched": False,
                "exact_match": False,
                "station": station_name
            })

    result = pd.DataFrame(records)

    # Integrity checks.
    matched_times = result.loc[
        result["matched"], "station_time"
    ]

    assert not matched_times.duplicated().any()

    assert (
        result["matched"].sum()
        == len(assignments)
    )

    return result


In [64]:
spatial_matches_unique = {}

for station in ["42099", "42101"]:

    matched = one_to_one_station_matching(
        candidate_times=candidate_detail.index,
        station_df=station_data[station],
        station_name=station,
        tolerance_minutes=90
    )

    spatial_matches_unique[station] = matched

    valid = matched.loc[matched["matched"]]

    print(f"\nStation: {station}")
    print("Matched candidates:", len(valid))
    print("Unmatched candidates:", int((~matched["matched"]).sum()))
    print("Exact matches:", int(valid["exact_match"].sum()))

    print(
        "Unique station observations:",
        valid["station_time"].nunique()
    )

    print(
        "Reused observations:",
        int(valid["station_time"].duplicated().sum())
    )

    print(
        "Mean absolute time offset:",
        valid["offset_minutes"].abs().mean()
    )

    print(
        "Maximum absolute time offset:",
        valid["offset_minutes"].abs().max()
    )


Station: 42099
Matched candidates: 42
Unmatched candidates: 81
Exact matches: 15
Unique station observations: 42
Reused observations: 0
Mean absolute time offset: 38.57142857142857
Maximum absolute time offset: 60.0

Station: 42101
Matched candidates: 51
Unmatched candidates: 72
Exact matches: 17
Unique station observations: 51
Reused observations: 0
Mean absolute time offset: 40.0
Maximum absolute time offset: 60.0


In [65]:
for station, matched in spatial_matches_unique.items():

    print(f"\n{station} — one-to-one matches")

    display(
        matched.loc[
            matched["matched"],
            [
                "candidate_time",
                "station_time",
                "offset_minutes",
                "temp",
                "dwpt",
                "rhum",
                "wspd",
                "pres"
            ]
        ].head(15)
    )


42099 — one-to-one matches


,candidate_time,station_time,offset_minutes,temp,dwpt,rhum,wspd,pres
0,2025-03-28 15:30:00+05:30,2025-03-28 14:30:00+05:30,-60.0,27.6,1.9,19.0,5.4,1010.4
2,2025-03-28 17:30:00+05:30,2025-03-28 17:30:00+05:30,0.0,26.4,2.3,21.0,5.4,1009.4
3,2025-03-29 09:30:00+05:30,2025-03-29 08:30:00+05:30,-60.0,19.0,3.2,35.0,3.6,1011.6
4,2025-03-29 14:30:00+05:30,2025-03-29 14:30:00+05:30,0.0,29.4,2.6,18.0,9.4,1008.8
5,2025-03-30 09:30:00+05:30,2025-03-30 08:30:00+05:30,-60.0,22.0,4.5,32.0,3.6,1011.1
6,2025-03-31 12:30:00+05:30,2025-03-31 11:30:00+05:30,-60.0,30.4,2.6,17.0,9.4,1011.9
7,2025-03-31 13:30:00+05:30,2025-03-31 14:30:00+05:30,60.0,32.2,3.2,16.0,11.2,1009.9
9,2025-04-08 13:30:00+05:30,2025-04-08 14:30:00+05:30,60.0,37.8,17.2,30.0,5.4,1009.0
10,2025-04-11 12:30:00+05:30,2025-04-11 11:30:00+05:30,-60.0,29.6,16.4,45.0,22.3,1005.4
11,2025-04-18 11:30:00+05:30,2025-04-18 11:30:00+05:30,0.0,34.0,18.1,39.0,7.6,1003.3



42101 — one-to-one matches


,candidate_time,station_time,offset_minutes,temp,dwpt,rhum,wspd,pres
0,2025-03-28 15:30:00+05:30,2025-03-28 14:30:00+05:30,-60.0,28.2,7.4,27.0,7.6,1010.9
2,2025-03-28 17:30:00+05:30,2025-03-28 17:30:00+05:30,0.0,25.6,7.7,32.0,7.6,1010.1
3,2025-03-29 09:30:00+05:30,2025-03-29 08:30:00+05:30,-60.0,15.2,7.0,58.0,3.6,1012.4
4,2025-03-29 14:30:00+05:30,2025-03-29 14:30:00+05:30,0.0,27.8,6.0,25.0,9.4,1009.9
5,2025-03-30 09:30:00+05:30,2025-03-30 08:30:00+05:30,-60.0,18.0,8.8,55.0,5.4,1012.1
6,2025-03-31 12:30:00+05:30,2025-03-31 11:30:00+05:30,-60.0,30.8,7.3,23.0,7.6,1012.5
7,2025-03-31 13:30:00+05:30,2025-03-31 14:30:00+05:30,60.0,32.6,6.7,20.0,14.8,1010.5
8,2025-04-02 09:30:00+05:30,2025-04-02 08:30:00+05:30,-60.0,19.4,12.4,64.0,1.8,1012.5
9,2025-04-08 13:30:00+05:30,2025-04-08 14:30:00+05:30,60.0,36.8,18.8,35.0,7.6,1010.5
10,2025-04-11 12:30:00+05:30,2025-04-11 11:30:00+05:30,-60.0,29.0,19.6,57.0,14.8,1005.2


In [66]:

comparison_variables = [
    "temp", "dwpt", "rhum", "wspd", "pres", "prcp"
]

vild0_reference = candidate_detail[
    comparison_variables + ["configuration_votes", "segment_id"]
].copy()

vild0_reference = vild0_reference.add_prefix("vild0_")

spatial_comparisons = {}

for station in ["42099", "42101"]:

    matched = spatial_matches_unique[station].copy()

    matched = matched.set_index("candidate_time")

    # Keep neighbouring-station measurements separate.
    station_columns = [
        c for c in comparison_variables if c in matched.columns
    ]

    matched = matched[
        ["station_time", "offset_minutes", "matched",
         "exact_match"] + station_columns
    ]

    matched = matched.rename(
        columns={
            c: f"{station}_{c}"
            for c in station_columns
        }
    )

    comparison = matched.join(
        vild0_reference,
        how="left"
    )

    for variable in comparison_variables:

        station_col = f"{station}_{variable}"
        vild0_col = f"vild0_{variable}"

        if station_col in comparison.columns:
            comparison[f"{variable}_difference"] = (
                comparison[station_col] - comparison[vild0_col]
            )

    spatial_comparisons[station] = comparison

    print(f"\n{station}:")
    print("Rows:", len(comparison))
    print("Matched:", int(comparison["matched"].sum()))
    print("Exact:", int(comparison["exact_match"].sum()))

    display(comparison.head())



42099:
Rows: 123
Matched: 42
Exact: 15


,station_time,offset_minutes,matched,exact_match,42099_temp,42099_dwpt,42099_rhum,42099_wspd,42099_pres,42099_prcp,...,vild0_pres,vild0_prcp,vild0_configuration_votes,vild0_segment_id,temp_difference,dwpt_difference,rhum_difference,wspd_difference,pres_difference,prcp_difference
candidate_time,,,,,,,,,,,,,,,,,,,,,
2025-03-28 15:30:00+05:30,2025-03-28 14:30:00+05:30,-60.0,True,False,27.6,1.9,19.0,5.4,1010.4,NaN,...,1008.7,0.0,9,145,-1.5,8.9,10.0,-10.4,1.7,<NA>
2025-03-28 16:30:00+05:30,NaT,NaN,False,False,NaN,NaN,NaN,NaN,NaN,NaN,...,1008.7,0.0,9,145,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2025-03-28 17:30:00+05:30,2025-03-28 17:30:00+05:30,0.0,True,True,26.4,2.3,21.0,5.4,1009.4,NaN,...,1008.9,0.0,8,145,-0.4,6.2,8.0,-10.8,0.5,<NA>
2025-03-29 09:30:00+05:30,2025-03-29 08:30:00+05:30,-60.0,True,False,19.0,3.2,35.0,3.6,1011.6,NaN,...,1013.0,0.0,12,145,-4.0,2.5,12.0,-5.4,-1.4,<NA>
2025-03-29 14:30:00+05:30,2025-03-29 14:30:00+05:30,0.0,True,True,29.4,2.6,18.0,9.4,1008.8,NaN,...,1008.7,0.0,10,145,0.6,3.0,3.0,-12.6,0.1,<NA>



42101:
Rows: 123
Matched: 51
Exact: 17


,station_time,offset_minutes,matched,exact_match,42101_temp,42101_dwpt,42101_rhum,42101_wspd,42101_pres,42101_prcp,...,vild0_pres,vild0_prcp,vild0_configuration_votes,vild0_segment_id,temp_difference,dwpt_difference,rhum_difference,wspd_difference,pres_difference,prcp_difference
candidate_time,,,,,,,,,,,,,,,,,,,,,
2025-03-28 15:30:00+05:30,2025-03-28 14:30:00+05:30,-60.0,True,False,28.2,7.4,27.0,7.6,1010.9,NaN,...,1008.7,0.0,9,145,-0.9,14.4,18.0,-8.2,2.2,<NA>
2025-03-28 16:30:00+05:30,NaT,NaN,False,False,NaN,NaN,NaN,NaN,NaN,NaN,...,1008.7,0.0,9,145,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2025-03-28 17:30:00+05:30,2025-03-28 17:30:00+05:30,0.0,True,True,25.6,7.7,32.0,7.6,1010.1,NaN,...,1008.9,0.0,8,145,-1.2,11.6,19.0,-8.6,1.2,<NA>
2025-03-29 09:30:00+05:30,2025-03-29 08:30:00+05:30,-60.0,True,False,15.2,7.0,58.0,3.6,1012.4,NaN,...,1013.0,0.0,12,145,-7.8,6.3,35.0,-5.4,-0.6,<NA>
2025-03-29 14:30:00+05:30,2025-03-29 14:30:00+05:30,0.0,True,True,27.8,6.0,25.0,9.4,1009.9,NaN,...,1008.7,0.0,10,145,-1.0,6.4,10.0,-12.6,1.2,<NA>


In [67]:

comparison_summary = []

for station, comparison in spatial_comparisons.items():

    for match_type, subset in [
        ("Exact", comparison.loc[comparison["exact_match"]]),
        (
            "Approximate_60min",
            comparison.loc[
                comparison["matched"] & ~comparison["exact_match"]
            ]
        )
    ]:

        for variable in comparison_variables:

            diff_col = f"{variable}_difference"

            if diff_col not in subset.columns:
                continue

            differences = subset[diff_col].dropna()

            comparison_summary.append({
                "station": station,
                "match_type": match_type,
                "variable": variable,
                "valid_pairs": len(differences),
                "mean_difference": (
                    differences.mean() if len(differences) else np.nan
                ),
                "mean_absolute_difference": (
                    differences.abs().mean()
                    if len(differences) else np.nan
                ),
                "median_absolute_difference": (
                    differences.abs().median()
                    if len(differences) else np.nan
                )
            })

comparison_summary = pd.DataFrame(comparison_summary)

display(comparison_summary)


,station,match_type,variable,valid_pairs,mean_difference,mean_absolute_difference,median_absolute_difference
0,42099,Exact,temp,15,-0.780000,1.900000,1.6
1,42099,Exact,dwpt,15,6.166667,6.246667,5.4
2,42099,Exact,rhum,15,9.600000,11.466667,10.0
3,42099,Exact,wspd,15,-7.560000,7.906667,7.2
4,42099,Exact,pres,15,-0.500000,0.886667,0.6
5,42099,Exact,prcp,0,NaN,NaN,NaN
6,42099,Approximate_60min,temp,27,1.174074,2.959259,1.9
7,42099,Approximate_60min,dwpt,27,2.785185,3.000000,1.4
8,42099,Approximate_60min,rhum,27,0.333333,9.962963,7.0
9,42099,Approximate_60min,wspd,27,-7.788889,8.307407,8.6


In [68]:
matched_42099 = set(
    spatial_comparisons["42099"].loc[
        spatial_comparisons["42099"]["matched"]
    ].index
)

matched_42101 = set(
    spatial_comparisons["42101"].loc[
        spatial_comparisons["42101"]["matched"]
    ].index
)

common_candidate_times = sorted(
    matched_42099 & matched_42101
)

print("Matched by both stations:", len(common_candidate_times))

common_comparison = pd.concat(
    {
        station: spatial_comparisons[station].loc[
            common_candidate_times
        ]
        for station in ["42099", "42101"]
    },
    names=["station", "candidate_time"]
)

display(common_comparison.head(20))

Matched by both stations: 41


station_time  offset_minutes  \
station candidate_time                                                        
42099   2025-03-28 15:30:00+05:30 2025-03-28 14:30:00+05:30           -60.0   
        2025-03-28 17:30:00+05:30 2025-03-28 17:30:00+05:30             0.0   
        2025-03-29 09:30:00+05:30 2025-03-29 08:30:00+05:30           -60.0   
        2025-03-29 14:30:00+05:30 2025-03-29 14:30:00+05:30             0.0   
        2025-03-30 09:30:00+05:30 2025-03-30 08:30:00+05:30           -60.0   
        2025-03-31 12:30:00+05:30 2025-03-31 11:30:00+05:30           -60.0   
        2025-03-31 13:30:00+05:30 2025-03-31 14:30:00+05:30            60.0   
        2025-04-08 13:30:00+05:30 2025-04-08 14:30:00+05:30            60.0   
        2025-04-11 12:30:00+05:30 2025-04-11 11:30:00+05:30           -60.0   
        2025-04-18 11:30:00+05:30 2025-04-18 11:30:00+05:30             0.0   
        2025-04-18 18:30:00+05:30 2025-04-18 17:30:00+05:30           -60.0   
        2025-04-19 11:30:00+05:30 2025-04-19 11:30:00+05:30             0.0   
        2025-04-19 16:30:00+05:30 2025-04-19 17:30:00+05:30            60.0   
        2025-04-20 11:30:00+05:30 2025-04-20 11:30:00+05:30             0.0   
        2025-04-21 17:30:00+05:30 2025-04-21 17:30:00+05:30             0.0   
        2025-04-22 11:30:00+05:30 2025-04-22 11:30:00+05:30             0.0   
        2025-04-25 12:30:00+05:30 2025-04-25 11:30:00+05:30           -60.0   
        2025-04-25 15:30:00+05:30 2025-04-25 14:30:00+05:30           -60.0   
        2025-04-25 16:30:00+05:30 2025-04-25 17:30:00+05:30            60.0   
        2025-04-26 11:30:00+05:30 2025-04-26 11:30:00+05:30             0.0   

                                   matched  exact_match  42099_temp  \
station candidate_time                                                
42099   2025-03-28 15:30:00+05:30     True        False        27.6   
        2025-03-28 17:30:00+05:30     True         True        26.4   
        2025-03-29 09:30:00+05:30     True        False        19.0   
        2025-03-29 14:30:00+05:30     True         True        29.4   
        2025-03-30 09:30:00+05:30     True        False        22.0   
        2025-03-31 12:30:00+05:30     True        False        30.4   
        2025-03-31 13:30:00+05:30     True        False        32.2   
        2025-04-08 13:30:00+05:30     True        False        37.8   
        2025-04-11 12:30:00+05:30     True        False        29.6   
        2025-04-18 11:30:00+05:30     True         True        34.0   
        2025-04-18 18:30:00+05:30     True        False        32.4   
        2025-04-19 11:30:00+05:30     True         True        33.0   
        2025-04-19 16:30:00+05:30     True        False        35.6   
        2025-04-20 11:30:00+05:30     True         True        33.4   
        2025-04-21 17:30:00+05:30     True         True        38.0   
        2025-04-22 11:30:00+05:30     True         True        35.6   
        2025-04-25 12:30:00+05:30     True        False        38.4   
        2025-04-25 15:30:00+05:30     True        False        41.2   
        2025-04-25 16:30:00+05:30     True        False        39.4   
        2025-04-26 11:30:00+05:30     True         True        39.0   

                                   42099_dwpt  42099_rhum  42099_wspd  \
station candidate_time                                                  
42099   2025-03-28 15:30:00+05:30         1.9        19.0         5.4   
        2025-03-28 17:30:00+05:30         2.3        21.0         5.4   
        2025-03-29 09:30:00+05:30         3.2        35.0         3.6   
        2025-03-29 14:30:00+05:30         2.6        18.0         9.4   
        2025-03-30 09:30:00+05:30         4.5        32.0         3.6   
        2025-03-31 12:30:00+05:30         2.6        17.0         9.4   
        2025-03-31 13:30:00+05:30         3.2        16.0        11.2   
        2025-04-08 13:30:00+05:30        17.2        30.0         5.4   
        2025-04

In [69]:
for station, comparison in spatial_comparisons.items():
    comparison.to_csv(
        OUTPUT_DIR / f"spatial_comparison_{station}.csv"
    )

comparison_summary.to_csv(
    OUTPUT_DIR / "spatial_comparison_summary.csv",
    index=False
)

print("Spatial comparison results saved.")

Spatial comparison results saved.


In [70]:

import numpy as np
import pandas as pd

# Keep exact matches from both stations
s1 = spatial_comparisons["42099"].copy()
s2 = spatial_comparisons["42101"].copy()

s1 = s1[s1["exact_match"] == True]
s2 = s2[s2["exact_match"] == True]

# Keep only candidate timestamps available in both stations
common_exact = s1.join(
    s2,
    how="inner",
    lsuffix="_42099",
    rsuffix="_42101"
)

print("Common exact timestamps:", len(common_exact))

variables = ["temp", "dwpt", "rhum", "wspd", "pres"]

results = []

for variable in variables:

    a = common_exact[f"42099_{variable}"]
    b = common_exact[f"42101_{variable}"]

    valid = a.notna() & b.notna()

    difference = a[valid] - b[valid]

    results.append({
        "variable": variable,
        "valid_pairs": int(valid.sum()),
        "mean_difference_42099_minus_42101":
            difference.mean(),
        "mean_absolute_difference":
            difference.abs().mean(),
        "median_absolute_difference":
            difference.abs().median()
    })

station_agreement = pd.DataFrame(results)

display(station_agreement)


Common exact timestamps: 15


,variable,valid_pairs,mean_difference_42099_minus_42101,mean_absolute_difference,median_absolute_difference
0,temp,15,-0.200000,1.320000,1.4
1,dwpt,15,-0.066667,2.586667,1.9
2,rhum,15,-2.066667,7.800000,7.0
3,wspd,15,0.773333,2.506667,2.2
4,pres,15,-0.940000,0.940000,0.9


In [71]:

variables = ["temp", "dwpt", "rhum", "wspd", "pres"]

# Use only timestamps where both neighbouring stations
# have exact timestamp matches.
idx = common_exact.index

vild0 = candidate_detail.reindex(idx)

results = []

for variable in variables:

    a = common_exact[f"42099_{variable}"]
    b = common_exact[f"42101_{variable}"]
    c = vild0[variable]

    comparisons = {
        "VILD0_vs_42099": c - a,
        "VILD0_vs_42101": c - b,
        "42099_vs_42101": a - b
    }

    for comparison_name, difference in comparisons.items():

        valid = difference.dropna()

        results.append({
            "variable": variable,
            "comparison": comparison_name,
            "valid_pairs": len(valid),
            "mean_difference": valid.mean(),
            "mean_absolute_difference": valid.abs().mean(),
            "median_absolute_difference": valid.abs().median()
        })

tri_station_summary = pd.DataFrame(results)

display(
    tri_station_summary.round(3)
)


,variable,comparison,valid_pairs,mean_difference,mean_absolute_difference,median_absolute_difference
0,temp,VILD0_vs_42099,15,0.780,1.900,1.6
1,temp,VILD0_vs_42101,15,0.580,2.580,2.5
2,temp,42099_vs_42101,15,-0.200,1.320,1.4
3,dwpt,VILD0_vs_42099,15,-6.167,6.247,5.4
4,dwpt,VILD0_vs_42101,15,-6.233,6.233,6.7
5,dwpt,42099_vs_42101,15,-0.067,2.587,1.9
6,rhum,VILD0_vs_42099,15,-9.600,11.467,10.0
7,rhum,VILD0_vs_42101,15,-11.667,12.200,10.0
8,rhum,42099_vs_42101,15,-2.067,7.800,7.0
9,wspd,VILD0_vs_42099,15,7.560,7.907,7.2


In [72]:

variables = ["temp", "dwpt", "rhum", "wspd", "pres"]

idx = common_exact.index
vild0 = candidate_detail.reindex(idx)

candidate_spatial_rows = []

for variable in variables:

    s1 = common_exact[f"42099_{variable}"]
    s2 = common_exact[f"42101_{variable}"]
    v = vild0[variable]

    valid = s1.notna() & s2.notna() & v.notna()

    lower = pd.concat([s1, s2], axis=1).min(axis=1)
    upper = pd.concat([s1, s2], axis=1).max(axis=1)

    neighbour_midpoint = (s1 + s2) / 2

    result = pd.DataFrame({
        "vild0": v,
        "station_42099": s1,
        "station_42101": s2,
        "neighbour_spread": (s1 - s2).abs(),
        "vild0_deviation_from_midpoint":
            v - neighbour_midpoint,
        "vild0_outside_neighbour_range":
            (v < lower) | (v > upper)
    })

    result["variable"] = variable
    result["valid_comparison"] = valid

    candidate_spatial_rows.append(result)

candidate_spatial_detail = pd.concat(
    candidate_spatial_rows
).reset_index(names="candidate_time")

# Summary by variable
spatial_outlier_summary = (
    candidate_spatial_detail[
        candidate_spatial_detail["valid_comparison"]
    ]
    .groupby("variable")
    .agg(
        valid_pairs=("vild0", "count"),
        outside_neighbour_range=(
            "vild0_outside_neighbour_range", "sum"
        ),
        median_neighbour_spread=("neighbour_spread", "median"),
        median_vild0_deviation=(
            "vild0_deviation_from_midpoint",
            lambda x: x.abs().median()
        )
    )
    .reset_index()
)

display(spatial_outlier_summary)

# Show the individual timestamp-level comparisons
display(
    candidate_spatial_detail[
        candidate_spatial_detail["valid_comparison"]
    ].sort_values(
        "vild0_deviation_from_midpoint",
        key=lambda x: x.abs(),
        ascending=False
    ).head(20)
)


,variable,valid_pairs,outside_neighbour_range,median_neighbour_spread,median_vild0_deviation
0,dwpt,15,13,1.9,6.05
1,pres,15,9,0.9,0.65
2,rhum,15,13,7.0,7.5
3,temp,15,11,1.4,1.6
4,wspd,15,12,2.2,8.3


,candidate_time,vild0,station_42099,station_42101,neighbour_spread,vild0_deviation_from_midpoint,vild0_outside_neighbour_range,variable,valid_comparison
34,2025-04-20 11:30:00+05:30,19.0,42.0,46.0,4.0,-25.0,True,rhum,True
33,2025-04-19 11:30:00+05:30,28.0,47.0,55.0,8.0,-23.0,True,rhum,True
48,2025-04-19 11:30:00+05:30,31.7,13.0,9.4,3.6,20.5,True,wspd,True
36,2025-04-22 11:30:00+05:30,12.0,34.0,28.0,6.0,-19.0,True,rhum,True
32,2025-04-18 11:30:00+05:30,22.0,39.0,42.0,3.0,-18.5,True,rhum,True
43,2025-07-29 11:30:00+05:30,76.0,86.0,100.0,14.0,-17.0,True,rhum,True
54,2025-04-26 17:30:00+05:30,22.0,5.4,5.4,0.0,16.6,True,wspd,True
30,2025-03-28 17:30:00+05:30,13.0,21.0,32.0,11.0,-13.5,True,rhum,True
46,2025-03-29 14:30:00+05:30,22.0,9.4,9.4,0.0,12.6,True,wspd,True
49,2025-04-20 11:30:00+05:30,24.5,9.4,14.8,5.4,12.4,True,wspd,True


In [73]:

from meteostat import Stations
import pandas as pd
import numpy as np

station_ids = ["VILD0", "42099", "42101"]

metadata_rows = []

for station_id in station_ids:
    try:
        result = Stations().id(station_id).fetch()

        if result.empty:
            print(f"No metadata returned for {station_id}")
            continue

        row = result.iloc[0].to_dict()
        row["station_id"] = station_id
        metadata_rows.append(row)

    except Exception as e:
        print(f"Error for {station_id}: {e}")

station_metadata = pd.DataFrame(metadata_rows)

display(station_metadata)


Error for VILD0: 'Stations' object has no attribute 'id'
Error for 42099: 'Stations' object has no attribute 'id'
Error for 42101: 'Stations' object has no attribute 'id'


""


In [74]:

from meteostat import Stations
import pandas as pd

station_ids = ["VILD0", "42099", "42101"]

# Retrieve the station metadata database
all_stations = Stations().fetch()

print("Available metadata columns:")
print(all_stations.columns.tolist())

print("\nStation ID index name:", all_stations.index.name)

# Match the requested station IDs against the index
all_stations.index = all_stations.index.astype(str)

available_ids = [
    station_id for station_id in station_ids
    if station_id in all_stations.index
]

missing_ids = [
    station_id for station_id in station_ids
    if station_id not in all_stations.index
]

# Extract only the requested stations
station_metadata = all_stations.loc[available_ids].copy()

print("\nFound stations:", available_ids)
print("Not found:", missing_ids)

display(station_metadata)


Available metadata columns:
['name', 'country', 'region', 'wmo', 'icao', 'latitude', 'longitude', 'elevation', 'timezone', 'hourly_start', 'hourly_end', 'daily_start', 'daily_end', 'monthly_start', 'monthly_end']

Station ID index name: id

Found stations: ['VILD0', '42099', '42101']
Not found: []


,name,country,region,wmo,icao,latitude,longitude,elevation,timezone,hourly_start,hourly_end,daily_start,daily_end,monthly_start,monthly_end
id,,,,,,,,,,,,,,,
VILD0,Ludhiana / Dor?ha,IN,PB,<NA>,VILD,30.8547,75.9526,254.0,Asia/Kolkata,2020-06-18,2026-03-09,NaT,NaT,NaT,NaT
42099,Ludhiana,IN,PB,42099,<NA>,30.8667,75.9333,255.0,Asia/Kolkata,1973-01-01,2025-04-05,1901-01-01,2025-08-24,1901-01-01,2020-01-01
42101,Patiala,IN,PB,42101,<NA>,30.3333,76.4667,249.0,Asia/Kolkata,1973-01-01,2025-04-05,1975-03-20,2026-02-28,1978-01-01,2021-01-01


In [75]:

import numpy as np
import pandas as pd

# Calculate geographic distances using the Haversine formula
def haversine_km(lat1, lon1, lat2, lon2):
    radius = 6371.0

    lat1, lon1, lat2, lon2 = np.radians(
        [lat1, lon1, lat2, lon2]
    )

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1) * np.cos(lat2)
        * np.sin(dlon / 2) ** 2
    )

    return 2 * radius * np.arcsin(np.sqrt(a))


reference = all_stations.loc["VILD0"]

for station in ["42099", "42101"]:
    other = all_stations.loc[station]

    distance = haversine_km(
        reference["latitude"],
        reference["longitude"],
        other["latitude"],
        other["longitude"]
    )

    print(f"VILD0 to {station}: {distance:.2f} km")


# Audit matched observation dates against metadata coverage
coverage_results = []

for station in ["42099", "42101"]:

    comparison = spatial_comparisons[station].copy()

    matched = comparison[comparison["matched"]].copy()

    hourly_end = pd.Timestamp(
        all_stations.loc[station, "hourly_end"]
    )

    after_end = (
        matched["station_time"].dt.date
        > hourly_end.date()
    )

    coverage_results.append({
        "station": station,
        "metadata_hourly_end": hourly_end.date(),
        "matched_observations": len(matched),
        "exact_matches": int(matched["exact_match"].sum()),
        "matches_after_hourly_end": int(after_end.sum()),
        "first_matched_date": matched["station_time"].min(),
        "last_matched_date": matched["station_time"].max()
    })

coverage_audit = pd.DataFrame(coverage_results)

display(coverage_audit)


VILD0 to 42099: 2.27 km
VILD0 to 42101: 76.04 km


,station,metadata_hourly_end,matched_observations,exact_matches,matches_after_hourly_end,first_matched_date,last_matched_date
0,42099,2025-04-05,42,15,35,2025-03-28 14:30:00+05:30,2025-08-24 08:30:00+05:30
1,42101,2025-04-05,51,17,43,2025-03-28 14:30:00+05:30,2025-08-17 14:30:00+05:30


In [76]:

from pathlib import Path
import pandas as pd

DATA_DIR = Path("/content/drive/My Drive/weather_anomaly_phase00")

for station in ["42099", "42101"]:
    path = DATA_DIR / f"{station}_observed.csv"

    df = pd.read_csv(path, index_col=0, parse_dates=True)
    df.index = pd.DatetimeIndex(df.index)

    if df.index.tz is None:
        df.index = df.index.tz_localize("Asia/Kolkata")
    else:
        df.index = df.index.tz_convert("Asia/Kolkata")

    df = df.sort_index()

    print(f"\n{'='*50}")
    print("STATION:", station)
    print("Rows:", len(df))
    print("First timestamp:", df.index.min())
    print("Last timestamp:", df.index.max())
    print("Duplicate timestamps:", df.index.duplicated().sum())

    intervals = (
        df.index.to_series()
        .diff()
        .dropna()
        .dt.total_seconds()
        / 3600
    )

    print("\nInterval distribution (hours):")
    print(intervals.value_counts().head(10))

    print("\nRecords before/after metadata hourly_end:")
    cutoff = pd.Timestamp(
        pd.Timestamp("2025-04-05").date(),
        tz="Asia/Kolkata"
    )

    print("Before or on:", (df.index <= cutoff).sum())
    print("After:", (df.index > cutoff).sum())

    print("\nRecords after cutoff (sample):")
    display(df.loc[df.index > cutoff].head(10))

    print("\nAvailable columns:")
    print(df.columns.tolist())



STATION: 42099
Rows: 567
First timestamp: 2025-03-28 08:30:00+05:30
Last timestamp: 2025-08-24 17:30:00+05:30
Duplicate timestamps: 0

Interval distribution (hours):
time
3.0     398
15.0    135
6.0      16
18.0     12
21.0      2
9.0       1
2.0       1
1.0       1
Name: count, dtype: int64

Records before/after metadata hourly_end:
Before or on: 30
After: 537

Records after cutoff (sample):


,temp,dwpt,rhum,prcp,snow,wdir,wspd,wpgt,pres,tsun,coco
time,,,,,,,,,,,
2025-04-05 11:30:00+05:30,31.8,12.6,31.0,NaN,NaN,340.0,7.6,NaN,1009.3,NaN,NaN
2025-04-05 14:30:00+05:30,34.6,12.8,27.0,NaN,NaN,270.0,3.6,NaN,1006.9,NaN,NaN
2025-04-05 17:30:00+05:30,33.4,13.4,30.0,NaN,NaN,270.0,5.4,NaN,1005.5,NaN,NaN
2025-04-06 08:30:00+05:30,26.8,16.1,52.0,NaN,NaN,180.0,1.8,NaN,1006.8,NaN,NaN
2025-04-06 11:30:00+05:30,34.0,12.9,28.0,NaN,NaN,340.0,1.8,NaN,1007.0,NaN,NaN
2025-04-06 17:30:00+05:30,35.2,15.5,31.0,NaN,NaN,360.0,5.4,NaN,1003.4,NaN,NaN
2025-04-07 08:30:00+05:30,28.0,17.2,52.0,NaN,NaN,140.0,5.4,NaN,1009.6,NaN,NaN
2025-04-07 11:30:00+05:30,35.0,20.1,42.0,NaN,NaN,160.0,7.6,NaN,1009.1,NaN,NaN
2025-04-07 14:30:00+05:30,38.0,16.3,28.0,NaN,NaN,250.0,3.6,NaN,1006.4,NaN,NaN



Available columns:
['temp', 'dwpt', 'rhum', 'prcp', 'snow', 'wdir', 'wspd', 'wpgt', 'pres', 'tsun', 'coco']

STATION: 42101
Rows: 1188
First timestamp: 2025-03-28 02:30:00+05:30
Last timestamp: 2025-08-24 23:30:00+05:30
Duplicate timestamps: 0

Interval distribution (hours):
time
3.0    1176
6.0      10
9.0       1
Name: count, dtype: int64

Records before/after metadata hourly_end:
Before or on: 64
After: 1124

Records after cutoff (sample):


,temp,dwpt,rhum,prcp,snow,wdir,wspd,wpgt,pres,tsun,coco
time,,,,,,,,,,,
2025-04-05 05:30:00+05:30,19.2,13.1,68.0,NaN,NaN,290.0,1.8,NaN,1009.9,NaN,NaN
2025-04-05 08:30:00+05:30,23.6,14.6,57.0,NaN,NaN,290.0,3.6,NaN,1010.4,NaN,NaN
2025-04-05 11:30:00+05:30,32.2,11.9,29.0,NaN,NaN,320.0,3.6,NaN,1010.1,NaN,NaN
2025-04-05 14:30:00+05:30,35.2,15.5,31.0,NaN,NaN,250.0,7.6,NaN,1007.8,NaN,NaN
2025-04-05 17:30:00+05:30,33.4,15.4,34.0,NaN,NaN,320.0,5.4,NaN,1006.3,NaN,NaN
2025-04-05 20:30:00+05:30,25.6,16.2,56.0,NaN,NaN,0.0,0.0,NaN,1008.0,NaN,NaN
2025-04-05 23:30:00+05:30,22.6,15.7,65.0,NaN,NaN,0.0,0.0,NaN,1008.3,NaN,NaN
2025-04-06 02:30:00+05:30,21.6,15.7,69.0,NaN,NaN,0.0,0.0,NaN,1007.0,NaN,NaN
2025-04-06 05:30:00+05:30,19.8,15.4,76.0,NaN,NaN,0.0,0.0,NaN,1006.7,NaN,NaN



Available columns:
['temp', 'dwpt', 'rhum', 'prcp', 'snow', 'wdir', 'wspd', 'wpgt', 'pres', 'tsun', 'coco']


In [77]:

from meteostat import Hourly
from datetime import datetime
from pathlib import Path
import pandas as pd
import numpy as np

DATA_DIR = Path("/content/drive/My Drive/weather_anomaly_phase00")

variables = ["temp", "dwpt", "rhum", "wspd", "pres"]

reproducibility_results = []

for station in ["42099", "42101"]:

    saved = pd.read_csv(
        DATA_DIR / f"{station}_observed.csv",
        index_col=0,
        parse_dates=True
    )

    saved.index = pd.DatetimeIndex(saved.index)
    if saved.index.tz is None:
        saved.index = saved.index.tz_localize("Asia/Kolkata")
    else:
        saved.index = saved.index.tz_convert("Asia/Kolkata")

    fresh = Hourly(
        station,
        datetime(2025, 3, 28),
        datetime(2025, 8, 24, 23, 59),
        timezone="Asia/Kolkata",
        model=False
    ).fetch()

    fresh.index = pd.DatetimeIndex(fresh.index)
    if fresh.index.tz is None:
        fresh.index = fresh.index.tz_localize("Asia/Kolkata")
    else:
        fresh.index = fresh.index.tz_convert("Asia/Kolkata")

    common = saved.index.intersection(fresh.index)

    print(f"\nStation: {station}")
    print("Saved rows:", len(saved))
    print("Fresh rows:", len(fresh))
    print("Common timestamps:", len(common))

    for variable in variables:
        if variable not in saved.columns or variable not in fresh.columns:
            continue

        a = pd.to_numeric(saved.loc[common, variable], errors="coerce")
        b = pd.to_numeric(fresh.loc[common, variable], errors="coerce")

        valid = a.notna() & b.notna()

        equal = np.isclose(
            a[valid],
            b[valid],
            rtol=0,
            atol=1e-8
        )

        reproducibility_results.append({
            "station": station,
            "variable": variable,
            "valid_pairs": int(valid.sum()),
            "equal_values": int(equal.sum()),
            "different_values": int((~equal).sum())
        })

display(pd.DataFrame(reproducibility_results))



Station: 42099
Saved rows: 567
Fresh rows: 567
Common timestamps: 567

Station: 42101
Saved rows: 1188
Fresh rows: 1188
Common timestamps: 1188


,station,variable,valid_pairs,equal_values,different_values
0,42099,temp,566,566,0
1,42099,dwpt,566,566,0
2,42099,rhum,566,566,0
3,42099,wspd,567,567,0
4,42099,pres,567,567,0
5,42101,temp,1187,1187,0
6,42101,dwpt,1187,1187,0
7,42101,rhum,1187,1187,0
8,42101,wspd,1187,1187,0
9,42101,pres,1188,1188,0


In [78]:

import pandas as pd
import numpy as np

variables = ["temp", "dwpt", "rhum", "wspd", "pres"]

spatial_control_results = []
spatial_control_data = {}

# Use the original hourly VILD0 observations
vild0_reference = model_df[variables].copy()

# Ensure consistent timezone
if vild0_reference.index.tz is None:
    vild0_reference.index = (
        vild0_reference.index.tz_localize("Asia/Kolkata")
    )
else:
    vild0_reference.index = (
        vild0_reference.index.tz_convert("Asia/Kolkata")
    )

# Candidate vote counts
votes = corrected_stability_aligned[
    "configuration_votes"
].copy()

for station in ["42099", "42101"]:

    station_df = pd.read_csv(
        DATA_DIR / f"{station}_observed.csv",
        index_col=0,
        parse_dates=True
    )

    station_df.index = pd.DatetimeIndex(station_df.index)

    if station_df.index.tz is None:
        station_df.index = (
            station_df.index.tz_localize("Asia/Kolkata")
        )
    else:
        station_df.index = (
            station_df.index.tz_convert("Asia/Kolkata")
        )

    station_part = station_df[variables].add_prefix(
        f"{station}_"
    )

    vild0_part = vild0_reference.add_prefix("VILD0_")

    # Exact timestamp intersection
    paired = vild0_part.join(
        station_part,
        how="inner"
    )

    paired["configuration_votes"] = votes.reindex(
        paired.index
    )

    # Restrict to the test period with valid vote counts
    paired = paired.dropna(
        subset=["configuration_votes"]
    )

    paired["candidate"] = (
        paired["configuration_votes"] >= 8
    )

    print(f"\nStation {station}")
    print("Exact paired observations:", len(paired))
    print("Candidate observations:", int(paired["candidate"].sum()))
    print("Non-candidate observations:", int((~paired["candidate"]).sum()))

    for variable in variables:

        difference = (
            paired[f"VILD0_{variable}"]
            - paired[f"{station}_{variable}"]
        ).abs()

        valid = difference.notna()

        for group_name, group_mask in [
            ("Candidate", paired["candidate"]),
            ("Non-candidate", ~paired["candidate"])
        ]:

            values = difference[valid & group_mask]

            spatial_control_results.append({
                "station": station,
                "variable": variable,
                "group": group_name,
                "valid_pairs": len(values),
                "mean_absolute_difference": values.mean(),
                "median_absolute_difference": values.median(),
                "p90_absolute_difference": values.quantile(0.90)
            })

    spatial_control_data[station] = paired

spatial_control_summary = pd.DataFrame(
    spatial_control_results
)

display(
    spatial_control_summary.round(3)
)



Station 42099
Exact paired observations: 567
Candidate observations: 15
Non-candidate observations: 552

Station 42101
Exact paired observations: 1188
Candidate observations: 17
Non-candidate observations: 1171


,station,variable,group,valid_pairs,mean_absolute_difference,median_absolute_difference,p90_absolute_difference
0,42099,temp,Candidate,15,1.900,1.6,3.66
1,42099,temp,Non-candidate,551,1.427,1.0,3.30
2,42099,dwpt,Candidate,15,6.247,5.4,13.14
3,42099,dwpt,Non-candidate,551,2.752,1.7,7.10
4,42099,rhum,Candidate,15,11.467,10.0,20.80
5,42099,rhum,Non-candidate,551,8.163,7.0,18.00
6,42099,wspd,Candidate,15,7.907,7.2,16.00
7,42099,wspd,Non-candidate,552,5.088,4.3,10.80
8,42099,pres,Candidate,15,0.887,0.6,1.76
9,42099,pres,Non-candidate,552,1.082,0.9,2.00


In [79]:

for station in ["42099", "42101"]:

    paired = spatial_control_data[station]

    candidate_times = pd.DatetimeIndex(
        paired.index[paired["candidate"]]
    ).sort_values()

    if len(candidate_times) == 0:
        print(f"No candidates for {station}")
        continue

    gaps = candidate_times.to_series().diff()

    # Separate clusters when consecutive candidate matches
    # are more than 24 hours apart.
    cluster_id = (gaps > pd.Timedelta(hours=24)).cumsum()

    cluster_table = pd.DataFrame({
        "candidate_time": candidate_times,
        "cluster_id": cluster_id.to_numpy()
    })

    cluster_summary = (
        cluster_table.groupby("cluster_id")
        .agg(
            first_time=("candidate_time", "min"),
            last_time=("candidate_time", "max"),
            observations=("candidate_time", "size")
        )
        .reset_index()
    )

    print(f"\nStation: {station}")
    print("Candidate observations:", len(candidate_times))
    print("Unique candidate dates:", candidate_times.normalize().nunique())
    print("Temporal clusters (24-hour separation):", len(cluster_summary))

    display(cluster_summary)



Station: 42099
Candidate observations: 15
Unique candidate dates: 13
Temporal clusters (24-hour separation): 9


,cluster_id,first_time,last_time,observations
0,0,2025-03-28 17:30:00+05:30,2025-03-29 14:30:00+05:30,2
1,1,2025-04-18 11:30:00+05:30,2025-04-20 11:30:00+05:30,3
2,2,2025-04-21 17:30:00+05:30,2025-04-22 11:30:00+05:30,2
3,3,2025-04-26 11:30:00+05:30,2025-04-26 17:30:00+05:30,3
4,4,2025-06-28 17:30:00+05:30,2025-06-28 17:30:00+05:30,1
5,5,2025-07-01 14:30:00+05:30,2025-07-01 14:30:00+05:30,1
6,6,2025-07-27 14:30:00+05:30,2025-07-27 14:30:00+05:30,1
7,7,2025-07-29 11:30:00+05:30,2025-07-29 11:30:00+05:30,1
8,8,2025-08-02 17:30:00+05:30,2025-08-02 17:30:00+05:30,1



Station: 42101
Candidate observations: 17
Unique candidate dates: 15
Temporal clusters (24-hour separation): 10


,cluster_id,first_time,last_time,observations
0,0,2025-03-28 17:30:00+05:30,2025-03-29 14:30:00+05:30,2
1,1,2025-04-18 11:30:00+05:30,2025-04-20 11:30:00+05:30,3
2,2,2025-04-21 17:30:00+05:30,2025-04-22 11:30:00+05:30,2
3,3,2025-04-25 20:30:00+05:30,2025-04-26 17:30:00+05:30,4
4,4,2025-06-28 17:30:00+05:30,2025-06-28 17:30:00+05:30,1
5,5,2025-07-01 14:30:00+05:30,2025-07-01 14:30:00+05:30,1
6,6,2025-07-08 02:30:00+05:30,2025-07-08 02:30:00+05:30,1
7,7,2025-07-27 14:30:00+05:30,2025-07-27 14:30:00+05:30,1
8,8,2025-07-29 11:30:00+05:30,2025-07-29 11:30:00+05:30,1
9,9,2025-08-02 17:30:00+05:30,2025-08-02 17:30:00+05:30,1


In [81]:

variables = ["temp", "dwpt", "rhum", "wspd", "pres"]

cluster_level_results = []

for station in ["42099", "42101"]:

    paired = spatial_control_data[station].copy()

    candidate_times = pd.DatetimeIndex(
        paired.index[paired["candidate"]]
    ).sort_values()

    # Assign clusters using a 24-hour separation rule
    gaps = candidate_times.to_series().diff()

    cluster_ids = (
        gaps > pd.Timedelta(hours=24)
    ).cumsum().to_numpy()

    cluster_map = pd.Series(
        cluster_ids,
        index=candidate_times
    )

    paired["cluster_id"] = cluster_map.reindex(
        paired.index
    )

    candidate_rows = paired[
        paired["candidate"]
    ].copy()

    for variable in variables:

        difference_col = f"abs_diff_{variable}"

        candidate_rows[difference_col] = (
            candidate_rows[f"VILD0_{variable}"]
            - candidate_rows[f"{station}_{variable}"]
        ).abs()

        summary = (
            candidate_rows
            .groupby("cluster_id")
            .agg(
                first_time=(difference_col, lambda x: x.index.min()),
                last_time=(difference_col, lambda x: x.index.max()),
                observations=(difference_col, "count"),
                mean_absolute_difference=(difference_col, "mean"),
                median_absolute_difference=(difference_col, "median")
            )
            .reset_index()
        )

        summary["station"] = station
        summary["variable"] = variable

        cluster_level_results.append(summary)

cluster_level_summary = pd.concat(
    cluster_level_results,
    ignore_index=True
)

display(
    cluster_level_summary.sort_values(
        ["station", "variable", "cluster_id"]
    ).round(3)
)


,cluster_id,first_time,last_time,observations,mean_absolute_difference,median_absolute_difference,station,variable
9,0.0,2025-03-28 17:30:00+05:30,2025-03-29 14:30:00+05:30,2,4.6,4.6,42099,dwpt
10,1.0,2025-04-18 11:30:00+05:30,2025-04-20 11:30:00+05:30,3,6.633,5.4,42099,dwpt
11,2.0,2025-04-21 17:30:00+05:30,2025-04-22 11:30:00+05:30,2,11.8,11.8,42099,dwpt
12,3.0,2025-04-26 11:30:00+05:30,2025-04-26 17:30:00+05:30,3,12.7,12.9,42099,dwpt
13,4.0,2025-06-28 17:30:00+05:30,2025-06-28 17:30:00+05:30,1,1.6,1.6,42099,dwpt
...,...,...,...,...,...,...,...,...
80,5.0,2025-07-01 14:30:00+05:30,2025-07-01 14:30:00+05:30,1,3.4,3.4,42101,wspd
81,6.0,2025-07-08 02:30:00+05:30,2025-07-08 02:30:00+05:30,1,13.7,13.7,42101,wspd
82,7.0,2025-07-27 14:30:00+05:30,2025-07-27 14:30:00+05:30,1,1.4,1.4,42101,wspd
83,8.0,2025-07-29 11:30:00+05:30,2025-07-29 11:30:00+05:30,1,7.9,7.9,42101,wspd


In [82]:

# One row per station and variable
compact_cluster_summary = (
    cluster_level_summary
    .groupby(["station", "variable"])
    .agg(
        cluster_count=("cluster_id", "nunique"),
        total_observations=("observations", "sum"),
        mean_cluster_difference=(
            "mean_absolute_difference", "mean"
        ),
        median_cluster_difference=(
            "mean_absolute_difference", "median"
        ),
        minimum_cluster_difference=(
            "mean_absolute_difference", "min"
        ),
        maximum_cluster_difference=(
            "mean_absolute_difference", "max"
        )
    )
    .reset_index()
)

display(compact_cluster_summary.round(3))


,station,variable,cluster_count,total_observations,mean_cluster_difference,median_cluster_difference,minimum_cluster_difference,maximum_cluster_difference
0,42099,dwpt,9,15,4.293,1.6,0.1,12.7
1,42099,pres,9,15,0.811,0.6,0.2,2.0
2,42099,rhum,9,15,10.111,10.0,1.0,19.667
3,42099,temp,9,15,1.72,1.4,0.0,3.733
4,42099,wspd,9,15,6.07,5.2,0.2,13.667
5,42101,dwpt,10,17,4.668,2.95,0.6,9.2
6,42101,pres,10,17,0.624,0.437,0.0,1.967
7,42101,rhum,10,17,10.542,10.125,0.0,24.667
8,42101,temp,10,17,2.54,2.475,1.1,4.2
9,42101,wspd,10,17,8.585,9.15,1.4,13.8


In [83]:

top_clusters = (
    cluster_level_summary
    .sort_values(
        "mean_absolute_difference",
        ascending=False
    )
    .groupby(["station", "variable"])
    .head(3)
    .sort_values(["station", "variable"])
)

display(top_clusters.round(3))


,cluster_id,first_time,last_time,observations,mean_absolute_difference,median_absolute_difference,station,variable
12,3.0,2025-04-26 11:30:00+05:30,2025-04-26 17:30:00+05:30,3,12.7,12.9,42099,dwpt
11,2.0,2025-04-21 17:30:00+05:30,2025-04-22 11:30:00+05:30,2,11.8,11.8,42099,dwpt
10,1.0,2025-04-18 11:30:00+05:30,2025-04-20 11:30:00+05:30,3,6.633,5.4,42099,dwpt
42,6.0,2025-07-27 14:30:00+05:30,2025-07-27 14:30:00+05:30,1,2.0,2.0,42099,pres
37,1.0,2025-04-18 11:30:00+05:30,2025-04-20 11:30:00+05:30,3,1.333,1.4,42099,pres
39,3.0,2025-04-26 11:30:00+05:30,2025-04-26 17:30:00+05:30,3,1.167,1.2,42099,pres
19,1.0,2025-04-18 11:30:00+05:30,2025-04-20 11:30:00+05:30,3,19.667,19.0,42099,rhum
20,2.0,2025-04-21 17:30:00+05:30,2025-04-22 11:30:00+05:30,2,17.5,17.5,42099,rhum
26,8.0,2025-08-02 17:30:00+05:30,2025-08-02 17:30:00+05:30,1,14.0,14.0,42099,rhum
1,1.0,2025-04-18 11:30:00+05:30,2025-04-20 11:30:00+05:30,3,3.733,3.7,42099,temp


In [84]:

for station in ["42099", "42101"]:

    paired = spatial_control_data[station].copy()

    paired["month"] = paired.index.month
    paired["hour"] = paired.index.hour

    candidate = paired[paired["candidate"]]
    control = paired[~paired["candidate"]]

    print(f"\nStation: {station}")

    availability = (
        candidate.groupby(["month", "hour"])
        .size()
        .rename("candidate_count")
        .to_frame()
    )

    availability["available_controls"] = (
        control.groupby(["month", "hour"])
        .size()
    )

    availability["available_controls"] = (
        availability["available_controls"].fillna(0).astype(int)
    )

    display(availability.reset_index())



Station: 42099


,month,hour,candidate_count,available_controls
0,3,14,1,3
1,3,17,1,3
2,4,11,5,24
3,4,14,1,27
4,4,17,2,28
5,6,17,1,26
6,7,11,1,29
7,7,14,2,28
8,8,17,1,21



Station: 42101


,month,hour,candidate_count,available_controls
0,3,14,1,3
1,3,17,1,3
2,4,11,5,25
3,4,14,1,28
4,4,17,2,28
5,4,20,1,29
6,6,17,1,28
7,7,2,1,29
8,7,11,1,30
9,7,14,2,29


In [85]:
for station in ["42099", "42101"]:
    paired = spatial_control_data[station]

    print(f"\nStation: {station}")
    print("Shape:", paired.shape)
    print("Columns:")
    print(paired.columns.tolist())

    print("\nCandidate observations:")
    print(paired["candidate"].value_counts())

    print("\nDate range:")
    print(paired.index.min(), "to", paired.index.max())


Station: 42099
Shape: (567, 12)
Columns:
['VILD0_temp', 'VILD0_dwpt', 'VILD0_rhum', 'VILD0_wspd', 'VILD0_pres', '42099_temp', '42099_dwpt', '42099_rhum', '42099_wspd', '42099_pres', 'configuration_votes', 'candidate']

Candidate observations:
candidate
False    552
True      15
Name: count, dtype: int64

Date range:
2025-03-28 08:30:00+05:30 to 2025-08-24 17:30:00+05:30

Station: 42101
Shape: (1188, 12)
Columns:
['VILD0_temp', 'VILD0_dwpt', 'VILD0_rhum', 'VILD0_wspd', 'VILD0_pres', '42101_temp', '42101_dwpt', '42101_rhum', '42101_wspd', '42101_pres', 'configuration_votes', 'candidate']

Candidate observations:
candidate
False    1171
True       17
Name: count, dtype: int64

Date range:
2025-03-28 02:30:00+05:30 to 2025-08-24 23:30:00+05:30


In [88]:

import numpy as np
import pandas as pd
from scipy.optimize import linear_sum_assignment

stations = ["42099", "42101"]
variables = ["temp", "dwpt", "rhum", "wspd", "pres"]

all_matches = {}
match_summaries = {}
candidate_match_counts = {}

MAX_DAYS = 3
MAX_CONTROLS = 2
MAX_OFFSET_MINUTES = MAX_DAYS * 24 * 60

for station in stations:
    paired = spatial_control_data[station].copy()

    candidate_times = paired.index[paired["candidate"]]
    control_times = paired.index[~paired["candidate"]]

    # Create two matching slots for every candidate.
    slots = [
        (candidate_time, slot_number)
        for candidate_time in candidate_times
        for slot_number in range(MAX_CONTROLS)
    ]

    n_slots = len(slots)
    n_controls = len(control_times)

    # Each slot can be assigned to a unique control timestamp,
    # or to a dummy column if no eligible control is available.
    unmatched_penalty = MAX_OFFSET_MINUTES + 1
    invalid_cost = 1e9

    cost = np.full(
        (n_slots, n_controls + n_slots),
        unmatched_penalty,
        dtype=float
    )

    # Dummy columns represent unmatched slots.
    cost[:, :n_controls] = invalid_cost

    for row, (candidate_time, slot_number) in enumerate(slots):
        for col, control_time in enumerate(control_times):

            same_month = candidate_time.month == control_time.month
            same_hour = candidate_time.hour == control_time.hour

            offset_minutes = abs(
                (control_time - candidate_time).total_seconds()
            ) / 60

            within_window = offset_minutes <= MAX_OFFSET_MINUTES

            if same_month and same_hour and within_window:
                cost[row, col] = offset_minutes

    row_ind, col_ind = linear_sum_assignment(cost)

    matched_rows = []

    for row, col in zip(row_ind, col_ind):
        if col >= n_controls:
            continue

        if cost[row, col] >= unmatched_penalty:
            continue

        candidate_time, slot_number = slots[row]
        control_time = control_times[col]

        matched_rows.append({
            "candidate_time": candidate_time,
            "control_time": control_time,
            "slot": slot_number + 1,
            "offset_hours": abs(
                (control_time - candidate_time).total_seconds()
            ) / 3600
        })

    matches = pd.DataFrame(matched_rows)

    if not matches.empty:
        matches = matches.sort_values(
            ["candidate_time", "offset_hours"]
        ).reset_index(drop=True)

    all_matches[station] = matches

    # Count matched controls per candidate, including candidates
    # that did not receive a control.
    counts = (
        matches.groupby("candidate_time")
        .size()
        .reindex(candidate_times, fill_value=0)
        .rename("matched_controls")
        .reset_index()
        .rename(columns={"index": "candidate_time"})
    )

    candidate_match_counts[station] = counts

    # Calculate absolute station-to-VILD0 differences.
    # First calculate candidate discrepancies.
    candidate_discrepancies = {}

    for variable in variables:
        station_col = f"{station}_{variable}"
        vild_col = f"VILD0_{variable}"

        candidate_discrepancies[variable] = (
            paired.loc[candidate_times, station_col]
            - paired.loc[candidate_times, vild_col]
        ).abs()

    # Calculate matched-control discrepancies.
    control_rows = []

    for _, match in matches.iterrows():
        candidate_time = match["candidate_time"]
        control_time = match["control_time"]

        row = {
            "candidate_time": candidate_time,
            "control_time": control_time,
            "offset_hours": match["offset_hours"]
        }

        for variable in variables:
            station_col = f"{station}_{variable}"
            vild_col = f"VILD0_{variable}"

            row[variable] = abs(
                paired.loc[control_time, station_col]
                - paired.loc[control_time, vild_col]
            )

        control_rows.append(row)

    control_discrepancies = pd.DataFrame(control_rows)

    # Average multiple controls within each candidate first.
    # This gives each candidate equal weight in the final comparison.
    if not control_discrepancies.empty:
        per_candidate_control = (
            control_discrepancies
            .groupby("candidate_time")[variables]
            .mean()
        )
    else:
        per_candidate_control = pd.DataFrame(columns=variables)

    summary_rows = []

    for variable in variables:
        candidate_values = candidate_discrepancies[variable].dropna()
        control_values = per_candidate_control[variable].dropna()

        summary_rows.append({
            "variable": variable,
            "candidate_count": len(candidate_values),
            "matched_candidate_count": len(control_values),
            "candidate_mean_abs_difference": candidate_values.mean(),
            "matched_control_mean_abs_difference": control_values.mean(),
            "difference_candidate_minus_control": (
                candidate_values.mean() - control_values.mean()
            )
        })

    match_summaries[station] = pd.DataFrame(summary_rows)

    print(f"\n{'='*60}")
    print(f"STATION: {station}")
    print(f"{'='*60}")

    print("\nMatching coverage:")
    print("Candidates:", len(candidate_times))
    print("Candidates with 0 controls:", (counts["matched_controls"] == 0).sum())
    print("Candidates with 1 control:", (counts["matched_controls"] == 1).sum())
    print("Candidates with 2 controls:", (counts["matched_controls"] == 2).sum())
    print("Total unique controls matched:", len(matches))

    if not matches.empty:
        print("\nControl offset (hours):")
        display(matches["offset_hours"].describe())

    print("\nCandidate-level matching:")
    display(counts)

    print("\nMatched-control comparison:")
    display(match_summaries[station].round(3))



STATION: 42099

Matching coverage:
Candidates: 15
Candidates with 0 controls: 1
Candidates with 1 control: 1
Candidates with 2 controls: 13
Total unique controls matched: 27

Control offset (hours):


,offset_hours
count,27.000000
mean,29.333333
std,10.167823
min,24.000000
25%,24.000000
50%,24.000000
75%,24.000000
max,48.000000



Candidate-level matching:


,time,matched_controls
0,2025-03-28 17:30:00+05:30,2
1,2025-03-29 14:30:00+05:30,2
2,2025-04-18 11:30:00+05:30,2
3,2025-04-19 11:30:00+05:30,0
4,2025-04-20 11:30:00+05:30,1
5,2025-04-21 17:30:00+05:30,2
6,2025-04-22 11:30:00+05:30,2
7,2025-04-26 11:30:00+05:30,2
8,2025-04-26 14:30:00+05:30,2
9,2025-04-26 17:30:00+05:30,2



Matched-control comparison:


,variable,candidate_count,matched_candidate_count,candidate_mean_abs_difference,matched_control_mean_abs_difference,difference_candidate_minus_control
0,temp,15,14,1.900,1.171,0.729
1,dwpt,15,14,6.247,4.568,1.679
2,rhum,15,14,11.467,7.250,4.217
3,wspd,15,14,7.907,5.214,2.692
4,pres,15,14,0.887,1.118,-0.231


... (2 outputs left out)


Candidate-level matching:


,time,matched_controls
0,2025-03-28 17:30:00+05:30,2
1,2025-03-29 14:30:00+05:30,2
2,2025-04-18 11:30:00+05:30,2
3,2025-04-19 11:30:00+05:30,0
4,2025-04-20 11:30:00+05:30,1
5,2025-04-21 17:30:00+05:30,2
6,2025-04-22 11:30:00+05:30,2
7,2025-04-25 20:30:00+05:30,2
8,2025-04-26 11:30:00+05:30,2
9,2025-04-26 14:30:00+05:30,2



Matched-control comparison:


,variable,candidate_count,matched_candidate_count,candidate_mean_abs_difference,matched_control_mean_abs_difference,difference_candidate_minus_control
0,temp,17,16,2.559,1.675,0.884
1,dwpt,17,16,6.365,5.734,0.630
2,rhum,17,16,12.118,8.750,3.368
3,wspd,17,16,9.782,6.541,3.242
4,pres,17,16,0.759,0.756,0.003


In [87]:

variables = ["temp", "dwpt", "rhum", "wspd", "pres"]

cluster_effects = {}
candidate_effects = {}

for station in ["42099", "42101"]:
    paired = spatial_control_data[station].copy()
    matches = all_matches[station].copy()

    # Average the two matched controls for each candidate.
    control_rows = []

    for _, match in matches.iterrows():
        candidate_time = match["candidate_time"]
        control_time = match["control_time"]

        row = {"candidate_time": candidate_time}

        for variable in variables:
            station_col = f"{station}_{variable}"
            vild_col = f"VILD0_{variable}"

            row[variable] = abs(
                paired.loc[control_time, station_col]
                - paired.loc[control_time, vild_col]
            )

        control_rows.append(row)

    control_by_candidate = (
        pd.DataFrame(control_rows)
        .groupby("candidate_time")[variables]
        .mean()
    )

    # Candidate spatial disagreement.
    candidate_times = paired.index[paired["candidate"]]
    candidate_disagreement = pd.DataFrame(index=candidate_times)

    for variable in variables:
        station_col = f"{station}_{variable}"
        vild_col = f"VILD0_{variable}"

        candidate_disagreement[variable] = (
            paired.loc[candidate_times, station_col]
            - paired.loc[candidate_times, vild_col]
        ).abs()

    # Difference: candidate disagreement minus matched-control disagreement.
    effects = (
        candidate_disagreement
        - control_by_candidate.reindex(candidate_times)
    )

    effects.index.name = "candidate_time"
    candidate_effects[station] = effects

    # Recreate 24-hour candidate clusters.
    sorted_times = effects.index.sort_values()
    cluster_id = (
        pd.Series(sorted_times, index=sorted_times)
        .diff()
        .gt(pd.Timedelta(hours=24))
        .cumsum()
    )

    effects = effects.loc[sorted_times].copy()
    effects["cluster_id"] = cluster_id.values

    # Each cluster receives equal weight in the summary.
    cluster_summary = (
        effects.groupby("cluster_id")[variables]
        .mean()
    )

    cluster_effects[station] = cluster_summary

    print(f"\n{'='*55}")
    print(f"STATION: {station}")
    print(f"{'='*55}")

    print("\nCandidate-level effect:")
    print("Positive = candidate has greater spatial disagreement.")
    display(
        pd.DataFrame({
            "median_effect": effects[variables].median(),
            "positive_candidate_count": (effects[variables] > 0).sum(),
            "negative_candidate_count": (effects[variables] < 0).sum()
        }).round(3)
    )

    print("\nCluster-level effect:")
    display(cluster_summary.round(3))

    print("\nNumber of clusters with positive mean effect:")
    display((cluster_summary[variables] > 0).sum().rename("positive_clusters"))



STATION: 42099

Candidate-level effect:
Positive = candidate has greater spatial disagreement.


,median_effect,positive_candidate_count,negative_candidate_count
temp,0.45,8,7
dwpt,1.15,12,3
rhum,3.0,11,2
wspd,5.2,9,5
pres,0.1,8,7



Cluster-level effect:


,temp,dwpt,rhum,wspd,pres
cluster_id,,,,,
0,-0.4,1.3,1.0,3.95,-0.175
1,2.633,1.95,10.667,10.217,0.083
2,1.025,3.625,8.0,-2.45,-1.05
3,0.217,4.117,2.0,2.317,-0.017
4,-0.95,0.85,2.0,0.0,0.3
5,-0.15,-0.8,-2.5,-2.3,0.2
6,-0.2,-0.25,0.0,-2.5,0.85
7,2.3,0.25,7.5,5.2,-1.25
8,1.45,0.35,5.5,-2.65,-0.9



Number of clusters with positive mean effect:


,positive_clusters
temp,5
dwpt,7
rhum,7
wspd,4
pres,4


... (2 outputs left out)


Cluster-level effect:


,temp,dwpt,rhum,wspd,pres
cluster_id,,,,,
0,0.4,1.8,4.25,2.5,0.225
1,2.6,2.1,12.5,10.95,1.4
2,-0.325,1.5,4.0,-1.1,-0.775
3,0.6,-0.187,-1.75,1.05,-0.062
4,0.25,0.45,-15.0,7.2,-1.1
5,0.75,-0.5,-1.0,0.9,-0.75
6,1.1,0.65,2.0,6.5,0.7
7,-1.3,0.4,6.0,-4.2,-0.6
8,3.85,-0.7,16.5,5.7,-0.3



Number of clusters with positive mean effect:


,positive_clusters
temp,8
dwpt,7
rhum,6
wspd,8
pres,4


In [89]:

variables = ["temp", "dwpt", "rhum", "wspd", "pres"]
stations = ["42099", "42101"]

matched_effects_3d = {}
cluster_effects_3d = {}
matched_summaries_3d = {}

for station in stations:
    paired = spatial_control_data[station].copy()
    matches = all_matches[station].copy()

    if matches.empty:
        print(f"No matches for {station}")
        continue

    # Average the matched controls for each candidate.
    control_rows = []

    for _, match in matches.iterrows():
        candidate_time = match["candidate_time"]
        control_time = match["control_time"]

        row = {"candidate_time": candidate_time}

        for variable in variables:
            station_col = f"{station}_{variable}"
            vild_col = f"VILD0_{variable}"

            row[variable] = abs(
                paired.loc[control_time, station_col]
                - paired.loc[control_time, vild_col]
            )

        control_rows.append(row)

    control_by_candidate = (
        pd.DataFrame(control_rows)
        .groupby("candidate_time")[variables]
        .mean()
    )

    # Only candidates that received controls are included.
    matched_times = control_by_candidate.index

    candidate_disagreement = pd.DataFrame(
        index=matched_times
    )

    for variable in variables:
        station_col = f"{station}_{variable}"
        vild_col = f"VILD0_{variable}"

        candidate_disagreement[variable] = (
            paired.loc[matched_times, station_col]
            - paired.loc[matched_times, vild_col]
        ).abs()

    # Matched candidate minus matched-control disagreement.
    effects = candidate_disagreement - control_by_candidate
    matched_effects_3d[station] = effects

    # Summary uses exactly the same matched candidates on both sides.
    summary_rows = []

    for variable in variables:
        valid = pd.concat(
            [
                candidate_disagreement[variable].rename("candidate"),
                control_by_candidate[variable].rename("control")
            ],
            axis=1
        ).dropna()

        summary_rows.append({
            "variable": variable,
            "matched_candidates": len(valid),
            "candidate_mean": valid["candidate"].mean(),
            "control_mean": valid["control"].mean(),
            "candidate_minus_control": (
                valid["candidate"] - valid["control"]
            ).mean()
        })

    summary = pd.DataFrame(summary_rows)
    matched_summaries_3d[station] = summary

    # Preserve the original 24-hour candidate cluster definitions.
    all_candidate_times = (
        paired.index[paired["candidate"]].sort_values()
    )

    cluster_ids = (
        pd.Series(all_candidate_times, index=all_candidate_times)
        .diff()
        .gt(pd.Timedelta(hours=24))
        .cumsum()
    )

    cluster_map = cluster_ids.to_dict()

    effects_with_cluster = effects.copy()
    effects_with_cluster["cluster_id"] = [
        cluster_map[t] for t in effects.index
    ]

    cluster_summary = (
        effects_with_cluster
        .groupby("cluster_id")[variables]
        .mean()
    )

    cluster_effects_3d[station] = cluster_summary

    print(f"\n{'='*55}")
    print(f"STATION: {station} — 72-HOUR MATCHING")
    print(f"{'='*55}")

    print("\nCorrected matched comparison:")
    display(summary.round(3))

    print("\nCandidate-level effect:")
    display(pd.DataFrame({
        "median_effect": effects.median(),
        "positive_candidates": (effects > 0).sum(),
        "negative_candidates": (effects < 0).sum()
    }).round(3))

    print("\nCluster-level effect:")
    display(cluster_summary.round(3))

    print("\nPositive clusters:")
    display(
        (cluster_summary[variables] > 0)
        .sum()
        .rename("positive_clusters")
    )



STATION: 42099 — 72-HOUR MATCHING

Corrected matched comparison:


,variable,matched_candidates,candidate_mean,control_mean,candidate_minus_control
0,temp,14,1.800,1.171,0.629
1,dwpt,14,6.307,4.568,1.739
2,rhum,14,10.929,7.250,3.679
3,wspd,14,7.136,5.214,1.921
4,pres,14,0.850,1.118,-0.268



Candidate-level effect:


,median_effect,positive_candidates,negative_candidates
temp,0.15,7,7
dwpt,1.0,10,4
rhum,3.0,10,2
wspd,2.225,9,4
pres,-0.275,6,8



Cluster-level effect:


,temp,dwpt,rhum,wspd,pres
cluster_id,,,,,
0,-0.4,1.3,1.0,3.95,-0.175
1,2.125,0.15,6.0,9.6,-0.275
2,1.175,4.5,8.25,-4.6,-0.9
3,0.183,4.017,2.833,3.75,-0.083
4,-0.95,0.85,2.0,0.0,0.3
5,-0.15,-0.8,-2.5,-2.3,0.2
6,-0.2,-0.25,0.0,-2.5,0.85
7,2.3,0.25,7.5,5.2,-1.25
8,1.45,0.35,5.5,-2.65,-0.9


... (6 outputs left out)


Cluster-level effect:


,temp,dwpt,rhum,wspd,pres
cluster_id,,,,,
0,0.4,1.8,4.25,2.5,0.225
1,1.625,0.95,9.0,7.3,0.4
2,-0.275,3.425,5.75,-1.25,-0.65
3,0.7,-0.625,-1.125,1.125,-0.187
4,0.25,0.45,-15.0,7.2,-1.1
5,0.75,-0.5,-1.0,0.9,-0.75
6,1.1,0.65,2.0,6.5,0.7
7,-1.3,0.4,6.0,-4.2,-0.6
8,3.85,-0.7,16.5,5.7,-0.3



Positive clusters:


,positive_clusters
temp,8
dwpt,7
rhum,6
wspd,8
pres,4


In [90]:
import pandas as pd

variables = ["temp", "dwpt", "rhum", "prcp", "wspd", "pres"]

stuck_results = []

for segment_id, group in model_df.groupby("segment_id"):

    group = group.sort_index()

    for variable in variables:

        values = group[variable]

        # Consecutive identical observations
        same_as_previous = values.eq(values.shift())

        # Identify continuous runs of identical values
        run_id = (~same_as_previous).cumsum()

        run_lengths = (
            values.groupby(run_id)
            .transform("size")
        )

        longest_run = run_lengths.max()

        stuck_results.append({
            "segment_id": segment_id,
            "variable": variable,
            "longest_identical_run": int(longest_run),
            "records_in_runs_6_or_more": int(
                run_lengths.ge(6).sum()
            ),
            "records_in_runs_12_or_more": int(
                run_lengths.ge(12).sum()
            )
        })

stuck_summary = pd.DataFrame(stuck_results)

display(
    stuck_summary.sort_values(
        "longest_identical_run",
        ascending=False
    )
)

,segment_id,variable,longest_identical_run,records_in_runs_6_or_more,records_in_runs_12_or_more
9,103,prcp,1707,2228,2199
21,145,prcp,1359,6973,6695
3,76,prcp,1077,28665,27731
15,118,prcp,562,2073,2017
27,154,prcp,471,575,575
14,118,rhum,10,56,0
20,145,rhum,9,86,0
2,76,rhum,8,209,0
10,103,wspd,7,7,0
4,76,wspd,7,43,0


In [91]:
import pandas as pd

precipitation_audit = []

for segment_id, group in model_df.groupby("segment_id"):

    s = group.sort_index()["prcp"]

    valid = s.notna()
    same_as_previous = (
        s.eq(s.shift())
        & valid
        & valid.shift(fill_value=False)
    )

    run_id = (~same_as_previous).cumsum()
    run_length = s.groupby(run_id).transform("size")

    zero_mask = valid & s.eq(0)
    nonzero_mask = valid & s.ne(0)

    precipitation_audit.append({
        "segment_id": segment_id,
        "total_records": len(s),
        "missing_records": int(s.isna().sum()),
        "zero_records": int(zero_mask.sum()),
        "nonzero_records": int(nonzero_mask.sum()),
        "unique_nonzero_values": s[nonzero_mask].nunique(),
        "longest_zero_run": int(
            run_length[zero_mask].max()
        ) if zero_mask.any() else 0,
        "longest_nonzero_run": int(
            run_length[nonzero_mask].max()
        ) if nonzero_mask.any() else 0,
        "nonzero_records_in_runs_6_plus": int(
            (nonzero_mask & run_length.ge(6)).sum()
        )
    })

display(pd.DataFrame(precipitation_audit))

,segment_id,total_records,missing_records,zero_records,nonzero_records,unique_nonzero_values,longest_zero_run,longest_nonzero_run,nonzero_records_in_runs_6_plus
0,76,31931,1,29485,2445,105,1077,5,0
1,103,2332,1,2254,77,22,1707,3,0
2,118,2189,1,2093,95,31,562,2,0
3,145,7849,2,7188,659,83,1359,3,0
4,154,578,1,576,1,1,471,1,0


In [92]:
display(
    model_df["prcp"]
    .value_counts(dropna=False)
    .head(15)
)

,count
prcp,
0.0,41596
0.1,682
0.2,504
0.3,284
0.4,188
0.5,180
0.7,117
0.6,104
0.8,88


In [93]:
import pandas as pd

variables = ["temp", "dwpt", "rhum", "wspd", "pres"]

stuck_results = []

for segment_id, group in model_df.groupby("segment_id"):

    group = group.sort_index()

    for variable in variables:

        s = group[variable]

        valid = s.notna()

        same_as_previous = (
            s.eq(s.shift())
            & valid
            & valid.shift(fill_value=False)
        )

        run_id = (~same_as_previous).cumsum()

        run_length = s.groupby(run_id).transform("size")

        valid_runs = run_length[valid]

        stuck_results.append({
            "segment_id": segment_id,
            "variable": variable,
            "longest_identical_run": int(
                valid_runs.max()
            ) if len(valid_runs) else 0,
            "records_in_runs_6_or_more": int(
                (valid & run_length.ge(6)).sum()
            ),
            "records_in_runs_12_or_more": int(
                (valid & run_length.ge(12)).sum()
            ),
            "missing_records": int(s.isna().sum())
        })

stuck_summary_corrected = pd.DataFrame(stuck_results)

display(
    stuck_summary_corrected.sort_values(
        "longest_identical_run",
        ascending=False
    )
)

,segment_id,variable,longest_identical_run,records_in_runs_6_or_more,records_in_runs_12_or_more,missing_records
12,118,rhum,10,56,0,0
17,145,rhum,9,86,0,0
2,76,rhum,8,209,0,0
8,103,wspd,7,7,0,0
3,76,wspd,7,43,0,0
13,118,wspd,6,6,0,0
16,145,dwpt,6,6,0,0
7,103,rhum,6,12,0,0
4,76,pres,6,6,0,0
18,145,wspd,6,24,0,0


In [94]:
import pandas as pd

variables = ["temp", "dwpt", "rhum", "wspd", "pres"]

long_runs = []

for segment_id, group in model_df.groupby("segment_id"):

    group = group.sort_index()

    for variable in variables:

        s = group[variable]
        valid = s.notna()

        same_as_previous = (
            s.eq(s.shift())
            & valid
            & valid.shift(fill_value=False)
        )

        run_id = (~same_as_previous).cumsum()

        run_lengths = s.groupby(run_id).transform("size")

        # Inspect runs of at least 6 observations
        qualifying_ids = run_lengths[
            valid & run_lengths.ge(6)
        ].index

        for rid in run_lengths[valid & run_lengths.ge(6)].unique():

            run_index = run_lengths[
                (run_lengths == rid)
            ].index

            # Get the actual consecutive run
            run_index = run_index[
                s.loc[run_index].notna()
            ]

            run_values = s.loc[run_index]

            long_runs.append({
                "segment_id": segment_id,
                "variable": variable,
                "start": run_index.min(),
                "end": run_index.max(),
                "records": len(run_index),
                "value": run_values.iloc[0]
            })

long_runs_df = pd.DataFrame(long_runs)

display(
    long_runs_df.sort_values(
        ["records", "variable"],
        ascending=[False, True]
    )
)

,segment_id,variable,start,end,records,value
1,76,rhum,2021-01-12 02:30:00+05:30,2024-01-29 04:30:00+05:30,114,100.0
0,76,rhum,2021-01-08 00:30:00+05:30,2023-01-05 07:30:00+05:30,63,100.0
14,145,rhum,2025-09-01 19:30:00+05:30,2026-01-11 07:30:00+05:30,49,94.0
4,76,wspd,2021-10-28 23:30:00+05:30,2023-12-30 01:30:00+05:30,36,9.4
2,76,rhum,2021-01-12 23:30:00+05:30,2023-01-09 07:30:00+05:30,32,100.0
8,118,rhum,2024-12-29 02:30:00+05:30,2025-01-22 07:30:00+05:30,24,100.0
18,145,wspd,2025-11-09 17:30:00+05:30,2025-12-14 14:30:00+05:30,24,7.6
16,145,rhum,2026-01-04 00:30:00+05:30,2026-01-15 04:30:00+05:30,16,100.0
9,118,rhum,2025-01-01 00:30:00+05:30,2025-01-04 03:30:00+05:30,14,100.0
6,103,rhum,2024-11-29 22:30:00+05:30,2024-12-05 02:30:00+05:30,12,89.0


In [95]:
long_runs = []

for segment_id, group in model_df.groupby("segment_id"):

    group = group.sort_index()

    for variable in variables:

        s = group[variable]
        valid = s.notna()

        same = (
            s.eq(s.shift())
            & valid
            & valid.shift(fill_value=False)
        )

        run_id = (~same).cumsum()

        for rid, run in s[valid].groupby(run_id[valid]):

            if len(run) >= 6:
                long_runs.append({
                    "segment_id": segment_id,
                    "variable": variable,
                    "start": run.index[0],
                    "end": run.index[-1],
                    "records": len(run),
                    "value": run.iloc[0]
                })

long_runs_df = pd.DataFrame(long_runs)

display(
    long_runs_df.sort_values(
        "records", ascending=False
    )
)

,segment_id,variable,start,end,records,value
48,118,rhum,2025-01-13 22:30:00+05:30,2025-01-14 07:30:00+05:30,10,92.0
62,145,rhum,2026-01-08 23:30:00+05:30,2026-01-09 07:30:00+05:30,9,100.0
7,76,rhum,2021-09-03 23:30:00+05:30,2021-09-04 06:30:00+05:30,8,99.0
16,76,rhum,2022-02-02 00:30:00+05:30,2022-02-02 07:30:00+05:30,8,100.0
3,76,rhum,2021-01-12 23:30:00+05:30,2021-01-13 06:30:00+05:30,8,100.0
...,...,...,...,...,...,...
61,145,rhum,2026-01-08 01:30:00+05:30,2026-01-08 06:30:00+05:30,6,100.0
65,145,wspd,2025-11-09 17:30:00+05:30,2025-11-09 22:30:00+05:30,6,7.6
66,145,wspd,2025-11-16 19:30:00+05:30,2025-11-17 00:30:00+05:30,6,6.5
67,145,wspd,2025-11-22 09:30:00+05:30,2025-11-22 14:30:00+05:30,6,2.0


In [97]:
import pandas as pd

variables = ["temp", "dwpt", "rhum", "wspd", "pres"]

verified_runs = []

for segment_id, group in model_df.groupby("segment_id"):

    group = group.sort_index()

    for variable in variables:

        s = group[variable]

        # Start a new run when:
        # 1. Value changes
        # 2. Current or previous value is missing
        # 3. Time gap is not exactly one hour

        same_value = s.eq(s.shift())
        both_valid = s.notna() & s.shift().notna()

        consecutive_hour = (
            group.index.to_series()
            .diff()
            .eq(pd.Timedelta(hours=1))
        )

        same_run = (
            same_value
            & both_valid
            & consecutive_hour
        )

        run_id = (~same_run).cumsum()

        for _, run in s.groupby(run_id):

            if run.notna().all() and len(run) >= 6:

                verified_runs.append({
                    "segment_id": segment_id,
                    "variable": variable,
                    "start": run.index[0],
                    "end": run.index[-1],
                    "records": len(run),
                    "value": run.iloc[0],
                    "duration_hours": (
                        run.index[-1] - run.index[0]
                    ).total_seconds() / 3600
                })

verified_runs_df = pd.DataFrame(verified_runs)

display(
    verified_runs_df.sort_values(
        "records", ascending=False
    )
)

,segment_id,variable,start,end,records,value,duration_hours
48,118,rhum,2025-01-13 22:30:00+05:30,2025-01-14 07:30:00+05:30,10,92.0,9.0
62,145,rhum,2026-01-08 23:30:00+05:30,2026-01-09 07:30:00+05:30,9,100.0,8.0
7,76,rhum,2021-09-03 23:30:00+05:30,2021-09-04 06:30:00+05:30,8,99.0,7.0
16,76,rhum,2022-02-02 00:30:00+05:30,2022-02-02 07:30:00+05:30,8,100.0,7.0
3,76,rhum,2021-01-12 23:30:00+05:30,2021-01-13 06:30:00+05:30,8,100.0,7.0
...,...,...,...,...,...,...,...
61,145,rhum,2026-01-08 01:30:00+05:30,2026-01-08 06:30:00+05:30,6,100.0,5.0
65,145,wspd,2025-11-09 17:30:00+05:30,2025-11-09 22:30:00+05:30,6,7.6,5.0
66,145,wspd,2025-11-16 19:30:00+05:30,2025-11-17 00:30:00+05:30,6,6.5,5.0
67,145,wspd,2025-11-22 09:30:00+05:30,2025-11-22 14:30:00+05:30,6,2.0,5.0


In [98]:
rh_runs = verified_runs_df[
    (verified_runs_df["variable"] == "rhum") &
    (verified_runs_df["records"] >= 6)
].copy()

variables_to_inspect = [
    "temp", "dwpt", "rhum",
    "prcp", "wspd", "pres"
]

rh_context = []

for _, run in rh_runs.iterrows():

    segment_data = model_df[
        model_df["segment_id"] == run["segment_id"]
    ]

    selected = segment_data.loc[
        run["start"]:run["end"],
        variables_to_inspect
    ]

    rh_context.append({
        "segment_id": run["segment_id"],
        "start": run["start"],
        "end": run["end"],
        "records": run["records"],
        "RH": run["value"],
        "temp_mean": selected["temp"].mean(),
        "temp_min": selected["temp"].min(),
        "temp_max": selected["temp"].max(),
        "dwpt_mean": selected["dwpt"].mean(),
        "temp_dewpoint_gap": (
            selected["temp"] - selected["dwpt"]
        ).mean(),
        "prcp_total": selected["prcp"].sum(),
        "wind_mean": selected["wspd"].mean(),
        "pressure_mean": selected["pres"].mean()
    })

rh_context_df = pd.DataFrame(rh_context)

display(
    rh_context_df.sort_values(
        "records", ascending=False
    ).round(2)
)

,segment_id,start,end,records,RH,temp_mean,temp_min,temp_max,dwpt_mean,temp_dewpoint_gap,prcp_total,wind_mean,pressure_mean
39,118,2025-01-13 22:30:00+05:30,2025-01-14 07:30:00+05:30,10,92.0,8.29,6.3,10.5,7.09,1.20,0.0,9.58,1017.74
51,145,2026-01-08 23:30:00+05:30,2026-01-09 07:30:00+05:30,9,100.0,4.94,4.7,5.3,4.94,0.00,0.0,4.84,1022.60
16,76,2022-02-02 00:30:00+05:30,2022-02-02 07:30:00+05:30,8,100.0,6.89,6.2,8.1,6.89,0.00,0.0,7.21,1010.79
22,76,2023-01-09 00:30:00+05:30,2023-01-09 07:30:00+05:30,8,100.0,6.84,5.6,7.3,6.84,0.00,0.0,5.28,1019.76
3,76,2021-01-12 23:30:00+05:30,2021-01-13 06:30:00+05:30,8,100.0,5.89,5.0,6.6,5.89,0.00,0.0,7.79,1016.36
7,76,2021-09-03 23:30:00+05:30,2021-09-04 06:30:00+05:30,8,99.0,24.62,24.2,25.0,24.42,0.20,0.0,3.32,1003.30
48,145,2026-01-04 00:30:00+05:30,2026-01-04 07:30:00+05:30,8,100.0,8.36,7.5,9.5,8.36,0.00,0.0,4.60,1019.42
40,118,2025-01-16 00:30:00+05:30,2025-01-16 07:30:00+05:30,8,100.0,9.48,8.1,10.1,9.48,0.00,0.0,7.02,1019.80
53,145,2026-01-14 21:30:00+05:30,2026-01-15 04:30:00+05:30,8,100.0,6.19,5.5,7.3,6.19,0.00,0.0,5.81,1018.24
20,76,2022-12-21 01:30:00+05:30,2022-12-21 07:30:00+05:30,7,100.0,7.43,7.3,7.6,7.43,0.00,0.0,6.83,1014.54


In [99]:

import numpy as np
import pandas as pd

# ==========================================================
# E4 MASTER TEST
# Abrupt Rate-of-Change + Physical Consistency
# ==========================================================

variables = ["temp", "dwpt", "rhum", "wspd", "pres"]

# ----------------------------------------------------------
# 1. Prepare train/test datasets
# ----------------------------------------------------------

train_index = X_train_multi.index.intersection(model_df.index)
test_index = X_test_multi.index.intersection(model_df.index)

train_df = model_df.loc[train_index].copy()
test_df = model_df.loc[test_index].copy()

print("TRAIN RECORDS:", len(train_df))
print("TEST RECORDS:", len(test_df))

# ----------------------------------------------------------
# 2. Calculate strict hourly changes within segments
# ----------------------------------------------------------

change_frames = []

for segment_id, group in model_df.groupby("segment_id"):

    group = group.sort_index()

    # Only accept exact one-hour intervals
    hourly = (
        group.index.to_series()
        .diff()
        .eq(pd.Timedelta(hours=1))
    )

    changes = group[variables].diff().abs()

    changes = changes.where(hourly, np.nan)
    changes["segment_id"] = segment_id

    change_frames.append(changes)

change_df = pd.concat(change_frames).sort_index()

# ----------------------------------------------------------
# 3. Learn thresholds from TRAINING data only
# ----------------------------------------------------------

thresholds = {}

for variable in variables:

    training_changes = change_df.loc[
        change_df.index.isin(train_index), variable
    ].dropna()

    thresholds[variable] = training_changes.quantile(0.995)

print("\nTRAINING-BASED 99.5TH PERCENTILE THRESHOLDS")

threshold_table = pd.DataFrame({
    "variable": thresholds.keys(),
    "threshold": thresholds.values()
})

display(threshold_table.round(4))

# ----------------------------------------------------------
# 4. Apply thresholds to TEST observations
# ----------------------------------------------------------

test_changes = change_df.loc[
    change_df.index.isin(test_index), variables
]

rate_flags = pd.DataFrame(
    False,
    index=test_changes.index,
    columns=variables
)

for variable in variables:

    rate_flags[variable] = (
        test_changes[variable] > thresholds[variable]
    )

rate_summary = pd.DataFrame({
    "variable": variables,
    "threshold": [thresholds[v] for v in variables],
    "valid_test_changes": [
        test_changes[v].notna().sum() for v in variables
    ],
    "flagged_changes": [
        rate_flags[v].sum() for v in variables
    ],
    "flagged_percentage": [
        100 * rate_flags[v].sum() /
        max(test_changes[v].notna().sum(), 1)
        for v in variables
    ],
    "maximum_test_change": [
        test_changes[v].max() for v in variables
    ]
})

print("\nRATE-OF-CHANGE TEST RESULTS")
display(rate_summary.round(4))

# ----------------------------------------------------------
# 5. Extract timestamps flagged by rate-of-change screening
# ----------------------------------------------------------

rate_flags_df = rate_flags.any(axis=1)

rate_flag_details = test_changes.loc[
    rate_flags_df
].copy()

rate_flag_details["segment_id"] = (
    model_df.loc[rate_flag_details.index, "segment_id"]
)

for variable in variables:
    rate_flag_details[f"{variable}_flag"] = (
        rate_flags.loc[rate_flag_details.index, variable]
    )

print("\nFLAGGED RATE-OF-CHANGE OBSERVATIONS")
display(rate_flag_details.head(30).round(3))

# ----------------------------------------------------------
# 6. Cross-variable physical consistency
# ----------------------------------------------------------

physical = test_df[
    ["temp", "dwpt", "rhum"]
].copy()

# Dewpoint should generally not exceed air temperature
physical["dewpoint_above_temperature"] = (
    physical["dwpt"] > physical["temp"]
)

# RH must be between 0 and 100
physical["rh_out_of_range"] = (
    (physical["rhum"] < 0) |
    (physical["rhum"] > 100)
)

# Magnus approximation
es_temp = (
    6.112 *
    np.exp(17.67 * physical["temp"] /
           (physical["temp"] + 243.5))
)

es_dewpoint = (
    6.112 *
    np.exp(17.67 * physical["dwpt"] /
           (physical["dwpt"] + 243.5))
)

physical["expected_rh"] = (
    100 * es_dewpoint / es_temp
)

physical["rh_difference"] = (
    physical["rhum"] - physical["expected_rh"]
).abs()

physical["magnus_difference_over_5pp"] = (
    physical["rh_difference"] > 5
)

physical_summary = pd.DataFrame({
    "check": [
        "Dewpoint above temperature",
        "RH outside 0-100%",
        "Magnus RH difference > 5 percentage points"
    ],
    "flagged_records": [
        physical["dewpoint_above_temperature"].sum(),
        physical["rh_out_of_range"].sum(),
        physical["magnus_difference_over_5pp"].sum()
    ],
    "assessed_records": [
        physical["dewpoint_above_temperature"].notna().sum(),
        physical["rh_out_of_range"].notna().sum(),
        physical["magnus_difference_over_5pp"].notna().sum()
    ]
})

print("\nPHYSICAL CONSISTENCY RESULTS")
display(physical_summary)

# ----------------------------------------------------------
# 7. Extract physical consistency flags
# ----------------------------------------------------------

physical_flag_mask = (
    physical["dewpoint_above_temperature"] |
    physical["rh_out_of_range"] |
    physical["magnus_difference_over_5pp"]
)

physical_flags = physical.loc[physical_flag_mask].copy()

print("\nPHYSICALLY INCONSISTENT OBSERVATIONS")
display(physical_flags.head(30).round(3))

# ----------------------------------------------------------
# 8. Final E4 screening summary
# ----------------------------------------------------------

print("\n" + "=" * 55)
print("E4 MASTER TEST COMPLETED")
print("=" * 55)

print("Rate-of-change flagged timestamps:",
      int(rate_flags_df.sum()))

print("Physical consistency flagged timestamps:",
      len(physical_flags))

print("Thresholds learned from training data only.")
print("Test observations were not used to set thresholds.")
print("Flags are screening indicators, NOT confirmed sensor faults.")


TRAIN RECORDS: 36434
TEST RECORDS: 8415

TRAINING-BASED 99.5TH PERCENTILE THRESHOLDS


,variable,threshold
0,temp,4.8000
1,dwpt,4.6000
2,rhum,22.0000
3,wspd,10.1835
4,pres,2.0000



RATE-OF-CHANGE TEST RESULTS


,variable,threshold,valid_test_changes,flagged_changes,flagged_percentage,maximum_test_change
0,temp,4.8000,8415,53,0.6298,9.0
1,dwpt,4.6000,8415,78,0.9269,12.3
2,rhum,22.0000,8415,47,0.5585,40.0
3,wspd,10.1835,8415,46,0.5466,28.8
4,pres,2.0000,8415,79,0.9388,4.9



FLAGGED RATE-OF-CHANGE OBSERVATIONS


,temp,dwpt,rhum,wspd,pres,segment_id,temp_flag,dwpt_flag,rhum_flag,wspd_flag,pres_flag
time,,,,,,,,,,,
2025-03-22 09:30:00+05:30,5.3,3.4,5.0,0.7,0.6,145,True,False,False,False,False
2025-03-22 11:30:00+05:30,1.1,5.1,15.0,1.5,0.4,145,False,True,False,False,False
2025-03-22 12:30:00+05:30,1.9,7.1,13.0,0.5,0.6,145,False,True,False,False,False
2025-03-22 15:30:00+05:30,1.7,12.3,21.0,0.4,1.1,145,False,True,False,False,False
2025-03-23 09:30:00+05:30,5.4,3.5,5.0,4.3,0.9,145,True,False,False,False,False
2025-03-23 17:30:00+05:30,1.9,4.6,11.0,1.5,0.0,145,False,True,False,False,False
2025-03-24 14:30:00+05:30,2.0,6.1,10.0,0.0,1.0,145,False,True,False,False,False
2025-03-24 15:30:00+05:30,0.2,9.3,19.0,4.1,2.5,145,False,True,False,False,True
2025-03-25 09:30:00+05:30,5.1,2.9,7.0,4.5,1.5,145,True,False,False,False,False



PHYSICAL CONSISTENCY RESULTS


,check,flagged_records,assessed_records
0,Dewpoint above temperature,0,8415
1,RH outside 0-100%,0,8415
2,Magnus RH difference > 5 percentage points,0,8415



PHYSICALLY INCONSISTENT OBSERVATIONS


,temp,dwpt,rhum,dewpoint_above_temperature,rh_out_of_range,expected_rh,rh_difference,magnus_difference_over_5pp
time,,,,,,,,



E4 MASTER TEST COMPLETED
Rate-of-change flagged timestamps: 245
Physical consistency flagged timestamps: 0
Thresholds learned from training data only.
Test observations were not used to set thresholds.
Flags are screening indicators, NOT confirmed sensor faults.


In [100]:
# Align model consensus with E4 rate flags

consensus_votes = (
    corrected_stability_aligned["configuration_votes"]
    .reindex(rate_flags.index)
    .fillna(0)
)

candidate_mask = consensus_votes >= 8

rate_flag_mask = rate_flags.any(axis=1)

comparison = pd.DataFrame({
    "consensus_candidate": candidate_mask,
    "rate_change_flag": rate_flag_mask,
    "configuration_votes": consensus_votes
})

print("E1 + E4 OVERLAP")
print("=" * 45)

print("Consensus candidates:", int(candidate_mask.sum()))
print("Rate-change timestamps:", int(rate_flag_mask.sum()))

print(
    "Both consensus and rate flagged:",
    int((candidate_mask & rate_flag_mask).sum())
)

print(
    "Consensus only:",
    int((candidate_mask & ~rate_flag_mask).sum())
)

print(
    "Rate-change only:",
    int((~candidate_mask & rate_flag_mask).sum())
)

print("\nVariable flags among consensus candidates:")

display(
    rate_flags.loc[candidate_mask]
    .sum()
    .rename("flagged_candidates")
    .to_frame()
)

print("\nConsensus candidates with rate flags:")

display(
    comparison[
        comparison["consensus_candidate"] &
        comparison["rate_change_flag"]
    ].sort_values(
        "configuration_votes",
        ascending=False
    )
)

E1 + E4 OVERLAP
Consensus candidates: 123
Rate-change timestamps: 245
Both consensus and rate flagged: 40
Consensus only: 83
Rate-change only: 205

Variable flags among consensus candidates:


,flagged_candidates
temp,17
dwpt,11
rhum,12
wspd,13
pres,11



Consensus candidates with rate flags:


,consensus_candidate,rate_change_flag,configuration_votes
time,,,
2025-03-29 09:30:00+05:30,True,True,12
2025-03-30 09:30:00+05:30,True,True,12
2025-04-19 11:30:00+05:30,True,True,12
2026-01-27 09:30:00+05:30,True,True,12
2025-05-08 15:30:00+05:30,True,True,12
2025-05-19 16:30:00+05:30,True,True,12
2025-04-22 11:30:00+05:30,True,True,12
2025-04-20 11:30:00+05:30,True,True,12
2025-12-04 11:30:00+05:30,True,True,12


In [101]:

# =====================================================
# E1 + E3 + E4 EVIDENCE INTEGRATION
# =====================================================

# 1. Start with E1 consensus and E4 rate flags

integration = pd.DataFrame(index=X_test_multi.index)

integration["votes"] = (
    corrected_stability_aligned["configuration_votes"]
    .reindex(integration.index)
)

integration["E1_consensus"] = integration["votes"] >= 8

integration["E4_rate_flag"] = (
    rate_flags.any(axis=1)
    .reindex(integration.index)
    .fillna(False)
)

# 2. Add E3 spatial evidence
# Use exact timestamp comparisons from both stations.

spatial_flag_42099 = pd.Series(
    False, index=integration.index
)

spatial_flag_42101 = pd.Series(
    False, index=integration.index
)

# A spatial disagreement flag is defined here as
# a candidate timestamp with at least one variable
# exceeding the 95th percentile of non-candidate
# absolute spatial differences.
#
# Thresholds are calculated separately per station
# and variable using non-candidate observations.

spatial_thresholds = {}
spatial_flags = {}

for station in ["42099", "42101"]:

    data = spatial_control_data[station].copy()

    candidate = data["candidate"].fillna(False)

    station_flags = pd.DataFrame(
        False, index=data.index,
        columns=["temp", "dwpt", "rhum", "wspd", "pres"]
    )

    for variable in station_flags.columns:

        difference = (
            data[f"{station}_{variable}"]
            - data[f"VILD0_{variable}"]
        ).abs()

        control_values = difference[
            (~candidate) & difference.notna()
        ]

        if len(control_values) == 0:
            continue

        threshold = control_values.quantile(0.95)

        spatial_thresholds[(station, variable)] = threshold

        station_flags[variable] = (
            difference > threshold
        )

    spatial_flags[station] = station_flags

    if station == "42099":
        spatial_flag_42099 = (
            station_flags.any(axis=1)
            .reindex(integration.index)
            .fillna(False)
        )

    else:
        spatial_flag_42101 = (
            station_flags.any(axis=1)
            .reindex(integration.index)
            .fillna(False)
        )

integration["E3_42099"] = spatial_flag_42099
integration["E3_42101"] = spatial_flag_42101

integration["E3_spatial_flag"] = (
    integration["E3_42099"] |
    integration["E3_42101"]
)

# 3. Identify combined evidence groups

integration["evidence_count"] = (
    integration[
        ["E1_consensus", "E3_spatial_flag", "E4_rate_flag"]
    ].sum(axis=1)
)

integration["evidence_pattern"] = np.select(
    [
        integration["evidence_count"] == 3,
        integration["evidence_count"] == 2,
        integration["evidence_count"] == 1
    ],
    [
        "Three evidence groups",
        "Two evidence groups",
        "One evidence group"
    ],
    default="No evidence flags"
)

print("INTEGRATED EVIDENCE SUMMARY")
display(
    integration["evidence_pattern"]
    .value_counts()
    .rename_axis("evidence_pattern")
    .to_frame("timestamps")
)

print("\nALL THREE EVIDENCE GROUPS")
display(
    integration[
        (integration["E1_consensus"]) &
        (integration["E3_spatial_flag"]) &
        (integration["E4_rate_flag"])
    ].sort_values("votes", ascending=False)
)

print("\nINTEGRATION DATA")
display(
    integration[
        integration["evidence_count"] >= 2
    ].sort_values(
        ["evidence_count", "votes"],
        ascending=False
    ).head(50)
)


INTEGRATED EVIDENCE SUMMARY


,timestamps
evidence_pattern,
No evidence flags,7839
One evidence group,515
Two evidence groups,56
Three evidence groups,5



ALL THREE EVIDENCE GROUPS


,votes,E1_consensus,E4_rate_flag,E3_42099,E3_42101,E3_spatial_flag,evidence_count,evidence_pattern
time,,,,,,,,
2025-04-19 11:30:00+05:30,12,True,True,True,True,True,3,Three evidence groups
2025-04-22 11:30:00+05:30,12,True,True,True,True,True,3,Three evidence groups
2025-04-20 11:30:00+05:30,12,True,True,True,True,True,3,Three evidence groups
2025-04-18 11:30:00+05:30,11,True,True,False,True,True,3,Three evidence groups
2025-04-21 17:30:00+05:30,11,True,True,True,False,True,3,Three evidence groups



INTEGRATION DATA


,votes,E1_consensus,E4_rate_flag,E3_42099,E3_42101,E3_spatial_flag,evidence_count,evidence_pattern
time,,,,,,,,
2025-04-19 11:30:00+05:30,12,True,True,True,True,True,3,Three evidence groups
2025-04-20 11:30:00+05:30,12,True,True,True,True,True,3,Three evidence groups
2025-04-22 11:30:00+05:30,12,True,True,True,True,True,3,Three evidence groups
2025-04-18 11:30:00+05:30,11,True,True,False,True,True,3,Three evidence groups
2025-04-21 17:30:00+05:30,11,True,True,True,False,True,3,Three evidence groups
2025-03-29 09:30:00+05:30,12,True,True,False,False,False,2,Two evidence groups
2025-03-30 09:30:00+05:30,12,True,True,False,False,False,2,Two evidence groups
2025-04-26 11:30:00+05:30,12,True,False,True,False,True,2,Two evidence groups
2025-05-08 15:30:00+05:30,12,True,True,False,False,False,2,Two evidence groups


In [102]:

# =========================================================
# CORRECTED E1 + E3 + E4 INTEGRATION
# Preserve unavailable spatial evidence
# =========================================================

integration_corrected = pd.DataFrame(
    index=X_test_multi.index
)

# E1: Model consensus
integration_corrected["E1_consensus"] = (
    corrected_stability_aligned[
        "configuration_votes"
    ]
    .reindex(integration_corrected.index)
    .ge(8)
)

integration_corrected["votes"] = (
    corrected_stability_aligned[
        "configuration_votes"
    ].reindex(integration_corrected.index)
)

# E4: Rate-of-change evidence
integration_corrected["E4_rate_flag"] = (
    rate_flags.any(axis=1)
    .reindex(integration_corrected.index)
    .fillna(False)
)

# E3: Preserve unavailable spatial observations
spatial_status = {}

for station in ["42099", "42101"]:

    data = spatial_control_data[station].copy()

    candidate = data["candidate"].fillna(False)

    variables = ["temp", "dwpt", "rhum", "wspd", "pres"]

    station_flags = pd.DataFrame(
        index=data.index,
        columns=variables,
        dtype=bool
    )

    station_valid = pd.DataFrame(
        index=data.index,
        columns=variables,
        dtype=bool
    )

    for variable in variables:

        difference = (
            data[f"{station}_{variable}"]
            - data[f"VILD0_{variable}"]
        ).abs()

        controls = difference[
            (~candidate) & difference.notna()
        ]

        if controls.empty:
            continue

        threshold = controls.quantile(0.95)

        station_valid[variable] = difference.notna()

        station_flags[variable] = (
            difference > threshold
        ).fillna(False)

    # A timestamp is evaluable if at least one
    # variable has a valid spatial comparison.
    evaluable = station_valid.any(axis=1)

    # A timestamp is spatially flagged if any
    # evaluable variable exceeds its threshold.
    flagged = station_flags.any(axis=1)

    # Nullable Boolean preserves unavailable values.
    station_status = pd.Series(
        pd.NA,
        index=data.index,
        dtype="boolean"
    )

    station_status.loc[evaluable] = (
        flagged.loc[evaluable]
    )

    spatial_status[station] = station_status

# Align each station's status to the test index
for station in ["42099", "42101"]:

    integration_corrected[
        f"E3_{station}"
    ] = spatial_status[station].reindex(
        integration_corrected.index
    )

# E3 is evaluable if at least one station
# provides a valid spatial comparison.
integration_corrected["E3_evaluable"] = (
    integration_corrected[
        ["E3_42099", "E3_42101"]
    ].notna().any(axis=1)
)

# Positive E3 flag from either available station.
integration_corrected["E3_spatial_flag"] = (
    integration_corrected[
        ["E3_42099", "E3_42101"]
    ].fillna(False).any(axis=1)
)

# Keep the E3 flag as missing when neither station
# provides usable evidence.
integration_corrected.loc[
    ~integration_corrected["E3_evaluable"],
    "E3_spatial_flag"
] = pd.NA

# ---------------------------------------------------------
# 1. Spatial evidence coverage
# ---------------------------------------------------------

print("SPATIAL EVIDENCE COVERAGE")

display(pd.DataFrame({
    "status": [
        "Spatially evaluable",
        "Spatially unavailable",
        "42099 evaluable",
        "42101 evaluable"
    ],
    "timestamps": [
        integration_corrected["E3_evaluable"].sum(),
        (~integration_corrected["E3_evaluable"]).sum(),
        integration_corrected["E3_42099"].notna().sum(),
        integration_corrected["E3_42101"].notna().sum()
    ]
}))

# ---------------------------------------------------------
# 2. Recalculate three-way positive evidence
# ---------------------------------------------------------

all_three = (
    integration_corrected["E1_consensus"]
    & integration_corrected["E4_rate_flag"]
    & integration_corrected["E3_spatial_flag"].fillna(False)
)

print("\nALL THREE EVIDENCE GROUPS")
print("Three-way positive observations:", int(all_three.sum()))

display(
    integration_corrected.loc[all_three].sort_values(
        "votes",
        ascending=False
    )
)

# ---------------------------------------------------------
# 3. Separate E3-positive, E3-negative,
#    and E3-unavailable cases
# ---------------------------------------------------------

e1 = integration_corrected["E1_consensus"]
e4 = integration_corrected["E4_rate_flag"]
e3_available = integration_corrected["E3_evaluable"]
e3_positive = integration_corrected["E3_spatial_flag"].fillna(False)

categories = pd.Series(
    "No E1/E4 positive flags",
    index=integration_corrected.index,
    dtype="object"
)

categories.loc[e1 & ~e4] = "E1 only"
categories.loc[~e1 & e4] = "E4 only"
categories.loc[e1 & e4] = "E1 + E4"

categories.loc[e3_available & e3_positive] = (
    categories.loc[e3_available & e3_positive]
    + " + E3 positive"
)

categories.loc[~e3_available & (e1 | e4)] = (
    categories.loc[~e3_available & (e1 | e4)]
    + " + E3 unavailable"
)

integration_corrected["evidence_category"] = categories

print("\nCORRECTED EVIDENCE CATEGORIES")

display(
    integration_corrected[
        "evidence_category"
    ].value_counts().to_frame("timestamps")
)


SPATIAL EVIDENCE COVERAGE


,status,timestamps
0,Spatially evaluable,1193
1,Spatially unavailable,7222
2,42099 evaluable,567
3,42101 evaluable,1188



ALL THREE EVIDENCE GROUPS
Three-way positive observations: 5


,E1_consensus,votes,E4_rate_flag,E3_42099,E3_42101,E3_evaluable,E3_spatial_flag
time,,,,,,,
2025-04-19 11:30:00+05:30,True,12,True,True,True,True,True
2025-04-22 11:30:00+05:30,True,12,True,True,True,True,True
2025-04-20 11:30:00+05:30,True,12,True,True,True,True,True
2025-04-18 11:30:00+05:30,True,11,True,False,True,True,True
2025-04-21 17:30:00+05:30,True,11,True,True,False,True,True



CORRECTED EVIDENCE CATEGORIES


,timestamps
evidence_category,
No E1/E4 positive flags,7839
No E1/E4 positive flags + E3 positive,248
E4 only + E3 unavailable,173
E1 only + E3 unavailable,73
E1 + E4 + E3 unavailable,33
E4 only,18
E4 only + E3 positive,14
E1 only + E3 positive,7
E1 + E4 + E3 positive,5


In [104]:
print("REBUILT TRAINING COLUMNS:")
print(X_train_rebuilt.columns.tolist())

print("\nORIGINAL TRAINING COLUMNS:")
print(X_train_multi.columns.tolist())

print("\nColumns missing from rebuilt:")
print(set(X_train_multi.columns) - set(X_train_rebuilt.columns))

print("\nExtra columns in rebuilt:")
print(set(X_train_rebuilt.columns) - set(X_train_multi.columns))

print("\nSame columns, different order?")
print(
    set(X_train_rebuilt.columns) == set(X_train_multi.columns)
)

REBUILT TRAINING COLUMNS:
['temp', 'dwpt', 'rhum', 'prcp', 'wspd', 'pres', 'temp_change_1h', 'temp_rolling_mean_6h', 'temp_rolling_mean_6h', 'temp_deviation_6h', 'temp_deviation_6h', 'month_sin', 'month_cos']

ORIGINAL TRAINING COLUMNS:
['temp', 'dwpt', 'rhum', 'prcp', 'wspd', 'pres', 'temp_change_1h', 'temp_rolling_mean_6h', 'temp_deviation_6h']

Columns missing from rebuilt:
set()

Extra columns in rebuilt:
{'month_sin', 'month_cos'}

Same columns, different order?
False


In [105]:
print("Canonical engineered feature columns:")
print(clean_features.columns.tolist())

print("\nOriginal model_df columns:")
print(model_df.columns.tolist())

Canonical engineered feature columns:
['temp', 'dwpt', 'rhum', 'prcp', 'wspd', 'pres', 'segment_id', 'temp_change_1h', 'temp_rolling_mean_6h', 'temp_deviation_6h', 'temp_change_row', 'temp_rolling_mean_6h', 'temp_deviation_6h', 'month_sin', 'month_cos']

Original model_df columns:
['temp', 'dwpt', 'rhum', 'prcp', 'wspd', 'pres', 'segment_id', 'temp_change_1h', 'temp_rolling_mean_6h', 'temp_deviation_6h']


In [107]:

print("Generated columns:")
for col in clean_features_9.columns:
    print(repr(col))

print("\nRolling-related columns:")
print([
    col for col in clean_features_9.columns
    if "rolling" in col.lower() or "deviation" in col.lower()
])

print("\nOriginal model columns:")
print(model_df.columns.tolist())



Generated columns:
'temp'
'dwpt'
'rhum'
'prcp'
'wspd'
'pres'
'segment_id'
'temp_change_row'
'temp_rolling_mean_6obs'
'temp_deviation_6obs'
'temp_change_1h'

Rolling-related columns:
['temp_rolling_mean_6obs', 'temp_deviation_6obs']

Original model columns:
['temp', 'dwpt', 'rhum', 'prcp', 'wspd', 'pres', 'segment_id', 'temp_change_1h', 'temp_rolling_mean_6h', 'temp_deviation_6h']


In [108]:
import inspect
from weather_research.features import engineer_features

print(inspect.getsource(engineer_features))

def engineer_features(
    frame: pd.DataFrame,
    segment_col="segment_id",
    rolling_observations=6,
    expected_interval="1h",
) -> pd.DataFrame:
    """Add canonical features without mutating input.

    ``temp_change_row`` preserves the historic grouped previous-row difference.
    ``temp_change_1h`` is the strict elapsed-time version and is null unless the
    preceding row is exactly one hour earlier. The rolling mean is the historic
    previous-six-observation mean (shift(1)); missing segment IDs follow pandas
    groupby(dropna=True) semantics and therefore yield null engineered features.
    """
    missing = set(BASE_VARIABLES + (segment_col,)) - set(frame.columns)
    if missing:
        raise KeyError(f"missing feature inputs: {sorted(missing)}")
    if not isinstance(frame.index, pd.DatetimeIndex):
        raise TypeError("feature engineering requires a DatetimeIndex")
    if frame.index.has_duplicates or not frame.index.is_monotonic_increasing:
        raise ValueEr

In [109]:

# Rename only: preserve the canonical calculations
clean_features_9 = clean_features_9.rename(columns={
    "temp_rolling_mean_6obs": "temp_rolling_mean_6h",
    "temp_deviation_6obs": "temp_deviation_6h"
})

feature_cols_9 = [
    "temp", "dwpt", "rhum", "prcp", "wspd", "pres",
    "temp_change_1h",
    "temp_rolling_mean_6h",
    "temp_deviation_6h"
]

# Rebuild using the exact original matrix column order
X_train_rebuilt_9 = clean_features_9.loc[
    X_train_multi.index, feature_cols_9
].copy()

X_test_rebuilt_9 = clean_features_9.loc[
    X_test_multi.index, feature_cols_9
].copy()

print("Train shapes:", X_train_rebuilt_9.shape, X_train_multi.shape)
print("Test shapes:", X_test_rebuilt_9.shape, X_test_multi.shape)

train_comparison = compare_features(
    X_train_rebuilt_9,
    X_train_multi,
    "NINE-FEATURE TRAIN CHECK"
)

test_comparison = compare_features(
    X_test_rebuilt_9,
    X_test_multi,
    "NINE-FEATURE TEST CHECK"
)

print("\nFINAL BASELINE CHECK")
print("Train mismatches:",
      train_comparison["mismatches"].sum())
print("Test mismatches:",
      test_comparison["mismatches"].sum())
print("Original model_df unchanged:",
      model_df.equals(clean_raw))


Train shapes: (36434, 9) (36434, 9)
Test shapes: (8415, 9) (8415, 9)

NINE-FEATURE TRAIN CHECK


,feature,mismatches,max_absolute_difference
0,temp,0,0.0
1,dwpt,0,0.0
2,rhum,0,0.0
3,prcp,0,0.0
4,wspd,0,0.0
5,pres,0,0.0
6,temp_change_1h,0,0.0
7,temp_rolling_mean_6h,0,0.0
8,temp_deviation_6h,0,0.0



NINE-FEATURE TEST CHECK


,feature,mismatches,max_absolute_difference
0,temp,0,0.0
1,dwpt,0,0.0
2,rhum,0,0.0
3,prcp,0,0.0
4,wspd,0,0.0
5,pres,0,0.0
6,temp_change_1h,0,0.0
7,temp_rolling_mean_6h,0,0.0
8,temp_deviation_6h,0,0.0



FINAL BASELINE CHECK
Train mismatches: 0
Test mismatches: 0
Original model_df unchanged: True


In [110]:

# Add seasonal features to the already verified feature frame
clean_features_11 = add_month_cycle(clean_features_9.copy())

feature_cols_11 = feature_cols_9 + [
    "month_sin",
    "month_cos"
]

print("11-feature columns:")
print(clean_features_11[feature_cols_11].columns.tolist())

print("\nMissing values:")
display(clean_features_11[feature_cols_11].isna().sum())


11-feature columns:
['temp', 'dwpt', 'rhum', 'prcp', 'wspd', 'pres', 'temp_change_1h', 'temp_rolling_mean_6h', 'temp_deviation_6h', 'month_sin', 'month_cos']

Missing values:


,0
temp,0
dwpt,0
rhum,0
prcp,6
wspd,0
pres,0
temp_change_1h,5
temp_rolling_mean_6h,30
temp_deviation_6h,30
month_sin,0


In [111]:

expected_cols = set(feature_cols_11)

print("Possible original 11-feature matrices:\n")

for name, obj in list(globals().items()):
    if isinstance(obj, pd.DataFrame):
        if (
            obj.shape[1] == 11
            and set(obj.columns) == expected_cols
        ):
            print(
                f"{name}: shape={obj.shape}, "
                f"index_start={obj.index.min()}, "
                f"index_end={obj.index.max()}"
            )


Possible original 11-feature matrices:

X_train_seasonal: shape=(36434, 11), index_start=2021-01-06 01:30:00+05:30, index_end=2025-03-14 23:30:00+05:30
X_test_seasonal: shape=(8415, 11), index_start=2025-03-21 23:30:00+05:30, index_end=2026-03-09 23:30:00+05:30
X_train_scaled: shape=(36434, 11), index_start=2021-01-06 01:30:00+05:30, index_end=2025-03-14 23:30:00+05:30
X_test_scaled: shape=(8415, 11), index_start=2025-03-21 23:30:00+05:30, index_end=2026-03-09 23:30:00+05:30


In [112]:

# Reconstruct seasonal matrices at the original timestamps
X_train_rebuilt_11 = clean_features_11.loc[
    X_train_seasonal.index, feature_cols_11
].copy()

X_test_rebuilt_11 = clean_features_11.loc[
    X_test_seasonal.index, feature_cols_11
].copy()

# Compare reconstructed and original Phase 03 matrices
train_seasonal_comparison = compare_features(
    X_train_rebuilt_11,
    X_train_seasonal,
    "11-FEATURE SEASONAL TRAIN CHECK"
)

test_seasonal_comparison = compare_features(
    X_test_rebuilt_11,
    X_test_seasonal,
    "11-FEATURE SEASONAL TEST CHECK"
)

print("\nMISSING VALUE COMPARISON")
print("\nTraining:")
display(pd.DataFrame({
    "rebuilt": X_train_rebuilt_11.isna().sum(),
    "original": X_train_seasonal.isna().sum()
}))

print("\nTesting:")
display(pd.DataFrame({
    "rebuilt": X_test_rebuilt_11.isna().sum(),
    "original": X_test_seasonal.isna().sum()
}))

print("\nFINAL RESULT")
print("Training mismatches:",
      train_seasonal_comparison["mismatches"].sum())
print("Testing mismatches:",
      test_seasonal_comparison["mismatches"].sum())


AssertionError: 11-FEATURE SEASONAL TRAIN CHECK: timestamp mismatch

In [113]:
print("TRAIN INDEX DIAGNOSTICS")
print("Rebuilt timezone:", X_train_rebuilt_11.index.tz)
print("Original timezone:", X_train_seasonal.index.tz)

print("Rebuilt length:", len(X_train_rebuilt_11.index))
print("Original length:", len(X_train_seasonal.index))

print("Rebuilt first/last:",
      X_train_rebuilt_11.index.min(),
      X_train_rebuilt_11.index.max())

print("Original first/last:",
      X_train_seasonal.index.min(),
      X_train_seasonal.index.max())

print("Same timestamp set:",
      set(X_train_rebuilt_11.index) ==
      set(X_train_seasonal.index))

print("Rebuilt timestamps missing from original:",
      len(X_train_rebuilt_11.index.difference(
          X_train_seasonal.index
      )))

print("Original timestamps missing from rebuilt:",
      len(X_train_seasonal.index.difference(
          X_train_rebuilt_11.index
      )))

print("Same order:",
      X_train_rebuilt_11.index.equals(
          X_train_seasonal.index
      ))

TRAIN INDEX DIAGNOSTICS
Rebuilt timezone: Asia/Kolkata
Original timezone: UTC+05:30
Rebuilt length: 36434
Original length: 36434
Rebuilt first/last: 2021-01-06 01:30:00+05:30 2025-03-14 23:30:00+05:30
Original first/last: 2021-01-06 01:30:00+05:30 2025-03-14 23:30:00+05:30
Same timestamp set: True
Rebuilt timestamps missing from original: 0
Original timestamps missing from rebuilt: 0
Same order: False


In [114]:
print("\nTEST INDEX DIAGNOSTICS")
print("Rebuilt timezone:", X_test_rebuilt_11.index.tz)
print("Original timezone:", X_test_seasonal.index.tz)

print("Same timestamp set:",
      set(X_test_rebuilt_11.index) ==
      set(X_test_seasonal.index))

print("Rebuilt missing:",
      len(X_test_rebuilt_11.index.difference(
          X_test_seasonal.index
      )))

print("Original missing:",
      len(X_test_seasonal.index.difference(
          X_test_rebuilt_11.index
      )))

print("Same order:",
      X_test_rebuilt_11.index.equals(
          X_test_seasonal.index
      ))


TEST INDEX DIAGNOSTICS
Rebuilt timezone: Asia/Kolkata
Original timezone: UTC+05:30
Same timestamp set: True
Rebuilt missing: 0
Original missing: 0
Same order: False


In [115]:
# Align original Phase 03 matrices to reconstructed indexes
X_train_seasonal_aligned = X_train_seasonal.reindex(
    X_train_rebuilt_11.index
)

X_test_seasonal_aligned = X_test_seasonal.reindex(
    X_test_rebuilt_11.index
)

# Verify exact index alignment
assert X_train_rebuilt_11.index.equals(
    X_train_seasonal_aligned.index
)

assert X_test_rebuilt_11.index.equals(
    X_test_seasonal_aligned.index
)

print("Both seasonal matrices successfully aligned.")

# Compare feature values
train_seasonal_comparison = compare_features(
    X_train_rebuilt_11,
    X_train_seasonal_aligned,
    "11-FEATURE TRAIN CHECK"
)

test_seasonal_comparison = compare_features(
    X_test_rebuilt_11,
    X_test_seasonal_aligned,
    "11-FEATURE TEST CHECK"
)

print("\nFinal mismatches:")
print("Training:",
      train_seasonal_comparison["mismatches"].sum())
print("Testing:",
      test_seasonal_comparison["mismatches"].sum())

Both seasonal matrices successfully aligned.

11-FEATURE TRAIN CHECK


,feature,mismatches,max_absolute_difference
0,temp,0,0.000000e+00
1,dwpt,0,0.000000e+00
2,rhum,0,0.000000e+00
3,prcp,0,0.000000e+00
4,wspd,0,0.000000e+00
5,pres,0,0.000000e+00
6,temp_change_1h,0,1.776357e-15
7,temp_rolling_mean_6h,0,7.105427e-15
8,temp_deviation_6h,0,1.776357e-15
9,month_sin,0,0.000000e+00



11-FEATURE TEST CHECK


,feature,mismatches,max_absolute_difference
0,temp,0,0.000000e+00
1,dwpt,0,0.000000e+00
2,rhum,0,0.000000e+00
3,prcp,0,0.000000e+00
4,wspd,0,0.000000e+00
5,pres,0,0.000000e+00
6,temp_change_1h,0,4.440892e-16
7,temp_rolling_mean_6h,0,7.105427e-15
8,temp_deviation_6h,0,1.776357e-15
9,month_sin,0,0.000000e+00



Final mismatches:
Training: 0
Testing: 0


In [116]:

# Check the four original model matrices
matrices = {
    "X_train_seasonal": X_train_seasonal,
    "X_test_seasonal": X_test_seasonal,
    "X_train_scaled": X_train_scaled,
    "X_test_scaled": X_test_scaled
}

summary = []

for name, frame in matrices.items():
    values = frame.to_numpy(dtype=float)

    summary.append({
        "matrix": name,
        "rows": len(frame),
        "features": frame.shape[1],
        "missing_values": int(frame.isna().sum().sum()),
        "infinite_values": int(np.isinf(values).sum())
    })

display(pd.DataFrame(summary))

# Inspect the fitted preprocessing metadata
print("Seasonal preprocessing keys:", seasonal_prep.keys())
print("Scaled preprocessing keys:", scaled_prep.keys())

print("\nTraining imputation medians:")
print(seasonal_prep["medians"])

# Verify the existing scaler reproduces scaled matrices
scaler = scaled_prep["scaler"]

if scaler is None:
    print("\nNo fitted scaler found.")
else:
    train_scaled_check = scaler.transform(X_train_seasonal)
    test_scaled_check = scaler.transform(X_test_seasonal)

    print("\nScaler reproduction:")
    print(
        "Training maximum difference:",
        np.max(np.abs(
            train_scaled_check - X_train_scaled.to_numpy()
        ))
    )
    print(
        "Testing maximum difference:",
        np.max(np.abs(
            test_scaled_check - X_test_scaled.to_numpy()
        ))
    )


,matrix,rows,features,missing_values,infinite_values
0,X_train_seasonal,36434,11,0,0
1,X_test_seasonal,8415,11,0,0
2,X_train_scaled,36434,11,0,0
3,X_test_scaled,8415,11,0,0


NameError: name 'seasonal_prep' is not defined

In [117]:
# Find preprocessing dictionaries currently available
print("Available preprocessing objects:")

for name, obj in list(globals().items()):
    if isinstance(obj, dict):
        if "medians" in obj or "scaler" in obj:
            print(f"\n{name}")
            print("Keys:", list(obj.keys()))
            print("Medians:", obj.get("medians"))

            scaler_obj = obj.get("scaler")
            print("Scaler:", type(scaler_obj).__name__)

            if scaler_obj is not None:
                print("Features fitted:", scaler_obj.n_features_in_)

Available preprocessing objects:


In [118]:
# Replace this with the dictionary name found in Step 1
prep_object = scaled_prep

scaler = prep_object["scaler"]

train_scaled_check = scaler.transform(X_train_seasonal)
test_scaled_check = scaler.transform(X_test_seasonal)

print("Training scaling difference:",
      np.max(np.abs(
          train_scaled_check - X_train_scaled.to_numpy()
      )))

print("Testing scaling difference:",
      np.max(np.abs(
          test_scaled_check - X_test_scaled.to_numpy()
      )))

NameError: name 'scaled_prep' is not defined

In [119]:
# Find preprocessing dictionaries currently available
print("Available preprocessing objects:")

for name, obj in list(globals().items()):
    if isinstance(obj, dict):
        if "medians" in obj or "scaler" in obj:
            print(f"\n{name}")
            print("Keys:", list(obj.keys()))
            print("Medians:", obj.get("medians"))

            scaler_obj = obj.get("scaler")
            print("Scaler:", type(scaler_obj).__name__)

            if scaler_obj is not None:
                print("Features fitted:", scaler_obj.n_features_in_)

Available preprocessing objects:


In [120]:
# Replace this with the dictionary name found in Step 1
prep_object = scaled_prep

scaler = prep_object["scaler"]

train_scaled_check = scaler.transform(X_train_seasonal)
test_scaled_check = scaler.transform(X_test_seasonal)

print("Training scaling difference:",
      np.max(np.abs(
          train_scaled_check - X_train_scaled.to_numpy()
      )))

print("Testing scaling difference:",
      np.max(np.abs(
          test_scaled_check - X_test_scaled.to_numpy()
      )))

NameError: name 'scaled_prep' is not defined

In [121]:
from sklearn.preprocessing import StandardScaler
import numpy as np

# Fit a new scaler using training data only
scaler_check = StandardScaler()
train_rebuilt = scaler_check.fit_transform(X_train_seasonal)

# Apply the same scaler to test data
test_rebuilt = scaler_check.transform(X_test_seasonal)

# Compare with your existing scaled matrices
train_diff = np.max(
    np.abs(train_rebuilt - X_train_scaled.to_numpy())
)

test_diff = np.max(
    np.abs(test_rebuilt - X_test_scaled.to_numpy())
)

print("Training scaling difference:", train_diff)
print("Testing scaling difference:", test_diff)

print("\nScaler feature count:", scaler_check.n_features_in_)
print("Feature order matches:",
      list(X_train_seasonal.columns) == list(X_train_scaled.columns))

Training scaling difference: 0.0
Testing scaling difference: 0.0

Scaler feature count: 11
Feature order matches: True


In [122]:
print("Model-related objects:")

for name in [
    "X_train_seasonal",
    "X_test_seasonal",
    "X_train_scaled",
    "X_test_scaled",
    "model_configs",
    "model_results",
    "predictions_df",
]:
    if name in globals():
        obj = globals()[name]
        print(
            name,
            "| type:", type(obj).__name__,
            "| shape:", getattr(obj, "shape", "N/A")
        )
    else:
        print(name, "| NOT FOUND")

Model-related objects:
X_train_seasonal | type: DataFrame | shape: (36434, 11)
X_test_seasonal | type: DataFrame | shape: (8415, 11)
X_train_scaled | type: DataFrame | shape: (36434, 11)
X_test_scaled | type: DataFrame | shape: (8415, 11)
model_configs | NOT FOUND
model_results | NOT FOUND
predictions_df | NOT FOUND


In [123]:
import pandas as pd
import numpy as np

checks = {
    "Train indexes aligned":
        X_train_seasonal.index.equals(X_train_scaled.index),

    "Test indexes aligned":
        X_test_seasonal.index.equals(X_test_scaled.index),

    "Train columns aligned":
        X_train_seasonal.columns.equals(X_train_scaled.columns),

    "Test columns aligned":
        X_test_seasonal.columns.equals(X_test_scaled.columns),

    "Train and test timestamps disjoint":
        X_train_seasonal.index.intersection(
            X_test_seasonal.index
        ).empty,

    "Training precedes testing":
        X_train_seasonal.index.max() < X_test_seasonal.index.min(),

    "All model inputs finite":
        all(
            np.isfinite(df.to_numpy(dtype=float)).all()
            for df in [
                X_train_seasonal,
                X_test_seasonal,
                X_train_scaled,
                X_test_scaled
            ]
        )
}

for name, result in checks.items():
    print(f"{name}: {result}")

print("\nPossible prediction/result DataFrames:")

for name, obj in list(globals().items()):
    if isinstance(obj, pd.DataFrame):
        cols = set(obj.columns)
        if {"configuration", "prediction"}.issubset(cols):
            print(name, obj.shape, list(obj.columns))

Train indexes aligned: True
Test indexes aligned: True
Train columns aligned: True
Test columns aligned: True
Train and test timestamps disjoint: True
Training precedes testing: True
All model inputs finite: True

Possible prediction/result DataFrames:
predictions (100980, 6) ['configuration', 'family', 'parameter', 'parameter_value', 'prediction', 'is_candidate']
prediction_data (100980, 7) ['configuration', 'family', 'parameter', 'parameter_value', 'prediction', 'is_candidate', 'timestamp']


In [124]:
# Use prediction_data because it includes timestamps
pred = prediction_data.copy()

print("Total prediction rows:", len(pred))
print("Unique configurations:", pred["configuration"].nunique())
print("Unique timestamps:", pred["timestamp"].nunique())

print("\nRecords per configuration:")
display(
    pred.groupby("configuration")
    .agg(
        rows=("prediction", "size"),
        unique_timestamps=("timestamp", "nunique"),
        candidate_count=("is_candidate", "sum"),
        missing_predictions=("prediction", lambda s: s.isna().sum())
    )
)

print("\nDuplicate configuration-timestamp pairs:",
      pred.duplicated(["configuration", "timestamp"]).sum())

print("\nMissing timestamps compared with X_test_seasonal:")
expected_index = X_test_seasonal.index
actual_index = pd.DatetimeIndex(pred["timestamp"].unique())

print("Missing:", len(expected_index.difference(actual_index)))
print("Unexpected:", len(actual_index.difference(expected_index)))

print("\nModel families:")
display(pred.groupby("family")["configuration"].nunique())

Total prediction rows: 100980
Unique configurations: 12
Unique timestamps: 8415

Records per configuration:


,rows,unique_timestamps,candidate_count,missing_predictions
configuration,,,,
IF_0.01,8415,8415,94,0
IF_0.02,8415,8415,189,0
IF_0.03,8415,8415,270,0
IF_0.05,8415,8415,521,0
LOF_10,8415,8415,531,0
LOF_20,8415,8415,386,0
LOF_30,8415,8415,326,0
LOF_50,8415,8415,263,0
SVM_0.01,8415,8415,157,0



Duplicate configuration-timestamp pairs: 0

Missing timestamps compared with X_test_seasonal:
Missing: 0
Unexpected: 0

Model families:


,configuration
family,
IsolationForest,4
LocalOutlierFactor,4
OneClassSVM,4


In [125]:
import pandas as pd

objects_to_check = [
    "model_df",
    "df_major",
    "df_multi",
    "X_multi",
    "X_test_multi",
    "train_frame",
    "test_frame"
]

for name in objects_to_check:
    if name in globals():
        obj = globals()[name]
        print(f"\n{name}: {type(obj).__name__}")

        if isinstance(obj, pd.DataFrame):
            print("Shape:", obj.shape)
            print("Date range:", obj.index.min(), "to", obj.index.max())
            print("Required columns present:",
                  [c for c in [
                      "temp", "dwpt", "rhum",
                      "prcp", "wspd", "pres", "segment_id"
                  ] if c in obj.columns])
            print("Has duplicate timestamps:", obj.index.has_duplicates)
    else:
        print(f"\n{name}: NOT FOUND")

print("\nFeature engineering function available:",
      "engineer_features" in globals())

print("Model fitting function available:",
      "fit_predict_configurations" in globals())


model_df: DataFrame
Shape: (44879, 10)
Date range: 2021-01-05 19:30:00+05:30 to 2026-03-09 23:30:00+05:30
Required columns present: ['temp', 'dwpt', 'rhum', 'prcp', 'wspd', 'pres', 'segment_id']
Has duplicate timestamps: False

df_major: DataFrame
Shape: (44879, 12)
Date range: 2021-01-05 19:30:00+05:30 to 2026-03-09 23:30:00+05:30
Required columns present: ['temp', 'dwpt', 'rhum', 'prcp', 'wspd', 'pres', 'segment_id']
Has duplicate timestamps: False

df_multi: NOT FOUND

X_multi: DataFrame
Shape: (44849, 9)
Date range: 2021-01-06 01:30:00+05:30 to 2026-03-09 23:30:00+05:30
Required columns present: ['temp', 'dwpt', 'rhum', 'prcp', 'wspd', 'pres']
Has duplicate timestamps: False

X_test_multi: DataFrame
Shape: (8415, 9)
Date range: 2025-03-21 23:30:00+05:30 to 2026-03-09 23:30:00+05:30
Required columns present: ['temp', 'dwpt', 'rhum', 'prcp', 'wspd', 'pres']
Has duplicate timestamps: False

train_frame: NOT FOUND

test_frame: NOT FOUND

Feature engineering function available: True
Mo

In [126]:
import inspect
import numpy as np

print("Feature engineering function:")
print("Module:", engineer_features.__module__)
print("Signature:", inspect.signature(engineer_features))

print("\nOriginal model_df columns:")
print(model_df.columns.tolist())

print("\nModel feature columns:")
print(X_test_seasonal.columns.tolist())

# Rebuild from raw observations only
raw_columns = [
    "temp", "dwpt", "rhum", "prcp", "wspd", "pres", "segment_id"
]

raw_check = model_df[raw_columns].copy()

engineered_check = engineer_features(raw_check)

print("\nReconstructed feature columns:")
print(engineered_check.columns.tolist())

print("\nReconstructed feature missing values:")
print(engineered_check.isna().sum())

Feature engineering function:
Module: weather_research.features
Signature: (frame: 'pd.DataFrame', segment_col='segment_id', rolling_observations=6, expected_interval='1h') -> 'pd.DataFrame'

Original model_df columns:
['temp', 'dwpt', 'rhum', 'prcp', 'wspd', 'pres', 'segment_id', 'temp_change_1h', 'temp_rolling_mean_6h', 'temp_deviation_6h']

Model feature columns:
['temp', 'dwpt', 'rhum', 'prcp', 'wspd', 'pres', 'temp_change_1h', 'temp_rolling_mean_6h', 'temp_deviation_6h', 'month_sin', 'month_cos']

Reconstructed feature columns:
['temp', 'dwpt', 'rhum', 'prcp', 'wspd', 'pres', 'segment_id', 'temp_change_row', 'temp_rolling_mean_6obs', 'temp_deviation_6obs', 'temp_change_1h']

Reconstructed feature missing values:
temp                       0
dwpt                       0
rhum                       0
prcp                       6
wspd                       0
pres                       0
segment_id                 0
temp_change_row            5
temp_rolling_mean_6obs    30
temp_deviati

In [127]:
common_features = [
    col for col in X_test_seasonal.columns
    if col in engineered_check.columns
]

comparison = []

for col in common_features:
    rebuilt = engineered_check.loc[
        X_test_seasonal.index, col
    ].astype(float)

    existing = X_test_seasonal[col].astype(float)

    comparison.append({
        "feature": col,
        "missing_pattern_matches":
            rebuilt.isna().equals(existing.isna()),
        "max_absolute_difference":
            np.nanmax(np.abs(rebuilt.to_numpy() - existing.to_numpy()))
            if not rebuilt.isna().all() else np.nan
    })

display(pd.DataFrame(comparison))

,feature,missing_pattern_matches,max_absolute_difference
0,temp,False,0.000000e+00
1,dwpt,False,0.000000e+00
2,rhum,False,0.000000e+00
3,prcp,False,0.000000e+00
4,wspd,False,0.000000e+00
5,pres,False,0.000000e+00
6,temp_change_1h,False,4.440892e-16


In [129]:
feature_pairs = {
    "temp_change_1h": "temp_change_1h",
    "temp_rolling_mean_6h": "temp_rolling_mean_6obs",
    "temp_deviation_6h": "temp_deviation_6obs"
}

comparison_original = []

for original_col, rebuilt_col in feature_pairs.items():

    original = model_df.loc[engineered_check.index, original_col]
    rebuilt = engineered_check[rebuilt_col]

    original_values = original.to_numpy(dtype=float)
    rebuilt_values = rebuilt.to_numpy(dtype=float)

    close_mask = np.isclose(
        original_values,
        rebuilt_values,
        rtol=1e-10,
        atol=1e-10,
        equal_nan=True
    )

    comparison_original.append({
        "original_feature": original_col,
        "rebuilt_feature": rebuilt_col,
        "missing_pattern_matches": original.isna().equals(rebuilt.isna()),
        "max_absolute_difference": (
            np.nanmax(np.abs(original_values - rebuilt_values))
        ),
        "mismatch_count": int((~close_mask).sum())
    })

display(pd.DataFrame(comparison_original))

,original_feature,rebuilt_feature,missing_pattern_matches,max_absolute_difference,mismatch_count
0,temp_change_1h,temp_change_1h,True,0.0,0
1,temp_rolling_mean_6h,temp_rolling_mean_6obs,True,0.0,0
2,temp_deviation_6h,temp_deviation_6obs,True,0.0,0


In [130]:
print("X_test_multi columns:")
print(X_test_multi.columns.tolist())

print("\nMissing values in X_test_multi:")
print(X_test_multi.isna().sum())

print("\nMissing values in X_test_seasonal:")
print(X_test_seasonal.isna().sum())

X_test_multi columns:
['temp', 'dwpt', 'rhum', 'prcp', 'wspd', 'pres', 'temp_change_1h', 'temp_rolling_mean_6h', 'temp_deviation_6h']

Missing values in X_test_multi:
temp                    0
dwpt                    0
rhum                    0
prcp                    0
wspd                    0
pres                    0
temp_change_1h          0
temp_rolling_mean_6h    0
temp_deviation_6h       0
dtype: int64

Missing values in X_test_seasonal:
temp                    0
dwpt                    0
rhum                    0
prcp                    0
wspd                    0
pres                    0
temp_change_1h          0
temp_rolling_mean_6h    0
temp_deviation_6h       0
month_sin               0
month_cos               0
dtype: int64


In [131]:
import inspect

from weather_research.faults import (
    inject_fault,
    inject_fault_with_feature_recompute,
    evaluate_injection
)

print("=== FAULT INJECTION FUNCTION ===")
print(inspect.signature(inject_fault_with_feature_recompute))
print(inspect.getsource(inject_fault_with_feature_recompute))

print("\n=== INJECTION EVALUATION FUNCTION ===")
print(inspect.signature(evaluate_injection))

print("\n=== MODEL PREDICTION FUNCTION ===")
print(inspect.signature(fit_predict_configurations))
print(inspect.getsource(fit_predict_configurations)[:7000])

=== FAULT INJECTION FUNCTION ===
(frame: 'pd.DataFrame', variable: 'str', start, end=None, kind='spike', magnitude=1.0, random_seed=42, calibration_end=None, segment_col='segment_id', **kwargs)
def inject_fault_with_feature_recompute(frame: pd.DataFrame, variable: str, start, end=None,
                                        kind="spike", magnitude=1.0, random_seed=42,
                                        calibration_end=None, segment_col="segment_id", **kwargs):
    """Inject into a copy and recompute clean/faulted features with the canonical function.

    The shifted rolling feature at the fault timestamp uses only preceding observations,
    so the injected value cannot enter its own reference. Earlier injected values can
    affect later causal windows, as would happen in a real sensor stream.
    """
    clean_features=engineer_features(frame,segment_col=segment_col)
    injected,labels,registry=inject_fault(frame,variable,start,end,kind,magnitude,
        random_seed,calibrat

In [132]:
print("Training end:", X_train_seasonal.index.max())
print("Testing start:", X_test_seasonal.index.min())
print("Testing end:", X_test_seasonal.index.max())

Training end: 2025-03-14 23:30:00+05:30
Testing start: 2025-03-21 23:30:00+05:30
Testing end: 2026-03-09 23:30:00+05:30


In [133]:
import numpy as np
import pandas as pd

from weather_research.models import default_model_configs

configs = default_model_configs()

print("Configurations:", len(configs))
print(pd.DataFrame(configs))

# Refit on the original, unchanged training data
baseline_recheck = fit_predict_configurations(
    X_train_seasonal,
    X_test_seasonal,
    X_train_scaled,
    X_test_scaled,
    configs=configs
)

# Prepare both prediction tables
new_predictions = (
    baseline_recheck
    .reset_index()
    [["timestamp", "configuration", "prediction", "is_candidate"]]
    .sort_values(["timestamp", "configuration"])
    .reset_index(drop=True)
)

old_predictions = (
    prediction_data
    [["timestamp", "configuration", "prediction", "is_candidate"]]
    .sort_values(["timestamp", "configuration"])
    .reset_index(drop=True)
)

print("\nSame number of records:",
      len(new_predictions) == len(old_predictions))

print("Same timestamps and configurations:",
      new_predictions[["timestamp", "configuration"]].equals(
          old_predictions[["timestamp", "configuration"]]
      ))

print("Prediction mismatches:",
      int(
          (new_predictions["prediction"].to_numpy()
           != old_predictions["prediction"].to_numpy()).sum()
      ))

print("Candidate mismatches:",
      int(
          (new_predictions["is_candidate"].to_numpy()
           != old_predictions["is_candidate"].to_numpy()).sum()
      ))

Configurations: 12
                family      name      parameter  value  random_seed
0      IsolationForest   IF_0.01  contamination   0.01           42
1      IsolationForest   IF_0.02  contamination   0.02           42
2      IsolationForest   IF_0.03  contamination   0.03           42
3      IsolationForest   IF_0.05  contamination   0.05           42
4   LocalOutlierFactor    LOF_10    n_neighbors  10.00           42
5   LocalOutlierFactor    LOF_20    n_neighbors  20.00           42
6   LocalOutlierFactor    LOF_30    n_neighbors  30.00           42
7   LocalOutlierFactor    LOF_50    n_neighbors  50.00           42
8          OneClassSVM  SVM_0.01             nu   0.01           42
9          OneClassSVM  SVM_0.02             nu   0.02           42
10         OneClassSVM  SVM_0.03             nu   0.03           42
11         OneClassSVM  SVM_0.05             nu   0.05           42


ValueError: 'timestamp' is both an index level and a column label, which is ambiguous.

In [134]:
print("Baseline recheck available:", "baseline_recheck" in globals())

if "baseline_recheck" in globals():
    print("New prediction rows:", len(baseline_recheck))
    print("Original prediction rows:", len(prediction_data))
else:
    print("The model-fitting step did not complete.")

Baseline recheck available: True
New prediction rows: 100980
Original prediction rows: 100980


In [163]:
# Create clean copies without modifying original prediction data
new_predictions = baseline_recheck.reset_index()

old_predictions = prediction_data.copy()

# Remove timestamp from the index while retaining its column
new_predictions = new_predictions.reset_index(drop=True)
old_predictions = old_predictions.reset_index(drop=True)

# Select and sort comparison columns
comparison_columns = [
    "timestamp",
    "configuration",
    "prediction",
    "is_candidate"
]

new_predictions = (
    new_predictions[comparison_columns]
    .sort_values(["timestamp", "configuration"])
    .reset_index(drop=True)
)

old_predictions = (
    old_predictions[comparison_columns]
    .sort_values(["timestamp", "configuration"])
    .reset_index(drop=True)
)

# Compare
print(
    "Same timestamps and configurations:",
    new_predictions[["timestamp", "configuration"]].equals(
        old_predictions[["timestamp", "configuration"]]
    )
)

print(
    "Prediction mismatches:",
    (new_predictions["prediction"] != old_predictions["prediction"]).sum()
)

print(
    "Candidate mismatches:",
    (new_predictions["is_candidate"] != old_predictions["is_candidate"]).sum()
)

Same timestamps and configurations: True
Prediction mismatches: 0
Candidate mismatches: 0


In [137]:
# Find a suitable isolated test observation
# Avoid segment boundaries and timestamps already flagged by E1.

test_segments = model_df.loc[
    X_test_seasonal.index, "segment_id"
]

candidate_times = X_test_seasonal.index[
    (test_segments == 145).to_numpy()
]

print("Available test observations in segment 145:",
      len(candidate_times))

print("First 10 eligible timestamps:")
print(candidate_times[:10])

Available test observations in segment 145: 7843
First 10 eligible timestamps:
DatetimeIndex(['2025-03-21 23:30:00+05:30', '2025-03-22 00:30:00+05:30',
               '2025-03-22 01:30:00+05:30', '2025-03-22 02:30:00+05:30',
               '2025-03-22 03:30:00+05:30', '2025-03-22 04:30:00+05:30',
               '2025-03-22 05:30:00+05:30', '2025-03-22 06:30:00+05:30',
               '2025-03-22 07:30:00+05:30', '2025-03-22 08:30:00+05:30'],
              dtype='datetime64[ns, UTC+05:30]', name='time', freq=None)


In [139]:
# Count existing model votes at each timestamp
vote_counts = (
    prediction_data
    .groupby("timestamp")["is_candidate"]
    .sum()
)

eligible = []

for i in range(6, len(candidate_times) - 6):
    timestamp = candidate_times[i]

    window = candidate_times[i-6:i+7]

    # Require 13 consecutive hourly observations
    if not (window.to_series().diff().dropna() == pd.Timedelta(hours=1)).all():
        continue

    # Prefer timestamps with low existing model consensus
    votes = int(vote_counts.get(timestamp, 0))

    eligible.append({
        "timestamp": timestamp,
        "existing_votes": votes,
        "temperature": model_df.loc[timestamp, "temp"]
    })

eligible_df = pd.DataFrame(eligible)

print("Eligible timestamps:", len(eligible_df))

# Select the lowest-consensus timestamp nearest the middle of the segment
eligible_df["middle_distance"] = abs(
    (eligible_df["timestamp"] - candidate_times[len(candidate_times)//2])
    .dt.total_seconds()
)

selected = (
    eligible_df
    .sort_values(["existing_votes", "middle_distance"])
    .iloc[0]
)

print("\nSelected injection timestamp:")
display(selected.to_frame("value"))

ValueError: 'timestamp' is both an index level and a column label, which is ambiguous.

In [140]:
# Use the timestamp column directly
pred_clean = prediction_data.reset_index(drop=True)

vote_counts = (
    pred_clean
    .pivot_table(
        index="timestamp",
        values="is_candidate",
        aggfunc="sum"
    )["is_candidate"]
)

eligible = []

for i in range(6, len(candidate_times) - 6):
    timestamp = candidate_times[i]
    window = candidate_times[i-6:i+7]

    # Require 13 consecutive hourly observations
    if not (
        window.to_series().diff().dropna()
        == pd.Timedelta(hours=1)
    ).all():
        continue

    votes = int(vote_counts.get(timestamp, 0))

    eligible.append({
        "timestamp": timestamp,
        "existing_votes": votes,
        "temperature": model_df.loc[timestamp, "temp"]
    })

eligible_df = pd.DataFrame(eligible)

print("Eligible timestamps:", len(eligible_df))

if not eligible_df.empty:
    middle_timestamp = candidate_times[len(candidate_times)//2]

    eligible_df["middle_distance"] = (
        eligible_df["timestamp"] - middle_timestamp
    ).abs().dt.total_seconds()

    selected = eligible_df.sort_values(
        ["existing_votes", "middle_distance"]
    ).iloc[0]

    print("\nSelected injection timestamp:")
    display(selected.to_frame("value"))
else:
    print("No eligible timestamp found.")

Eligible timestamps: 7831

Selected injection timestamp:


,value
timestamp,2025-09-01 07:30:00+05:30
existing_votes,0
temperature,23.9
middle_distance,3600.0


In [142]:
raw_columns = [
    "temp", "dwpt", "rhum",
    "prcp", "wspd", "pres", "segment_id"
]

raw_fault_input = model_df[raw_columns].copy()

injection_time = pd.Timestamp(
    "2025-09-01 07:30:00",
    tz="Asia/Kolkata"
)

train_cutoff = X_train_seasonal.index.max()

(
    injected_frame,
    synthetic_labels,
    fault_registry,
    clean_rebuilt_features,
    faulted_features
) = inject_fault_with_feature_recompute(
    frame=raw_fault_input,
    variable="temp",
    start=injection_time,
    kind="spike",
    magnitude=10.0,
    calibration_end=train_cutoff
)

In [143]:
print("Original temperature:",
      raw_fault_input.loc[injection_time, "temp"])

print("Injected temperature:",
      injected_frame.loc[injection_time, "temp"])

print("Expected synthetic labels:", int(synthetic_labels.sum()))

print("\nFault registry:")
display(fault_registry)

print("\nOriginal data unchanged:",
      model_df.loc[injection_time, "temp"] == 23.9)

print("\nChanged raw temperature timestamps:")
changed = raw_fault_input["temp"].ne(injected_frame["temp"])
print(raw_fault_input.index[changed].tolist())

Original temperature: 23.9
Injected temperature: 33.9
Expected synthetic labels: 1

Fault registry:


,timestamp,variable,fault_kind,synthetic_fault_label,value_modified,injection_interval_start,injection_interval_end,injected_timestamps_json,random_seed,magnitude,calibration_end,reference
0,2025-09-01 07:30:00+05:30,temp,spike,True,True,2025-09-01 07:30:00+05:30,2025-09-01 07:30:00+05:30,"[""2025-09-01 07:30:00+05:30""]",42,10.0,2025-03-14 23:30:00+05:30,unaltered held-out raw value; training rows ar...



Original data unchanged: True

Changed raw temperature timestamps:
[Timestamp('2025-09-01 07:30:00+0530', tz='Asia/Kolkata')]


In [144]:
# Inspect the injection timestamp and six subsequent observations

inspection_times = pd.date_range(
    start=injection_time,
    periods=7,
    freq="1h"
)

feature_columns = [
    "temp",
    "temp_change_1h",
    "temp_rolling_mean_6obs",
    "temp_deviation_6obs"
]

propagation = pd.DataFrame(index=inspection_times)

for col in feature_columns:
    propagation[f"{col}_clean"] = (
        clean_rebuilt_features.loc[inspection_times, col]
    )
    propagation[f"{col}_faulted"] = (
        faulted_features.loc[inspection_times, col]
    )
    propagation[f"{col}_difference"] = (
        propagation[f"{col}_faulted"]
        - propagation[f"{col}_clean"]
    )

display(propagation)

,temp_clean,temp_faulted,temp_difference,temp_change_1h_clean,temp_change_1h_faulted,temp_change_1h_difference,temp_rolling_mean_6obs_clean,temp_rolling_mean_6obs_faulted,temp_rolling_mean_6obs_difference,temp_deviation_6obs_clean,temp_deviation_6obs_faulted,temp_deviation_6obs_difference
2025-09-01 07:30:00+05:30,23.9,33.9,10.0,0.3,10.3,10.0,24.050000,24.050000,0.000000,-0.15,9.85,10.0
2025-09-01 08:30:00+05:30,24.3,24.3,0.0,0.4,-9.6,-10.0,23.933333,25.600000,1.666667,0.366667,-1.3,-1.666667
2025-09-01 09:30:00+05:30,24.3,24.3,0.0,0.0,0.0,0.0,23.900000,25.566667,1.666667,0.4,-1.266667,-1.666667
2025-09-01 10:30:00+05:30,24.3,24.3,0.0,0.0,0.0,0.0,23.933333,25.600000,1.666667,0.366667,-1.3,-1.666667
2025-09-01 11:30:00+05:30,24.5,24.5,0.0,0.2,0.2,0.0,24.016667,25.683333,1.666667,0.483333,-1.183333,-1.666667
2025-09-01 12:30:00+05:30,24.6,24.6,0.0,0.1,0.1,0.0,24.150000,25.816667,1.666667,0.45,-1.216667,-1.666667
2025-09-01 13:30:00+05:30,24.4,24.4,0.0,-0.2,-0.2,0.0,24.316667,25.983333,1.666667,0.083333,-1.583333,-1.666667


In [145]:
# Extract faulted test-period features
faulted_test_raw = faulted_features.loc[
    X_test_seasonal.index
].copy()

clean_test_raw = clean_rebuilt_features.loc[
    X_test_seasonal.index
].copy()

# Add seasonal features
for frame in [faulted_test_raw, clean_test_raw]:
    month = frame.index.month

    frame["month_sin"] = np.sin(2 * np.pi * month / 12)
    frame["month_cos"] = np.cos(2 * np.pi * month / 12)

# Match original model feature order
feature_columns = X_train_seasonal.columns.tolist()

faulted_test_raw = faulted_test_raw[feature_columns]
clean_test_raw = clean_test_raw[feature_columns]

print("Faulted test shape:", faulted_test_raw.shape)
print("Clean test shape:", clean_test_raw.shape)

print("\nFaulted missing values:")
print(faulted_test_raw.isna().sum())

print("\nClean reconstruction vs original input:")
print(
    "Maximum difference:",
    np.nanmax(
        np.abs(
            clean_test_raw.to_numpy(dtype=float)
            - X_test_seasonal.to_numpy(dtype=float)
        )
    )
)

KeyError: "['temp_rolling_mean_6h', 'temp_deviation_6h'] not in index"

In [146]:
# Extract test-period features
faulted_test_raw = faulted_features.loc[
    X_test_seasonal.index
].copy()

clean_test_raw = clean_rebuilt_features.loc[
    X_test_seasonal.index
].copy()

# Rename reconstructed features to original model names
rename_map = {
    "temp_rolling_mean_6obs": "temp_rolling_mean_6h",
    "temp_deviation_6obs": "temp_deviation_6h"
}

faulted_test_raw = faulted_test_raw.rename(columns=rename_map)
clean_test_raw = clean_test_raw.rename(columns=rename_map)

# Add seasonal features
for frame in [faulted_test_raw, clean_test_raw]:
    month = frame.index.month
    frame["month_sin"] = np.sin(2 * np.pi * month / 12)
    frame["month_cos"] = np.cos(2 * np.pi * month / 12)

# Match original model feature order
feature_columns = X_train_seasonal.columns.tolist()

faulted_test_raw = faulted_test_raw[feature_columns]
clean_test_raw = clean_test_raw[feature_columns]

print("Faulted test shape:", faulted_test_raw.shape)
print("Clean test shape:", clean_test_raw.shape)

print("\nFaulted missing values:")
print(faulted_test_raw.isna().sum())

print("\nClean reconstruction maximum difference:")
print(
    np.nanmax(
        np.abs(
            clean_test_raw.to_numpy(dtype=float)
            - X_test_seasonal.to_numpy(dtype=float)
        )
    )
)

Faulted test shape: (8415, 11)
Clean test shape: (8415, 11)

Faulted missing values:
temp                    0
dwpt                    0
rhum                    0
prcp                    0
wspd                    0
pres                    0
temp_change_1h          0
temp_rolling_mean_6h    0
temp_deviation_6h       0
month_sin               0
month_cos               0
dtype: int64

Clean reconstruction maximum difference:
7.105427357601002e-15


In [147]:
from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd

# Fit exclusively on original training data
fault_scaler = StandardScaler()
fault_scaler.fit(X_train_seasonal)

# Transform clean and faulted test matrices
clean_scaled_check = pd.DataFrame(
    fault_scaler.transform(clean_test_raw),
    index=clean_test_raw.index,
    columns=clean_test_raw.columns
)

faulted_test_scaled = pd.DataFrame(
    fault_scaler.transform(faulted_test_raw),
    index=faulted_test_raw.index,
    columns=faulted_test_raw.columns
)

# Verify clean scaled reconstruction
clean_scaled_difference = np.max(
    np.abs(
        clean_scaled_check.to_numpy()
        - X_test_scaled.to_numpy()
    )
)

print("Clean scaled reconstruction difference:",
      clean_scaled_difference)

print("Faulted scaled matrix shape:",
      faulted_test_scaled.shape)

print("Faulted scaled matrix finite:",
      np.isfinite(faulted_test_scaled.to_numpy()).all())

print("Training data unchanged:",
      X_train_seasonal.equals(X_train_seasonal.copy()))

Clean scaled reconstruction difference: 8.881784197001252e-16
Faulted scaled matrix shape: (8415, 11)
Faulted scaled matrix finite: True
Training data unchanged: True


In [151]:
from weather_research.models import fit_predict_configurations
import pandas as pd

# 1. Generate predictions on the faulted test dataset
faulted_recheck = fit_predict_configurations(
    x_train_unscaled=X_train_seasonal,
    x_test_unscaled=faulted_test_raw,
    x_train_scaled=X_train_scaled,
    x_test_scaled=faulted_test_scaled,
    configs=configs
)

# 2. Convert predictions to series with a unique MultiIndex to avoid duplicates
clean_pred = baseline_recheck.set_index("configuration", append=True)["prediction"]
fault_pred = faulted_recheck.set_index("configuration", append=True)["prediction"]

# 3. Align by unique (timestamp, configuration) keys
comparison = pd.concat(
    [
        clean_pred.rename("clean"),
        fault_pred.rename("faulted")
    ],
    axis=1
)

comparison["prediction_changed"] = (
    comparison["clean"] != comparison["faulted"]
)

print("Total prediction changes:",
      int(comparison["prediction_changed"].sum()))

print("\nChanges by configuration:")
display(
    comparison.groupby(level="configuration")
    ["prediction_changed"]
    .sum()
    .to_frame("changed_predictions")
)

# Inspect the injection timestamp
print("\nPredictions at injected timestamp:")
display(comparison.loc[injection_time])

# Inspect the next six observations
following_times = pd.date_range(
    injection_time + pd.Timedelta(hours=1),
    periods=6,
    freq="1h"
)

print("\nPredictions during rolling-feature propagation:")
display(comparison.loc[
    comparison.index.get_level_values("timestamp").isin(following_times)
].reset_index())

Total prediction changes: 21

Changes by configuration:


,changed_predictions
configuration,
IF_0.01,3
IF_0.02,3
IF_0.03,2
IF_0.05,1
LOF_10,1
LOF_20,1
LOF_30,2
LOF_50,1
SVM_0.01,2



Predictions at injected timestamp:


,clean,faulted,prediction_changed
configuration,,,
SVM_0.03,1,-1,True
IF_0.01,1,-1,True
SVM_0.05,1,-1,True
IF_0.02,1,-1,True
LOF_50,1,-1,True
LOF_10,1,-1,True
LOF_30,1,-1,True
IF_0.05,1,-1,True
SVM_0.01,1,-1,True



Predictions during rolling-feature propagation:


,timestamp,configuration,clean,faulted,prediction_changed
0,2025-09-01 08:30:00+05:30,LOF_20,-1,-1,False
1,2025-09-01 08:30:00+05:30,LOF_30,-1,-1,False
2,2025-09-01 08:30:00+05:30,LOF_50,-1,-1,False
3,2025-09-01 08:30:00+05:30,LOF_10,-1,-1,False
4,2025-09-01 08:30:00+05:30,IF_0.02,1,-1,True
...,...,...,...,...,...
67,2025-09-01 13:30:00+05:30,LOF_10,1,1,False
68,2025-09-01 13:30:00+05:30,IF_0.05,-1,-1,False
69,2025-09-01 13:30:00+05:30,IF_0.01,1,1,False
70,2025-09-01 13:30:00+05:30,SVM_0.03,-1,-1,False


In [154]:
# Work with the aligned comparison MultiIndex
comparison = comparison.copy()

comparison["change_type"] = "Unchanged"

comparison.loc[
    (comparison["clean"] == 1) &
    (comparison["faulted"] == -1),
    "change_type"
] = "Newly detected"

comparison.loc[
    (comparison["clean"] == -1) &
    (comparison["faulted"] == 1),
    "change_type"
] = "No longer detected"

# Define affected timestamps:
# injection timestamp + six subsequent observations
affected_times = pd.DatetimeIndex(
    [injection_time] + list(following_times)
)

timestamps = comparison.index.get_level_values("timestamp")
comparison["affected_window"] = timestamps.isin(affected_times)

print("Prediction change directions:")
display(
    comparison.groupby("change_type")
    .size()
    .to_frame("count")
)

print("\nChanges inside and outside affected window:")
display(
    comparison.groupby(
        ["affected_window", "change_type"]
    ).size().unstack(fill_value=0)
)

print("\nNew detections by configuration:")
display(
    comparison[comparison["change_type"] == "Newly detected"]
    .groupby(level="configuration")
    .size()
    .to_frame("new_detections")
)

Prediction change directions:


,count
change_type,
Newly detected,17
No longer detected,4
Unchanged,100959



Changes inside and outside affected window:


change_type,Newly detected,No longer detected,Unchanged
affected_window,,,
False,0,0,100896
True,17,4,63



New detections by configuration:


,new_detections
configuration,
IF_0.01,1
IF_0.02,2
IF_0.03,2
IF_0.05,1
LOF_10,1
LOF_20,1
LOF_30,1
LOF_50,1
SVM_0.01,2


In [155]:
# Reconstruct existing model votes safely
pred_for_selection = prediction_data.reset_index(drop=True).copy()

vote_counts = (
    pred_for_selection
    .pivot_table(
        index="timestamp",
        values="is_candidate",
        aggfunc="sum"
    )["is_candidate"]
)

# Use the original segment 145 timestamps
candidate_times = pd.DatetimeIndex(
    model_df.index[
        (model_df["segment_id"] == 145)
        & model_df.index.isin(X_test_seasonal.index)
    ]
)

eligible_5 = []

for i in range(6, len(candidate_times) - 6):

    timestamp = candidate_times[i]
    window = candidate_times[i-6:i+7]

    # Require 13 consecutive hourly observations
    if not (
        window.to_series().diff().dropna()
        == pd.Timedelta(hours=1)
    ).all():
        continue

    # Exclude the previous experiment and its nearby window
    if abs(timestamp - injection_time) < pd.Timedelta(hours=24):
        continue

    votes = int(vote_counts.get(timestamp, 0))

    if votes == 0:
        eligible_5.append({
            "timestamp": timestamp,
            "existing_votes": votes,
            "temperature": model_df.loc[timestamp, "temp"]
        })

eligible_5_df = pd.DataFrame(eligible_5)

print("Eligible timestamps:", len(eligible_5_df))

if len(eligible_5_df) > 0:
    middle_timestamp = candidate_times[len(candidate_times)//2]

    eligible_5_df["middle_distance"] = (
        eligible_5_df["timestamp"] - middle_timestamp
    ).abs().dt.total_seconds()

    selected_5 = eligible_5_df.sort_values(
        "middle_distance"
    ).iloc[0]

    print("\nSelected timestamp for +5°C experiment:")
    display(selected_5.to_frame("value"))
else:
    print("No suitable timestamp found.")

Eligible timestamps: 6624

Selected timestamp for +5°C experiment:


,value
timestamp,2025-09-02 07:30:00+05:30
existing_votes,0
temperature,23.9
middle_distance,82800.0


In [156]:
injection_time_5 = pd.Timestamp(
    "2025-09-02 07:30:00",
    tz="Asia/Kolkata"
)

raw_fault_input_5 = model_df[
    ["temp", "dwpt", "rhum", "prcp", "wspd", "pres", "segment_id"]
].copy()

(
    injected_frame_5,
    synthetic_labels_5,
    fault_registry_5,
    clean_features_5,
    faulted_features_5
) = inject_fault_with_feature_recompute(
    frame=raw_fault_input_5,
    variable="temp",
    start=injection_time_5,
    kind="spike",
    magnitude=5.0,
    calibration_end=X_train_seasonal.index.max()
)

In [157]:
print("Original temperature:",
      raw_fault_input_5.loc[injection_time_5, "temp"])

print("Injected temperature:",
      injected_frame_5.loc[injection_time_5, "temp"])

print("Synthetic fault labels:",
      int(synthetic_labels_5.sum()))

print("Original data unchanged:",
      model_df.loc[injection_time_5, "temp"] == 23.9)

print("Changed raw timestamps:",
      raw_fault_input_5.index[
          raw_fault_input_5["temp"].ne(injected_frame_5["temp"])
      ].tolist())

display(fault_registry_5)

Original temperature: 23.9
Injected temperature: 28.9
Synthetic fault labels: 1
Original data unchanged: True
Changed raw timestamps: [Timestamp('2025-09-02 07:30:00+0530', tz='Asia/Kolkata')]


,timestamp,variable,fault_kind,synthetic_fault_label,value_modified,injection_interval_start,injection_interval_end,injected_timestamps_json,random_seed,magnitude,calibration_end,reference
0,2025-09-02 07:30:00+05:30,temp,spike,True,True,2025-09-02 07:30:00+05:30,2025-09-02 07:30:00+05:30,"[""2025-09-02 07:30:00+05:30""]",42,5.0,2025-03-14 23:30:00+05:30,unaltered held-out raw value; training rows ar...


In [159]:
inspection_times_5 = pd.date_range(
    start=injection_time_5,
    periods=7,
    freq="1h"
)

feature_columns_5 = [
    "temp",
    "temp_change_1h",
    "temp_rolling_mean_6obs",
    "temp_deviation_6obs"
]

propagation_5 = pd.DataFrame(index=inspection_times_5)

for col in feature_columns_5:
    propagation_5[f"{col}_clean"] = (
        clean_features_5.loc[inspection_times_5, col]
    )

    propagation_5[f"{col}_faulted"] = (
        faulted_features_5.loc[inspection_times_5, col]
    )

    propagation_5[f"{col}_difference"] = (
        propagation_5[f"{col}_faulted"]
        - propagation_5[f"{col}_clean"]
    )

display(propagation_5)

,temp_clean,temp_faulted,temp_difference,temp_change_1h_clean,temp_change_1h_faulted,temp_change_1h_difference,temp_rolling_mean_6obs_clean,temp_rolling_mean_6obs_faulted,temp_rolling_mean_6obs_difference,temp_deviation_6obs_clean,temp_deviation_6obs_faulted,temp_deviation_6obs_difference
2025-09-02 07:30:00+05:30,23.9,28.9,5.0,0.7,5.7,5.0,23.133333,23.133333,0.000000,0.766667,5.766667,5.0
2025-09-02 08:30:00+05:30,24.7,24.7,0.0,0.8,-4.2,-5.0,23.250000,24.083333,0.833333,1.45,0.616667,-0.833333
2025-09-02 09:30:00+05:30,26.0,26.0,0.0,1.3,1.3,0.0,23.500000,24.333333,0.833333,2.5,1.666667,-0.833333
2025-09-02 10:30:00+05:30,26.6,26.6,0.0,0.6,0.6,0.0,23.983333,24.816667,0.833333,2.616667,1.783333,-0.833333
2025-09-02 11:30:00+05:30,26.8,26.8,0.0,0.2,0.2,0.0,24.583333,25.416667,0.833333,2.216667,1.383333,-0.833333
2025-09-02 12:30:00+05:30,27.2,27.2,0.0,0.4,0.4,0.0,25.200000,26.033333,0.833333,2.0,1.166667,-0.833333
2025-09-02 13:30:00+05:30,27.3,27.3,0.0,0.1,0.1,0.0,25.866667,26.700000,0.833333,1.433333,0.6,-0.833333


In [160]:
inspection_times_5 = pd.date_range(
    start=injection_time_5,
    periods=7,
    freq="1h"
)

feature_columns_5 = [
    "temp",
    "temp_change_1h",
    "temp_rolling_mean_6obs",
    "temp_deviation_6obs"
]

propagation_5 = pd.DataFrame(index=inspection_times_5)

for col in feature_columns_5:
    propagation_5[f"{col}_clean"] = (
        clean_features_5.loc[inspection_times_5, col]
    )

    propagation_5[f"{col}_faulted"] = (
        faulted_features_5.loc[inspection_times_5, col]
    )

    propagation_5[f"{col}_difference"] = (
        propagation_5[f"{col}_faulted"]
        - propagation_5[f"{col}_clean"]
    )

display(propagation_5)

,temp_clean,temp_faulted,temp_difference,temp_change_1h_clean,temp_change_1h_faulted,temp_change_1h_difference,temp_rolling_mean_6obs_clean,temp_rolling_mean_6obs_faulted,temp_rolling_mean_6obs_difference,temp_deviation_6obs_clean,temp_deviation_6obs_faulted,temp_deviation_6obs_difference
2025-09-02 07:30:00+05:30,23.9,28.9,5.0,0.7,5.7,5.0,23.133333,23.133333,0.000000,0.766667,5.766667,5.0
2025-09-02 08:30:00+05:30,24.7,24.7,0.0,0.8,-4.2,-5.0,23.250000,24.083333,0.833333,1.45,0.616667,-0.833333
2025-09-02 09:30:00+05:30,26.0,26.0,0.0,1.3,1.3,0.0,23.500000,24.333333,0.833333,2.5,1.666667,-0.833333
2025-09-02 10:30:00+05:30,26.6,26.6,0.0,0.6,0.6,0.0,23.983333,24.816667,0.833333,2.616667,1.783333,-0.833333
2025-09-02 11:30:00+05:30,26.8,26.8,0.0,0.2,0.2,0.0,24.583333,25.416667,0.833333,2.216667,1.383333,-0.833333
2025-09-02 12:30:00+05:30,27.2,27.2,0.0,0.4,0.4,0.0,25.200000,26.033333,0.833333,2.0,1.166667,-0.833333
2025-09-02 13:30:00+05:30,27.3,27.3,0.0,0.1,0.1,0.0,25.866667,26.700000,0.833333,1.433333,0.6,-0.833333


In [162]:

# ================================================================
# WEATHER STATION ANOMALY DETECTION
# SYNTHETIC FAULT INJECTION MASTER PIPELINE
# Version: 1.0
#
# Current scope: Single-point temperature spike magnitude study
# Experiments: +5 C, +10 C, +15 C
# Models: 12 configurations
# ================================================================

import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler

from weather_research.faults import (
    inject_fault_with_feature_recompute
)

from weather_research.models import (
    fit_predict_configurations,
    default_model_configs
)

# ------------------------------------------------
# 1. CONFIGURATION AND PRECONDITION CHECKS
# ------------------------------------------------

RAW_COLUMNS = [
    "temp", "dwpt", "rhum",
    "prcp", "wspd", "pres", "segment_id"
]

FEATURE_RENAME = {
    "temp_rolling_mean_6obs": "temp_rolling_mean_6h",
    "temp_deviation_6obs": "temp_deviation_6h"
}

TRAIN_CUTOFF = X_train_seasonal.index.max()

FEATURE_COLUMNS = X_train_seasonal.columns.tolist()

if "baseline_recheck" not in globals():
    raise RuntimeError("Verified clean baseline predictions are missing.")

if not X_train_seasonal.index.equals(X_train_scaled.index):
    raise AssertionError("Training matrices are not aligned.")

if not X_test_seasonal.index.equals(X_test_scaled.index):
    raise AssertionError("Testing matrices are not aligned.")

if not np.isfinite(X_train_seasonal.to_numpy()).all():
    raise ValueError("Unscaled training matrix contains non-finite values.")

if not np.isfinite(X_train_scaled.to_numpy()).all():
    raise ValueError("Scaled training matrix contains non-finite values.")

if not np.isfinite(X_test_seasonal.to_numpy()).all():
    raise ValueError("Unscaled testing matrix contains non-finite values.")

if not np.isfinite(X_test_scaled.to_numpy()).all():
    raise ValueError("Scaled testing matrix contains non-finite values.")

# Recreate the original training-only scaler.
# Never fit this scaler on test or faulted data.

master_scaler = StandardScaler()
master_scaler.fit(X_train_seasonal)

scaling_check = np.max(np.abs(
    master_scaler.transform(X_test_seasonal)
    - X_test_scaled.to_numpy()
))

if scaling_check > 1e-10:
    raise AssertionError("Training-only scaling does not reproduce baseline.")

# Use the original model configurations.
MASTER_CONFIGS = (
    configs.copy() if "configs" in globals()
    else default_model_configs()
)

if len(MASTER_CONFIGS) != 12:
    raise AssertionError("Expected exactly 12 model configurations.")

print("Master pipeline initialized.")
print("Training observations:", len(X_train_seasonal))
print("Testing observations:", len(X_test_seasonal))
print("Features:", len(FEATURE_COLUMNS))
print("Model configurations:", len(MASTER_CONFIGS))
print("Scaler reproduction difference:", scaling_check)


# ------------------------------------------------
# 2. SAFE PREDICTION ALIGNMENT
# ------------------------------------------------

def prediction_series(frame):

    frame = frame.copy()

    # Resolve timestamp index/column ambiguity.
    if "timestamp" in frame.columns:
        frame.index = pd.RangeIndex(len(frame))
        frame = frame.set_index("timestamp")

    if "configuration" not in frame.columns:
        raise ValueError("Configuration column is missing.")

    return frame.set_index(
        "configuration",
        append=True
    )["prediction"]


CLEAN_PREDICTIONS = prediction_series(baseline_recheck)


# ------------------------------------------------
# 3. SELECT A CLEAN INJECTION TIMESTAMP
# ------------------------------------------------

def select_injection_timestamp(excluded_times=None):

    excluded_times = excluded_times or []

    prediction_table = prediction_data.reset_index(
        drop=True
    ).copy()

    vote_counts = (
        prediction_table
        .pivot_table(
            index="timestamp",
            values="is_candidate",
            aggfunc="sum"
        )["is_candidate"]
    )

    candidate_times = pd.DatetimeIndex(
        model_df.index[
            (model_df["segment_id"] == 145)
            & model_df.index.isin(X_test_seasonal.index)
        ]
    )

    eligible = []

    for i in range(6, len(candidate_times) - 6):

        timestamp = candidate_times[i]
        window = candidate_times[i-6:i+7]

        # Require 13 consecutive hourly observations.
        if not (
            window.to_series().diff().dropna()
            == pd.Timedelta(hours=1)
        ).all():
            continue

        # Keep experiments separated by at least 24 hours.
        if any(
            abs(timestamp - old_time) < pd.Timedelta(hours=24)
            for old_time in excluded_times
        ):
            continue

        votes = int(vote_counts.get(timestamp, 0))

        # Prefer zero-vote timestamps.
        if votes != 0:
            continue

        eligible.append({
            "timestamp": timestamp,
            "existing_votes": votes,
            "temperature": model_df.loc[timestamp, "temp"]
        })

    eligible_df = pd.DataFrame(eligible)

    if eligible_df.empty:
        raise RuntimeError("No suitable injection timestamp found.")

    middle_timestamp = candidate_times[len(candidate_times)//2]

    eligible_df["middle_distance"] = (
        eligible_df["timestamp"] - middle_timestamp
    ).abs().dt.total_seconds()

    selected = eligible_df.sort_values(
        "middle_distance"
    ).iloc[0]

    return selected


# ------------------------------------------------
# 4. FEATURE RECONSTRUCTION
# ------------------------------------------------

def prepare_model_features(engineered_frame):

    # Align source timezone representation with test index timezone representation
    if engineered_frame.index.tz is not None and X_test_seasonal.index.tz is not None:
        engineered_frame.index = engineered_frame.index.tz_convert(X_test_seasonal.index.tz)

    test_frame = engineered_frame.loc[
        X_test_seasonal.index
    ].copy()

    # Rename features to match the original model.
    test_frame = test_frame.rename(
        columns=FEATURE_RENAME
    )

    # Add monthly seasonal features.
    month = test_frame.index.month

    test_frame["month_sin"] = np.sin(
        2 * np.pi * month / 12
    )

    test_frame["month_cos"] = np.cos(
        2 * np.pi * month / 12
    )

    # Restore exact original feature order.
    test_frame = test_frame[FEATURE_COLUMNS]

    if not test_frame.index.equals(X_test_seasonal.index):
        raise AssertionError("Test timestamps are not aligned.")

    if not np.isfinite(test_frame.to_numpy(dtype=float)).all():
        raise ValueError(
            "Faulted model features contain missing or infinite values. "
            "Resolve imputation using training-only medians before proceeding."
        )

    return test_frame


# ------------------------------------------------
# 5. RUN ONE COMPLETE INJECTION EXPERIMENT
# ------------------------------------------------

def run_spike_experiment(
    experiment_name,
    timestamp,
    magnitude
):

    print("\n" + "=" * 65)
    print("STARTING:", experiment_name)
    print("=" * 65)

    # Always start from untouched original observations.
    raw_input = model_df[RAW_COLUMNS].copy(deep=True)

    # Inject fault into a separate copy.
    (
        injected_frame,
        synthetic_labels,
        fault_registry,
        clean_features,
        faulted_features
    ) = inject_fault_with_feature_recompute(
        frame=raw_input,
        variable="temp",
        start=timestamp,
        kind="spike",
        magnitude=magnitude,
        calibration_end=TRAIN_CUTOFF
    )

    # Injection verification.
    changed_raw = raw_input["temp"].ne(
        injected_frame["temp"]
    )

    if int(synthetic_labels.sum()) != 1:
        raise AssertionError("Expected exactly one synthetic fault label.")

    if int(changed_raw.sum()) != 1:
        raise AssertionError("Expected exactly one changed raw observation.")

    if not model_df.loc[timestamp, "temp"] == raw_input.loc[timestamp, "temp"]:
        raise AssertionError("Original dataset was unexpectedly modified.")

    # Reconstruct clean and faulted test inputs.
    clean_test = prepare_model_features(clean_features)

    faulted_test = prepare_model_features(faulted_features)

    # Validate clean reconstruction.
    clean_difference = np.max(np.abs(
        clean_test.to_numpy(dtype=float)
        - X_test_seasonal.to_numpy(dtype=float)
    ))

    if clean_difference > 1e-10:
        raise AssertionError("Clean feature reconstruction failed.")

    # Apply unchanged training-fitted scaler.
    clean_scaled = pd.DataFrame(
        master_scaler.transform(clean_test),
        index=clean_test.index,
        columns=clean_test.columns
    )

    faulted_scaled = pd.DataFrame(
        master_scaler.transform(faulted_test),
        index=faulted_test.index,
        columns=faulted_test.columns
    )

    scaled_difference = np.max(np.abs(
        clean_scaled.to_numpy()
        - X_test_scaled.to_numpy()
    ))

    if scaled_difference > 1e-10:
        raise AssertionError("Clean scaling reproduction failed.")

    # Fit configurations on unchanged training observations.
    faulted_predictions = fit_predict_configurations(
        x_train_unscaled=X_train_seasonal,
        x_test_unscaled=faulted_test,
        x_train_scaled=X_train_scaled,
        x_test_scaled=faulted_scaled,
        configs=MASTER_CONFIGS
    )

    # Compare against verified clean baseline.
    faulted_pred = prediction_series(faulted_predictions)

    if not CLEAN_PREDICTIONS.index.equals(faulted_pred.index):
        raise AssertionError("Clean and faulted prediction indexes differ.")

    comparison = pd.concat(
        [
            CLEAN_PREDICTIONS.rename("clean"),
            faulted_pred.rename("faulted")
        ],
        axis=1
    )

    comparison["prediction_changed"] = (
        comparison["clean"] != comparison["faulted"]
    )

    comparison["change_type"] = "Unchanged"

    comparison.loc[
        (comparison["clean"] == 1)
        & (comparison["faulted"] == -1),
        "change_type"
    ] = "Newly detected"

    comparison.loc[
        (comparison["clean"] == -1)
        & (comparison["faulted"] == 1),
        "change_type"
    ] = "No longer detected"

    # Seven timestamps: injection plus six following observations.
    affected_times = pd.date_range(
        timestamp,
        periods=7,
        freq="1h"
    )

    timestamps = comparison.index.get_level_values(0)

    comparison["affected_window"] = timestamps.isin(
        affected_times
    )

    # Direct detection at the injected timestamp.
    direct = comparison.xs(
        timestamp,
        level=0
    )

    direct_detection_count = int(
        (direct["clean"].eq(1) & direct["faulted"].eq(-1)).sum()
    )

    direct_anomaly_count = int(
        direct["faulted"].eq(-1).sum()
    )

    # Summary counts.
    change_counts = comparison["change_type"].value_counts()

    changes_inside = int(
        comparison.loc[
            comparison["affected_window"],
            "prediction_changed"
        ].sum()
    )

    changes_outside = int(
        comparison.loc[
            ~comparison["affected_window"],
            "prediction_changed"
        ].sum()
    )

    # Per-configuration summary.
    configuration_summary = (
        comparison.reset_index()
        .groupby("configuration")
        .agg(
            changed_predictions=("prediction_changed", "sum"),
            newly_detected=(
                "change_type",
                lambda s: (s == "Newly detected").sum()
            ),
            no_longer_detected=(
                "change_type",
                lambda s: (s == "No longer detected").sum()
            )
        )
    )

    # Timestamp-level propagation summary.
    propagation_summary = (
        comparison.reset_index()
        .loc[lambda x: x["timestamp"].isin(affected_times)]
        .groupby("timestamp")
        .agg(
            newly_detected=(
                "change_type",
                lambda s: (s == "Newly detected").sum()
            ),
            no_longer_detected=(
                "change_type",
                lambda s: (s == "No longer detected").sum()
            ),
            unchanged=(
                "change_type",
                lambda s: (s == "Unchanged").sum()
            )
        )
    )

    summary = {
        "experiment": experiment_name,
        "timestamp": timestamp,
        "magnitude_C": magnitude,
        "original_temp_C": raw_input.loc[timestamp, "temp"],
        "injected_temp_C": injected_frame.loc[timestamp, "temp"],
        "synthetic_labels": int(synthetic_labels.sum()),
        "direct_new_detections": direct_detection_count,
        "direct_anomalous_predictions": direct_anomaly_count,
        "total_prediction_changes": int(
            comparison["prediction_changed"].sum()
        ),
        "newly_detected": int(
            (comparison["change_type"] == "Newly detected").sum()
        ),
        "no_longer_detected": int(
            (comparison["change_type"] == "No longer detected").sum()
        ),
        "changes_inside_window": changes_inside,
        "changes_outside_window": changes_outside,
        "clean_reconstruction_difference": clean_difference,
        "clean_scaling_difference": scaled_difference
    }

    print("\nEXPERIMENT SUMMARY")
    display(pd.DataFrame([summary]))

    print("\nPREDICTION CHANGE DIRECTIONS")
    display(change_counts.to_frame("count"))

    print("\nCONFIGURATION RESULTS")
    display(configuration_summary)

    print("\nTEMPORAL PROPAGATION RESULTS")
    display(propagation_summary)

    print("\nFAULT REGISTRY")
    display(fault_registry)

    return {
        "summary": summary,
        "comparison": comparison,
        "predictions": faulted_predictions,
        "registry": fault_registry,
        "labels": synthetic_labels,
        "injected_frame": injected_frame,
        "clean_features": clean_features,
        "faulted_features": faulted_features,
        "configuration_summary": configuration_summary,
        "propagation_summary": propagation_summary
    }


# ------------------------------------------------
# 6. DEFINE EXPERIMENTS
# ------------------------------------------------

# Previously verified injection timestamps.
experiment_specs = [
    {
        "name": "Temperature spike +5C",
        "timestamp": pd.Timestamp(
            "2025-09-02 07:30:00",
            tz="Asia/Kolkata"
        ),
        "magnitude": 5.0
    },
    {
        "name": "Temperature spike +10C",
        "timestamp": pd.Timestamp(
            "2025-09-01 07:30:00",
            tz="Asia/Kolkata"
        ),
        "magnitude": 10.0
    }
]

# Automatically select a third timestamp for +15C.
excl = [
    spec["timestamp"] for spec in experiment_specs
]

selected_15 = select_injection_timestamp(
    excluded_times=excl
)

experiment_specs.append({
    "name": "Temperature spike +15C",
    "timestamp": selected_15["timestamp"],
    "magnitude": 15.0
})

print("\nEXPERIMENT PLAN")
display(pd.DataFrame(experiment_specs))


# ------------------------------------------------
# 7. EXECUTE EXPERIMENTS
# ------------------------------------------------

spike_test_results = {}

for spec in experiment_specs:

    name = spec["name"]

    result = run_spike_experiment(
        experiment_name=name,
        timestamp=spec["timestamp"],
        magnitude=spec["magnitude"]
    )

    spike_test_results[name] = result


# ------------------------------------------------
# 8. FINAL COMPARISON TABLE
# ------------------------------------------------

master_summary = pd.DataFrame([
    result["summary"]
    for result in spike_test_results.values()
])

print("\n" + "=" * 65)
print("SYNTHETIC TEMPERATURE SPIKE STUDY — FINAL SUMMARY")
print("=" * 65)

display(master_summary)

print("\nAll experiments completed.")


Master pipeline initialized.
Training observations: 36434
Testing observations: 8415
Features: 11
Model configurations: 12
Scaler reproduction difference: 0.0

EXPERIMENT PLAN


,name,timestamp,magnitude
0,Temperature spike +5C,2025-09-02 07:30:00+05:30,5.0
1,Temperature spike +10C,2025-09-01 07:30:00+05:30,10.0
2,Temperature spike +15C,2025-08-31 07:30:00+05:30,15.0



STARTING: Temperature spike +5C

EXPERIMENT SUMMARY


,experiment,timestamp,magnitude_C,original_temp_C,injected_temp_C,synthetic_labels,direct_new_detections,direct_anomalous_predictions,total_prediction_changes,newly_detected,no_longer_detected,changes_inside_window,changes_outside_window,clean_reconstruction_difference,clean_scaling_difference
0,Temperature spike +5C,2025-09-02 07:30:00+05:30,5.0,23.9,28.9,1,9,9,16,15,1,16,0,7.105427e-15,8.881784e-16



PREDICTION CHANGE DIRECTIONS


,count
change_type,
Unchanged,100964
Newly detected,15
No longer detected,1


... (25 outputs left out)

,timestamp,variable,fault_kind,synthetic_fault_label,value_modified,injection_interval_start,injection_interval_end,injected_timestamps_json,random_seed,magnitude,calibration_end,reference
0,2025-08-31 07:30:00+05:30,temp,spike,True,True,2025-08-31 07:30:00+05:30,2025-08-31 07:30:00+05:30,"[""2025-08-31 07:30:00+05:30""]",42,15.0,2025-03-14 23:30:00+05:30,unaltered held-out raw value; training rows ar...



SYNTHETIC TEMPERATURE SPIKE STUDY — FINAL SUMMARY


,experiment,timestamp,magnitude_C,original_temp_C,injected_temp_C,synthetic_labels,direct_new_detections,direct_anomalous_predictions,total_prediction_changes,newly_detected,no_longer_detected,changes_inside_window,changes_outside_window,clean_reconstruction_difference,clean_scaling_difference
0,Temperature spike +5C,2025-09-02 07:30:00+05:30,5.0,23.9,28.9,1,9,9,16,15,1,16,0,7.105427e-15,8.881784e-16
1,Temperature spike +10C,2025-09-01 07:30:00+05:30,10.0,23.9,33.9,1,12,12,21,17,4,21,0,7.105427e-15,8.881784e-16
2,Temperature spike +15C,2025-08-31 07:30:00+05:30,15.0,24.7,39.7,1,12,12,22,22,0,22,0,7.105427e-15,8.881784e-16



All experiments completed.


In [164]:
import numpy as np
import pandas as pd
import json
from pathlib import Path
from sklearn.preprocessing import StandardScaler

# ---------------- CONFIGURATION ----------------

START = pd.Timestamp(
    "2025-09-02 07:30:00",
    tz="Asia/Kolkata"
)

SEED = 42
RUN_SPATIAL_DIAGNOSTIC = True

RAW_COLS = [
    "temp", "dwpt", "rhum",
    "prcp", "wspd", "pres", "segment_id"
]

RENAME = {
    "temp_rolling_mean_6obs": "temp_rolling_mean_6h",
    "temp_deviation_6obs": "temp_deviation_6h"
}

BOUNDS = {
    "temp": (-90, 65),
    "dwpt": (-100, 60),
    "rhum": (0, 100),
    "prcp": (0, np.inf),
    "wspd": (0, np.inf),
    "pres": (800, 1100)
}

# Controlled temperature experiments
CONTROLLED = [2, 5, 10, 15]

# Point faults in other weather variables
POINT = {
    "dwpt": [-5, -2, 2, 5],
    "rhum": [-20, -10, 10, 20],
    "prcp": [0.5, 2, 5],
    "wspd": [2, 5, 8],
    "pres": [-5, -3, 3, 5]
}

# Temporal fault experiments
TEMPORAL = [
    ("step", 3, 3),
    ("step", 6, 3),
    ("step", 12, 3),

    ("drift", 3, 3),
    ("drift", 6, 3),
    ("drift", 12, 3),

    ("stuck", 3, None),
    ("stuck", 6, None),
    ("stuck", 12, None),

    ("noise", 3, 1),
    ("noise", 6, 1),
    ("noise", 12, 1),

    ("dropout", 1, None),
    ("dropout", 3, None),
    ("dropout", 6, None)
]

# ---------------- VERIFY PHASE 03 ----------------

needed = [
    "model_df",
    "X_train_seasonal",
    "X_test_seasonal",
    "X_train_scaled",
    "X_test_scaled",
    "baseline_recheck",
    "configs",
    "fit_predict_configurations"
]

missing = [
    name for name in needed
    if name not in globals()
]

if missing:
    raise RuntimeError(
        "Run Phase 03 baseline cells first. Missing: "
        + str(missing)
    )

if len(configs) != 12:
    raise ValueError(
        f"Expected 12 configurations, got {len(configs)}"
    )

if not model_df.index.is_monotonic_increasing:
    raise ValueError("model_df index is not sorted.")

if model_df.index.has_duplicates:
    raise ValueError("Duplicate timestamps found in model_df.")

if not X_train_seasonal.index.equals(X_train_scaled.index):
    raise AssertionError("Training indexes do not match.")

if not X_test_seasonal.index.equals(X_test_scaled.index):
    raise AssertionError("Testing indexes do not match.")

# Import canonical feature engineering function
try:
    engineer_features
except NameError:
    from weather_research.features import engineer_features

print("Environment verification successful.")

print("\nDataset shapes:")
print("Raw dataset:", model_df.shape)
print("Training:", X_train_seasonal.shape)
print("Testing:", X_test_seasonal.shape)
print("Baseline predictions:", baseline_recheck.shape)

print("\nModel configurations:", len(configs))
print("Experiment timestamp:", START)

Environment verification successful.

Dataset shapes:
Raw dataset: (44879, 10)
Training: (36434, 11)
Testing: (8415, 11)
Baseline predictions: (100980, 6)

Model configurations: 12
Experiment timestamp: 2025-09-02 07:30:00+05:30


In [166]:
# ---------------- PREPARE CLEAN DATA ----------------

BASE = model_df[RAW_COLS].copy(deep=True)

# Fit scaler ONLY on training data
SCALER = StandardScaler()
SCALER.fit(X_train_seasonal)

# Training-only imputation values
TRAIN_MEDIANS = X_train_seasonal.median(
    numeric_only=True
)

# Verify scaler reproduction
scale_error = float(
    np.max(
        np.abs(
            SCALER.transform(X_test_seasonal)
            - X_test_scaled.to_numpy()
        )
    )
)

print("Scaler reconstruction error:", scale_error)

if scale_error > 1e-10:
    raise AssertionError(
        "Scaler mismatch. Stop and investigate."
    )


# ---------------- FEATURE RECONSTRUCTION ----------------

def make_features(raw):

    f = engineer_features(
        raw.copy(deep=True),
        segment_col="segment_id"
    )

    f = f.rename(columns=RENAME)

    # Seasonal features
    month = f.index.month

    f["month_sin"] = np.sin(
        2 * np.pi * month / 12
    )

    f["month_cos"] = np.cos(
        2 * np.pi * month / 12
    )

    cols = X_train_seasonal.columns.tolist()

    absent = sorted(
        set(cols) - set(f.columns)
    )

    if absent:
        raise KeyError(
            "Missing engineered columns: " + str(absent)
        )

    # Use training-only medians
    f = f[cols].fillna(TRAIN_MEDIANS)

    if not np.isfinite(
        f.to_numpy(dtype=float)
    ).all():
        raise ValueError(
            "Non-finite values found in model features."
        )

    return f


# Rebuild clean features
CLEAN_FEATURES = make_features(BASE)

CLEAN_TEST = CLEAN_FEATURES.loc[
    X_test_seasonal.index
]

# Compare with original test features
rebuild_error = float(
    np.max(
        np.abs(
            CLEAN_TEST.to_numpy(dtype=float)
            - X_test_seasonal.to_numpy(dtype=float)
        )
    )
)

print("\nFeature reconstruction error:", rebuild_error)

if rebuild_error > 1e-8:
    raise AssertionError(
        "Clean feature reconstruction mismatch."
    )

print("\nClean feature reconstruction successful.")
print("Clean test shape:", CLEAN_TEST.shape)


# ---------------- BASELINE PREDICTIONS ----------------

def pred_series(frame):

    d = frame.copy()

    if "timestamp" in d.columns:
        ts = pd.DatetimeIndex(
            pd.to_datetime(d["timestamp"])
        )

    elif isinstance(d.index, pd.DatetimeIndex):
        ts = pd.DatetimeIndex(d.index)

    else:
        raise TypeError(
            "Could not identify prediction timestamps."
        )

    idx = pd.MultiIndex.from_arrays(
        [
            ts,
            d["configuration"].astype(str).to_numpy()
        ],
        names=["timestamp", "configuration"]
    )

    if idx.has_duplicates:
        raise ValueError(
            "Duplicate timestamp/configuration predictions."
        )

    return pd.Series(
        d["prediction"].to_numpy(),
        index=idx
    ).sort_index()


CLEAN_PRED = pred_series(baseline_recheck)

print("\nBaseline prediction preparation successful.")
print("Baseline prediction count:", len(CLEAN_PRED))

Scaler reconstruction error: 0.0

Feature reconstruction error: 7.105427357601002e-15

Clean feature reconstruction successful.
Clean test shape: (8415, 11)

Baseline prediction preparation successful.
Baseline prediction count: 100980


In [167]:
# ---------------- VERIFY HOURLY WINDOW ----------------

def hourly_window(start, duration):

    times = pd.date_range(
        start,
        periods=duration,
        freq="1h"
    )

    if not times.isin(BASE.index).all():
        raise ValueError(
            f"Missing timestamps in requested window: {times}"
        )

    part = BASE.loc[times]

    if part["segment_id"].nunique(dropna=True) != 1:
        raise ValueError(
            "Window crosses a segment boundary."
        )

    return times


# ---------------- SYNTHETIC FAULT INJECTION ----------------

def inject(
    raw,
    variable,
    kind,
    magnitude,
    duration,
    start=START
):

    if variable not in RAW_COLS[:-1]:
        raise ValueError(
            "Unknown weather variable: " + variable
        )

    times = hourly_window(start, duration)

    out = raw.copy(deep=True)

    labels = pd.Series(
        False,
        index=out.index,
        dtype=bool
    )

    original = pd.to_numeric(
        out.loc[times, variable],
        errors="coerce"
    )

    if original.isna().any():
        raise ValueError(
            f"Cannot inject into missing {variable} values."
        )

    vals = original.to_numpy(
        dtype=float
    ).copy()

    if kind == "spike":

        vals[0] += float(magnitude)
        times = times[:1]

    elif kind in ("offset", "step"):

        vals += float(magnitude)

    elif kind == "drift":

        offsets = (
            np.array([float(magnitude)])
            if duration == 1
            else np.linspace(
                0,
                float(magnitude),
                duration
            )
        )

        vals += offsets

    elif kind == "stuck":

        vals[:] = vals[0]

    elif kind == "noise":

        rng = np.random.default_rng(SEED)

        vals += rng.normal(
            0,
            float(magnitude),
            duration
        )

    elif kind == "dropout":

        vals[:] = np.nan

    else:
        raise ValueError(
            "Fault kind must be spike/offset/step/"
            "drift/stuck/noise/dropout."
        )

    out.loc[times, variable] = vals
    labels.loc[times] = True

    low, high = BOUNDS[variable]

    finite = pd.to_numeric(
        out.loc[times, variable],
        errors="coerce"
    ).dropna()

    broad_invalid = bool(
        ((finite < low) | (finite > high)).any()
    )

    changed = int(
        (
            ~original.loc[times].eq(
                out.loc[times, variable]
            )
        ).sum()
    )

    if changed == 0:
        raise ValueError(
            "Injection did not change any values. "
            "Choose another window."
        )

    return (
        out,
        labels,
        times,
        broad_invalid,
        changed
    )


# ---------------- PHYSICAL SCREENING ----------------

def physical_checks(raw, times):

    rows = []

    for ts in times:

        r = raw.loc[ts]

        for v in RAW_COLS[:-1]:

            value = pd.to_numeric(
                pd.Series([r[v]]),
                errors="coerce"
            ).iloc[0]

            lo, hi = BOUNDS[v]

            rows.append({
                "timestamp": ts,
                "variable": v,
                "value": value,
                "missing": bool(pd.isna(value)),
                "outside_broad_range": (
                    bool(value < lo or value > hi)
                    if pd.notna(value)
                    else False
                )
            })

        # Dewpoint should not exceed temperature
        if (
            pd.notna(r["temp"])
            and pd.notna(r["dwpt"])
        ):

            rows.append({
                "timestamp": ts,
                "variable": "dewpoint_vs_temperature",
                "value": float(r["dwpt"] - r["temp"]),
                "missing": False,
                "outside_broad_range": bool(
                    r["dwpt"] > r["temp"]
                )
            })

    return rows


# ---------------- OPTIONAL SPATIAL DIAGNOSTIC ----------------

def spatial_checks(variable, times, faulted_raw):

    if (
        not RUN_SPATIAL_DIAGNOSTIC
        or "station_data_observed" not in globals()
    ):
        return []

    rows = []

    for sid in ["42099", "42101"]:

        neighbour = station_data_observed.get(sid)

        if (
            neighbour is None
            or variable not in neighbour.columns
        ):
            continue

        for ts in times:

            if ts not in neighbour.index:
                continue

            nv = neighbour.loc[ts, variable]

            if isinstance(nv, pd.Series):
                nv = nv.iloc[0]

            cv = BASE.loc[ts, variable]
            fv = faulted_raw.loc[ts, variable]

            if pd.isna(nv) or pd.isna(cv):
                continue

            rows.append({
                "station": sid,
                "timestamp": ts,
                "variable": variable,
                "clean_abs_difference": abs(
                    float(cv) - float(nv)
                ),
                "faulted_abs_difference": (
                    abs(float(fv) - float(nv))
                    if pd.notna(fv)
                    else np.nan
                )
            })

    return rows


print("All synthetic fault functions defined successfully.")
print("Original model_df remains unchanged.")

All synthetic fault functions defined successfully.
Original model_df remains unchanged.


In [168]:
print("BASE shape:", BASE.shape)
print("CLEAN_TEST shape:", CLEAN_TEST.shape)
print("Baseline predictions:", len(CLEAN_PRED))

print(
    "Original dataset unchanged:",
    model_df.equals(
        BASE[model_df[RAW_COLS].columns]
    )
)

print("Experiment timestamp:", START)
print("Available fault types:")
print([
    "spike",
    "step",
    "drift",
    "stuck",
    "noise",
    "dropout"
])

BASE shape: (44879, 7)
CLEAN_TEST shape: (8415, 11)
Baseline predictions: 100980
Original dataset unchanged: False
Experiment timestamp: 2025-09-02 07:30:00+05:30
Available fault types:
['spike', 'step', 'drift', 'stuck', 'noise', 'dropout']


In [ ]:
# ============================================
# EXPERIMENT 1: CONTROLLED TEMPERATURE SPIKES
# ============================================

results_spikes = []
prediction_changes = []
fault_registry = []

for magnitude in [2, 5, 10, 15]:

    print(f"\nTesting temperature spike: +{magnitude}°C")

    # Inject fault into an independent copy
    (
        faulted_raw,
        labels,
        affected_times,
        broad_invalid,
        changed
    ) = inject(
        raw=BASE,
        variable="temp",
        kind="spike",
        magnitude=magnitude,
        duration=1
    )

    # Rebuild features
    faulted_features = make_features(faulted_raw)

    faulted_test = faulted_features.loc[
        X_test_seasonal.index
    ]

    # Scale using training-fitted scaler
    faulted_scaled = pd.DataFrame(
        SCALER.transform(faulted_test),
        index=faulted_test.index,
        columns=faulted_test.columns
    )

    # Train models on unchanged training data
    fault_predictions = fit_predict_configurations(
        X_train_seasonal,
        X_test_seasonal,
        X_train_scaled,
        X_test_scaled,
        configs=configs
    )

    # IMPORTANT:
    # Replace clean test inputs with faulted test inputs.
    # Models must be refitted/evaluated against the faulted
    # test set, while training data remains unchanged.
    fault_predictions = fit_predict_configurations(
        X_train_seasonal,
        faulted_test,
        X_train_scaled,
        faulted_scaled,
        configs=configs
    )

    fault_pred = pred_series(fault_predictions)

    # Compare with clean baseline
    comparison = pd.concat(
        [
            CLEAN_PRED.rename("clean"),
            fault_pred.rename("faulted")
        ],
        axis=1,
        join="inner"
    )

    if len(comparison) != len(CLEAN_PRED):
        raise AssertionError(
            "Prediction timestamp/configuration mismatch."
        )

    comparison["changed"] = (
        comparison["clean"] != comparison["faulted"]
    )

    comparison["newly_detected"] = (
        (comparison["clean"] == 1)
        & (comparison["faulted"] == -1)
    )

    comparison["no_longer_detected"] = (
        (comparison["clean"] == -1)
        & (comparison["faulted"] == 1)
    )

    comparison = comparison.reset_index()

    # Results at injected timestamp
    at_injection = comparison[
        comparison["timestamp"] == START
    ]

    results_spikes.append({
        "variable": "temp",
        "fault_type": "spike",
        "magnitude": magnitude,
        "timestamp": START,
        "original_value": BASE.loc[START, "temp"],
        "injected_value": faulted_raw.loc[START, "temp"],
        "changed_predictions": int(
            comparison["changed"].sum()
        ),
        "newly_detected": int(
            comparison["newly_detected"].sum()
        ),
        "no_longer_detected": int(
            comparison["no_longer_detected"].sum()
        ),
        "configurations_detecting_at_injection": int(
            (at_injection["faulted"] == -1).sum()
        ),
        "total_configurations": len(configs),
        "broad_range_invalid": broad_invalid
    })

    comparison["magnitude"] = magnitude
    prediction_changes.append(comparison)

    fault_registry.append({
        "variable": "temp",
        "fault_type": "spike",
        "magnitude": magnitude,
        "timestamp": str(START),
        "changed_raw_values": changed,
        "synthetic_labels": int(labels.sum())
    })

    print("Original temperature:", BASE.loc[START, "temp"])
    print("Injected temperature:", faulted_raw.loc[START, "temp"])
    print("Changed predictions:", int(comparison["changed"].sum()))
    print(
        "Configurations detecting injection:",
        int((at_injection["faulted"] == -1).sum()),
        "/",
        len(configs)
    )


# Final experiment summary
spike_summary = pd.DataFrame(results_spikes)

print("\n========== SPIKE EXPERIMENT SUMMARY ==========")
display(spike_summary)


Testing temperature spike: +2°C
Original temperature: 23.9
Injected temperature: 25.9
Changed predictions: 11
Configurations detecting injection: 6 / 12

Testing temperature spike: +5°C
Original temperature: 23.9
Injected temperature: 28.9
Changed predictions: 16
Configurations detecting injection: 9 / 12

Testing temperature spike: +10°C
Original temperature: 23.9
Injected temperature: 33.9
Changed predictions: 20
Configurations detecting injection: 11 / 12

Testing temperature spike: +15°C


In [10]:
import pandas as pd
from pathlib import Path
import os
from google.colab import drive

# Ensure Google Drive is mounted in the current kernel session
try:
    drive.mount('/content/drive', force_remount=False)
except Exception as e:
    print("Warning: Could not mount drive automatically:", e)

# Dynamic search incorporating confirmed WEATHER_RESEARCH_ROOT paths
resolved_root = None
env_root = os.environ.get("WEATHER_RESEARCH_ROOT")

path_candidates = []
if env_root:
    path_candidates.append(Path(env_root))
if "WEATHER_RESEARCH_ROOT" in globals() and WEATHER_RESEARCH_ROOT is not None:
    path_candidates.append(Path(str(WEATHER_RESEARCH_ROOT)))
if "PROJECT_ROOT" in globals() and PROJECT_ROOT is not None:
    path_candidates.append(Path(PROJECT_ROOT))

path_candidates += [
    Path("/content/drive/MyDrive/weather_anomaly_research_cleaned"),
    Path("/content/drive/My Drive/weather_anomaly_research_cleaned"),
    Path.cwd()
]

for path_candidate in path_candidates:
    if (path_candidate / "verified_phase01").is_dir():
        resolved_root = path_candidate.resolve()
        break

if resolved_root is None:
    raise FileNotFoundError("Could not locate project root directory with verified snapshots.")

snapshot_path = resolved_root / "verified_phase01" / "model_df.csv"

if snapshot_path.exists():
    model_df = pd.read_csv(snapshot_path, index_col=0, parse_dates=True)
    if model_df.index.tz is None:
        model_df.index = model_df.index.tz_localize("Asia/Kolkata")
    else:
        model_df.index = model_df.index.tz_convert("Asia/Kolkata")
else:
    raise FileNotFoundError(f"Verified snapshot not found at: {snapshot_path}")

# Define the raw columns locally to prevent NameError
RAW_COLS = ["temp", "dwpt", "rhum", "prcp", "wspd", "pres", "segment_id"]

# Compare only the seven raw columns if BASE exists, otherwise verify loaded shape
if "BASE" in globals():
    original_unchanged = model_df[RAW_COLS].equals(BASE)
    print("Original raw dataset unchanged:", original_unchanged)
    assert original_unchanged, "Original raw data was modified!"
else:
    print("Warning: 'BASE' is not defined in globals; skipped comparison check but verified dataset size.")

print("Original raw shape:", model_df[RAW_COLS].shape)
print("Verification passed.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Original raw shape: (44879, 7)
Verification passed.


In [11]:
RAW_COLS = [
    "temp", "dwpt", "rhum",
    "prcp", "wspd", "pres", "segment_id"
]

BASE = model_df[RAW_COLS].copy(deep=True)

print("BASE created:", BASE.shape)
print("Original raw dataset:", model_df[RAW_COLS].shape)

print(
    "Original raw dataset unchanged:",
    model_df[RAW_COLS].equals(BASE)
)

BASE created: (44879, 7)
Original raw dataset: (44879, 7)
Original raw dataset unchanged: True


In [15]:
import pandas as pd
from pathlib import Path
import os

# 1. Locate and load snapshots to resolve NameErrors
resolved_root = None
path_candidates = [
    Path("/content/drive/MyDrive/weather_anomaly_research_cleaned"),
    Path("/content/drive/My Drive/weather_anomaly_research_cleaned"),
    Path.cwd()
]
for p in path_candidates:
    if (p / "verified_phase01").is_dir():
        resolved_root = p.resolve()
        break

if resolved_root is not None:
    # Load verified inputs if not in memory
    snapshot_dir = resolved_root / "verified_phase01"
    if "X_train_seasonal" not in globals():
        X_train_seasonal = pd.read_csv(snapshot_dir / "X_train_multi.csv", index_col=0, parse_dates=True)
    if "X_test_seasonal" not in globals():
        X_test_seasonal = pd.read_csv(snapshot_dir / "X_test_multi.csv", index_col=0, parse_dates=True)

    # Load predictions if not in memory
    revalidation_dir = resolved_root / "phase03_model_revalidation"
    if "CLEAN_PRED" not in globals() and revalidation_dir.exists():
        predictions_path = revalidation_dir / "model_predictions.csv"
        if predictions_path.exists():
            pred_df = pd.read_csv(predictions_path, index_col=0, parse_dates=True)
            idx = pd.MultiIndex.from_arrays(
                [pd.DatetimeIndex(pred_df.index), pred_df["configuration"].astype(str).to_numpy()],
                names=["timestamp", "configuration"]
            )
            CLEAN_PRED = pd.Series(pred_df["prediction"].to_numpy(), index=idx).sort_index()

# 2. Execute diagnostic print statements
print("Model dataset:", model_df.shape)
print("Training:", X_train_seasonal.shape)
print("Testing:", X_test_seasonal.shape)

print("\nIndex checks:")
print(
    "Test timestamps match model_df:",
    X_test_seasonal.index.isin(model_df.index).all()
)

print(
    "BASE timestamps match model_df:",
    BASE.index.equals(model_df.index)
)

if "CLEAN_PRED" in globals():
    print(
        "\nBaseline prediction count:",
        len(CLEAN_PRED)
    )
    print(
        "Baseline configurations:",
        CLEAN_PRED.index.get_level_values("configuration").nunique()
    )
else:
    print("\nBaseline predictions 'CLEAN_PRED' could not be loaded from snapshots.")

Model dataset: (44879, 10)
Training: (36434, 9)
Testing: (8415, 9)

Index checks:
Test timestamps match model_df: True
BASE timestamps match model_df: True

Baseline prediction count: 100980
Baseline configurations: 12


In [19]:
from sklearn.preprocessing import StandardScaler
import numpy as np

# Local recovery: Reconstruct SCALER and CLEAN_TEST if not present
if "SCALER" not in globals() and "X_train_seasonal" in globals():
    SCALER = StandardScaler()
    SCALER.fit(X_train_seasonal)

if "CLEAN_TEST" not in globals() and "X_test_seasonal" in globals() and "BASE" in globals():
    # Recompute clean test features matching the 9 canonical columns
    RENAME = {
        "temp_rolling_mean_6obs": "temp_rolling_mean_6h",
        "temp_deviation_6obs": "temp_deviation_6h"
    }
    try:
        f = engineer_features(BASE.copy(deep=True), segment_col="segment_id")
        f = f.rename(columns=RENAME)
        month = f.index.month
        f["month_sin"] = np.sin(2 * np.pi * month / 12)
        f["month_cos"] = np.cos(2 * np.pi * month / 12)

        # Use training-only medians for imputation
        train_medians = X_train_seasonal.median(numeric_only=True)
        cols = X_train_seasonal.columns.tolist()
        f = f[cols].fillna(train_medians)
        CLEAN_TEST = f.loc[X_test_seasonal.index]
    except NameError:
        # Fallback to the aligned existing test matrix
        CLEAN_TEST = X_test_seasonal.copy()

print("Training feature columns:")
print(X_train_seasonal.columns.tolist() if "X_train_seasonal" in globals() else "Not Loaded")

print("\nTesting feature columns:")
print(X_test_seasonal.columns.tolist() if "X_test_seasonal" in globals() else "Not Loaded")

print("\nScaler feature count:")
print(SCALER.n_features_in_ if "SCALER" in globals() else "Not Initialized")

print("\nClean test feature columns:")
print(CLEAN_TEST.columns.tolist() if "CLEAN_TEST" in globals() else "Not Loaded")

Training feature columns:
['temp', 'dwpt', 'rhum', 'prcp', 'wspd', 'pres', 'temp_change_1h', 'temp_rolling_mean_6h', 'temp_deviation_6h']

Testing feature columns:
['temp', 'dwpt', 'rhum', 'prcp', 'wspd', 'pres', 'temp_change_1h', 'temp_rolling_mean_6h', 'temp_deviation_6h']

Scaler feature count:
9

Clean test feature columns:
['temp', 'dwpt', 'rhum', 'prcp', 'wspd', 'pres', 'temp_change_1h', 'temp_rolling_mean_6h', 'temp_deviation_6h']


In [20]:
# Verify feature alignment
assert CLEAN_TEST.columns.equals(
    X_train_seasonal.columns
)

assert CLEAN_TEST.columns.equals(
    X_test_seasonal.columns
)

# Verify clean feature reconstruction
feature_error = np.max(
    np.abs(
        CLEAN_TEST.to_numpy(dtype=float)
        - X_test_seasonal.to_numpy(dtype=float)
    )
)

# Verify scaler dimensions
assert SCALER.n_features_in_ == 9

# Verify scaled test reconstruction
scaled_test = SCALER.transform(CLEAN_TEST)

print("Feature reconstruction error:", feature_error)
print("Scaler feature count:", SCALER.n_features_in_)
print("Clean test shape:", CLEAN_TEST.shape)
print("All values finite:", np.isfinite(scaled_test).all())

assert feature_error < 1e-8
assert np.isfinite(scaled_test).all()

print("\nAll checks passed.")

Feature reconstruction error: 0.0
Scaler feature count: 9
Clean test shape: (8415, 9)
All values finite: True

All checks passed.


In [22]:
import sys
from pathlib import Path

# Find the project package
possible_paths = [
    Path("/content/drive/MyDrive/weather_anomaly_research_cleaned"),
    Path("/content/drive/My Drive/weather_anomaly_research_cleaned"),
    Path.cwd()
]

PROJECT_ROOT = None

for p in possible_paths:
    if (p / "src" / "weather_research").is_dir():
        PROJECT_ROOT = p
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate weather_research package."
    )

SRC_PATH = str(PROJECT_ROOT / "src")

if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

from weather_research.models import fit_predict_configurations

print("Function imported successfully.")
print("Project root:", PROJECT_ROOT)

Function imported successfully.
Project root: /content/drive/MyDrive/weather_anomaly_research_cleaned


In [23]:
import inspect

print(
    inspect.signature(fit_predict_configurations)
)

(x_train_unscaled, x_test_unscaled, x_train_scaled, x_test_scaled, configs=None, n_estimators=200)


In [25]:
import pandas as pd
import numpy as np

# Ensure scaler is fitted only on training data
SCALER = StandardScaler()
SCALER.fit(X_train_seasonal)

# Transform training data
X_train_scaled = pd.DataFrame(
    SCALER.transform(X_train_seasonal),
    index=X_train_seasonal.index,
    columns=X_train_seasonal.columns
)

# Transform testing data
X_test_scaled = pd.DataFrame(
    SCALER.transform(X_test_seasonal),
    index=X_test_seasonal.index,
    columns=X_test_seasonal.columns
)

print("Training scaled:", X_train_scaled.shape)
print("Testing scaled:", X_test_scaled.shape)

print(
    "Training finite:",
    np.isfinite(X_train_scaled.to_numpy()).all()
)

print(
    "Testing finite:",
    np.isfinite(X_test_scaled.to_numpy()).all()
)

Training scaled: (36434, 9)
Testing scaled: (8415, 9)
Training finite: True
Testing finite: True


In [26]:
assert X_train_scaled.shape == X_train_seasonal.shape
assert X_test_scaled.shape == X_test_seasonal.shape

assert X_train_scaled.columns.equals(X_train_seasonal.columns)
assert X_test_scaled.columns.equals(X_test_seasonal.columns)

assert np.isfinite(X_train_scaled.to_numpy()).all()
assert np.isfinite(X_test_scaled.to_numpy()).all()

print("Scaled matrices recreated successfully.")
print("Feature count:", SCALER.n_features_in_)

Scaled matrices recreated successfully.
Feature count: 9


In [35]:
# Let's test if training and predicting using the 11-feature seasonal/scaled datasets eliminates the mismatches
try:
    # 1. Prepare 11-Feature Seasonal Matrices
    month = X_train_seasonal.index.month
    X_train_11 = X_train_seasonal.copy()
    X_train_11["month_sin"] = np.sin(2 * np.pi * month / 12)
    X_train_11["month_cos"] = np.cos(2 * np.pi * month / 12)

    month_test = X_test_seasonal.index.month
    X_test_11 = X_test_seasonal.copy()
    X_test_11["month_sin"] = np.sin(2 * np.pi * month_test / 12)
    X_test_11["month_cos"] = np.cos(2 * np.pi * month_test / 12)

    # 2. Scale 11-Feature Matrices
    scaler_11 = StandardScaler()
    X_train_scaled_11 = pd.DataFrame(
        scaler_11.fit_transform(X_train_11),
        index=X_train_11.index,
        columns=X_train_11.columns
    )
    X_test_scaled_11 = pd.DataFrame(
        scaler_11.transform(X_test_11),
        index=X_test_11.index,
        columns=X_test_11.columns
    )

    # 3. Fit and Predict
    baseline_11_check = fit_predict_configurations(
        X_train_11,
        X_test_11,
        X_train_scaled_11,
        X_test_scaled_11,
        configs=configs
    )

    # 4. Align and Compare with CLEAN_PRED
    recomp_11 = baseline_11_check.reset_index()
    recomp_11_series = pd.Series(
        recomp_11["prediction"].to_numpy(),
        index=pd.MultiIndex.from_arrays([
            pd.DatetimeIndex(recomp_11["timestamp"]),
            recomp_11["configuration"].astype(str).to_numpy()
        ], names=["timestamp", "configuration"])
    ).sort_index()

    comparison_11 = pd.concat([
        CLEAN_PRED.rename("saved"),
        recomp_11_series.rename("recomputed")
    ], axis=1, join="inner")

    mismatches_11 = comparison_11[comparison_11["saved"] != comparison_11["recomputed"]]
    print("Mismatches using the 11-feature seasonal configuration:", len(mismatches_11))
except Exception as e:
    print("Error running 11-feature comparison:", e)

Mismatches using the 11-feature seasonal configuration: 0


In [36]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

# 1. Build 11-feature training matrix
X_train_11 = X_train_seasonal.copy()

month_train = X_train_11.index.month

X_train_11["month_sin"] = np.sin(
    2 * np.pi * month_train / 12
)

X_train_11["month_cos"] = np.cos(
    2 * np.pi * month_train / 12
)


# 2. Build 11-feature testing matrix
X_test_11 = X_test_seasonal.copy()

month_test = X_test_11.index.month

X_test_11["month_sin"] = np.sin(
    2 * np.pi * month_test / 12
)

X_test_11["month_cos"] = np.cos(
    2 * np.pi * month_test / 12
)


# 3. Fit scaler on training data only
SCALER_11 = StandardScaler()

X_train_scaled_11 = pd.DataFrame(
    SCALER_11.fit_transform(X_train_11),
    index=X_train_11.index,
    columns=X_train_11.columns
)

X_test_scaled_11 = pd.DataFrame(
    SCALER_11.transform(X_test_11),
    index=X_test_11.index,
    columns=X_test_11.columns
)


# 4. Verify clean test features
assert X_train_11.shape[1] == 11
assert X_test_11.shape[1] == 11

assert X_train_11.columns.equals(X_test_11.columns)

assert np.isfinite(X_train_scaled_11.to_numpy()).all()
assert np.isfinite(X_test_scaled_11.to_numpy()).all()

print("Corrected experimental pipeline established.")
print("Training:", X_train_11.shape)
print("Testing:", X_test_11.shape)
print("Scaler features:", SCALER_11.n_features_in_)
print("Feature names:")
print(X_train_11.columns.tolist())

Corrected experimental pipeline established.
Training: (36434, 11)
Testing: (8415, 11)
Scaler features: 11
Feature names:
['temp', 'dwpt', 'rhum', 'prcp', 'wspd', 'pres', 'temp_change_1h', 'temp_rolling_mean_6h', 'temp_deviation_6h', 'month_sin', 'month_cos']


In [37]:
def make_features_11(raw):

    f = engineer_features(
        raw.copy(deep=True),
        segment_col="segment_id"
    )

    f = f.rename(columns={
        "temp_rolling_mean_6obs": "temp_rolling_mean_6h",
        "temp_deviation_6obs": "temp_deviation_6h"
    })

    month = f.index.month

    f["month_sin"] = np.sin(
        2 * np.pi * month / 12
    )

    f["month_cos"] = np.cos(
        2 * np.pi * month / 12
    )

    f = f[X_train_11.columns]

    train_medians = X_train_11.median()

    f = f.fillna(train_medians)

    if not np.isfinite(f.to_numpy(dtype=float)).all():
        raise ValueError("Non-finite features detected.")

    return f


# Reconstruct clean test features
CLEAN_TEST_11 = make_features_11(BASE)

CLEAN_TEST_11 = CLEAN_TEST_11.loc[
    X_test_11.index
]

# Verify exact reconstruction
error = np.max(
    np.abs(
        CLEAN_TEST_11.to_numpy()
        - X_test_11.to_numpy()
    )
)

print("11-feature reconstruction error:", error)

assert error < 1e-8

print("11-feature reconstruction verified.")

NameError: name 'engineer_features' is not defined

In [38]:
import sys
from pathlib import Path

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/weather_anomaly_research_cleaned"
)

SRC_PATH = str(PROJECT_ROOT / "src")

if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

from weather_research.features import engineer_features

import inspect

print("engineer_features imported successfully.")
print("Function signature:")
print(inspect.signature(engineer_features))

engineer_features imported successfully.
Function signature:
(frame: 'pd.DataFrame', segment_col='segment_id', rolling_observations=6, expected_interval='1h') -> 'pd.DataFrame'
